# Q-maze, do it yourself: from numpy basics to Q-learning, planning and a neural network

This notebook is built so that you **type everything**. The code is written in the markdown (text) cells, fully commented. Under each one is an empty code cell where *you* type it in and run it, then compare with the expected output in the next cell. Typing is slow on purpose: it is how the code ends up in your head and not just on your screen.

**What you will build:** a small maze with walls, and an agent that starts knowing nothing and teaches itself the shortest route to the goal. You will write the maze, the Q-table, the update rule and the training loop. Then, in Part E, you will see the same problem solved three more ways: by exact calculation (value iteration), by "imagining" extra experience (Dyna-Q), and with a **neural network trained from scratch in numpy** using replay memory and a target network, which is the machinery behind deep Q-learning.

I assume no data-science background. Every piece of Python and numpy is explained the first time it appears, and a cheat sheet and glossary are at the end.

**Note on your assignment.** I do not know exactly what your assignment's "qmaze" asks for. If it uses a plain table (a grid of numbers), Parts A to D are the core. If it uses a neural network with experience replay (as many online "maze with Keras" tutorials do), Part E is the core. Both are here, so please read the assignment against this map and tell me if something is missing.

## How to use this notebook

**The loop for every step:**
1. Read the explanation above the code.
2. Type the code into the empty code cell below it, including the comments (short explanations you will be glad of later).
3. Run the cell (`Shift+Enter`).
4. Compare with the "Expected output" cell, then read what the output means.
5. Do the "Try this yourself" bit when there is one, and predict the answer *before* running.

**Setup.** Python 3.9 or newer and `numpy` (`pip install numpy`). Use Jupyter or VS Code with the Jupyter extension.

**Cells share memory.** Names created in one cell exist in later ones, so run the steps **in order**. If things behave strangely, restart the kernel and run everything from the top.

**If your output does not match,** check: (1) typos, especially brackets and commas; (2) indentation (Python uses the spaces at the start of a line to mean "this belongs inside the thing above"); (3) a skipped or reordered step; (4) a missing `random.seed(...)` line. Cells with randomness start by setting a seed, so your numbers should match exactly. Differences in the 15th decimal place are harmless.

**Errors are normal.** `NameError` usually means you skipped an earlier step; `SyntaxError` means a typo. Read the last line of the error first.

## Read this first: how a computer reads code, and how to read the line-by-line sections

If you are new to programming, the most useful thing you can learn is to read code **the way the computer does**: one line at a time, working out the *value* and the *type* of everything. The sections called "Line by line" below do this for you with real numbers. Here is the vocabulary they use.

**A variable is a labelled box.** The line `x = 5` makes a box, sticks the label `x` on it, and puts the value `5` inside. Later lines use the label to get the value out. A new `x = 7` replaces what is in the box. The `=` sign means *store*, never "is equal to". (To ask "is it equal?" you write `==`.)

**Every value has a type**, which decides what you can do with it. The types in this notebook:

| Type | What it is | Example | Notes |
|---|---|---|---|
| `int` | whole number | `3` | `3 + 4` is `7` |
| `float` | decimal number | `0.05` | `3.0` is a float even though it looks whole |
| `bool` | True or False | `True` | the result of questions like `3 > 2` |
| `str` | text | `"cart"` | in quotes |
| `list` | ordered row of items, **changeable** | `[1, 2, 3]` | square brackets; items counted from 0 |
| `tuple` | ordered group, **fixed** once made | `(3, 4)` | round brackets; good for positions |
| `dict` | look-up table of key to value | `{(0, 0): 5}` | curly brackets |
| numpy array | block of numbers of one type, with fast maths | `np.array([1, 2])` | `*` and `+` act on every item |
| `function` | a stored recipe | `step` | runs only when *called* |

**The same symbol can mean different things for different types.** `[1, 2] * 2` repeats a list, giving `[1, 2, 1, 2]`. But `np.array([1, 2]) * 2` multiplies every item, giving `[2 4]`. So always ask: *what type is this?*

**Expressions are worked out from the inside outward.** Take `round(2 * (3 + 4), 1)`. The computer does the innermost brackets first: `3 + 4` becomes `7`; then `2 * 7` becomes `14`; then `round(14, 1)` becomes `14`. The sections below show this as a numbered list for every line, with the type and value of each piece, so you can check your own reasoning against it.

**A function is a recipe.** *Defining* it (`def step(state, action):`) only stores the recipe. *Calling* it (`step(my_state, 1)`) runs it: the values you pass in (the **arguments**) fill the empty slots (the **parameters**), the indented lines run top to bottom, and whatever follows `return` comes back out as the value of the call. A function name *without* brackets (`step`) is the recipe itself, which can be passed to other functions. A name *with* brackets (`step(...)`) means "run it now".

**Counting starts at 0.** In `[10, 20, 30]`, position 0 holds `10`, position 1 holds `20`. `x[1:3]` is a *slice*: positions 1 up to, but not including, 3.

**Indentation is meaning.** In Python the spaces at the start of a line show which lines belong inside a function, loop or `if`. Four spaces per level.

**Loops repeat lines.** `for k in range(3):` runs its indented lines three times, with `k` equal to 0, then 1, then 2. `while condition:` repeats as long as the condition is `True`. When a line sits in a loop, the notebook shows its **first pass** and tells you how many times it ran.

### How the "Line by line" sections are laid out

For every line you type you will see:

- **Creates:** the name, its **type** and its **value** at that moment. This answers "what *is* this thing?"
- **Evaluated inside-out:** the pieces of the expression in the order the computer works them out, each with its type and value.
- **Prints:** exactly what appears on screen.
- **Why / what it means:** why the line is there, in plain words.
- For **functions**, a trace of **one concrete call**: what goes in, what each line produces, and what comes back out.

You do not need to memorise the vocabulary. Whenever a word is new, it is explained where it first appears, and the table above is always here to look back at.

## The map of this notebook

- **Part A: the toolkit.** The numpy and Python you need, introduced as needed. (Steps 1 to 3)
- **Part B: the maze world.** The maze, how cells become row numbers, what a move does, and a way to compute the true shortest path so we can check our learner. (Steps 4 to 7)
- **Part C: Q-learning.** The return, the Q-table, one update by hand, the training loop, and watching the learned values spread backward from the goal. (Steps 8 to 15)
- **Part D: experiments.** Breaking the learner on purpose to see why each piece matters. (Step 16)
- **Part E: beyond.** Value iteration, Dyna-Q, and a neural-network Q-function with replay memory. (Steps 17 to 23)

---
# Part A: the toolkit

A small slice of numpy and Python does nearly everything in this project. We learn exactly that slice.

## Step 1: Arrays: a grid of numbers

**The idea first.** A maze is a grid, so it fits a numpy **array**: a block of numbers, all of one type, that you can look into and do arithmetic on all at once. The walls, and later the Q-table, are arrays.

This step has **12 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 1.1: type this

```python
import numpy as np                        # numpy: fast arrays of numbers (nickname np)
```

- **Creates `np`** — type: **module (a library of tools)**; value: `<module numpy>`

In [ ]:
# Step 1.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.2: type this

```python
walls = np.array([[0, 0, 1],
                  [0, 1, 0],
                  [0, 0, 0]])             # a 3x3 array: 1 = wall, 0 = open
```

- **Creates `walls`** — type: **numpy array, shape (3, 3), holding whole numbers**; value: `[[0, 0, 1], [0, 1, 0], [0, 0, 0]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[[0, 0, 1], [0, 1, 0], [0, 0, 0]]` → *list of 3 items (each a list)*: `[[0, 0, 1], [0, 1, 0], [0, 0, 0]]`

- **Why / what it means:** A list of three lists, each holding three `int`s, becomes a numpy array with **3 rows and 3 columns** (shape `(3, 3)`). Each inner list is one row. We use `1` for a wall and `0` for open floor, so the grid is a map.

In [ ]:
# Step 1.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.3: type this

```python
print(walls.shape)                        # (rows, columns)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 0]]`
2. `walls.shape` → *tuple of 2 items (each an int)*: `(3, 3)`

**Prints:**
```text
(3, 3)
```

In [ ]:
# Step 1.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.4: type this

```python
print(walls[1, 1])                        # row 1, column 1 -> 1
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 0]]`
2. `walls[1, 1]` → *numpy integer (a whole number)*: `1`

**Prints:**
```text
1
```

In [ ]:
# Step 1.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.5: type this

```python
walls[2, 2] = 1                           # assign into one position
```

- **Changes `walls` in place** — type: **numpy array, shape (3, 3), holding whole numbers**; value now: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`

- **Why / what it means:** **Assignment into an array:** `walls[2, 2]` names one position (row 2, column 2: counting from 0, the bottom-right) and `= 1` stores a new value there. The array is **changed in place**: no new array is made, the old one is modified. This is how we will write walls into the maze and numbers into the Q-table.

In [ ]:
# Step 1.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.6: type this

```python
print(walls)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`

**Prints:**
```text
[[0 0 1]
 [0 1 0]
 [0 0 1]]
```

In [ ]:
# Step 1.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.7: type this

```python
print(walls.sum())                        # how many walls: add everything up
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`
2. `walls.sum()` → *numpy integer (a whole number)*: `3`

**Prints:**
```text
3
```

In [ ]:
# Step 1.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.8: type this

```python
print(walls[0])                           # the whole of row 0
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`
2. `walls[0]` → *numpy array, shape (3,), holding whole numbers*: `[0, 0, 1]`

**Prints:**
```text
[0 0 1]
```

In [ ]:
# Step 1.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.9: type this

```python
print(walls[:, 2])                        # ":" = every row, so this is column 2
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`
2. `walls[:, 2]` → *numpy array, shape (3,), holding whole numbers*: `[1, 0, 1]`

**Prints:**
```text
[1 0 1]
```

In [ ]:
# Step 1.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.10: type this

```python
print(walls[(1, 1)])                      # a tuple works as an index too
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`
2. `walls[1, 1]` → *numpy integer (a whole number)*: `1`

**Prints:**
```text
1
```

In [ ]:
# Step 1.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.11: type this

```python
print(walls == 1)                         # True/False for every position
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`
2. `walls == 1` → *numpy array, shape (3, 3), holding True/False values*: `[[False, False, True], [False, True, False], [False, False, True]]`

**Prints:**
```text
[[False False  True]
 [False  True False]
 [False False  True]]
```

In [ ]:
# Step 1.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1.12: type this

```python
print(walls.reshape(-1))                  # flatten to one long row of 9 numbers
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (3, 3), holding whole numbers*: `[[0, 0, 1], [0, 1, 0], [0, 0, 1]]`
2. `walls.reshape(-1)` → *numpy array, shape (9,), holding whole numbers*: `[0, 0, 1, 0, 1, 0, 0, 0, 1]`

**Prints:**
```text
[0 0 1 0 1 0 0 0 1]
```

In [ ]:
# Step 1.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 1 complete: everything it printed, and what it means

```text
(3, 3)
1
[[0 0 1]
 [0 1 0]
 [0 0 1]]
3
[0 0 1]
[1 0 1]
1
[[False False  True]
 [False  True False]
 [False False  True]]
[0 0 1 0 1 0 0 0 1]
```

`(3, 3)` is the shape, and `1` is the item at row 1, column 1. After setting the bottom-right to 1, the array holds three walls, and `walls.sum()` is `3`. Row 0 is `[0 0 1]`, and column 2 is `[1 0 1]`. The mask shows `True` exactly where the 1s are. The flattened array reads the grid left to right, top to bottom: `[0 0 1 0 1 0 0 0 1]`.

**Try this yourself.** Predict, then check: `walls[1:, :2]` (rows from 1 on, the first two columns) and `walls.sum(axis=0)` (we will meet `axis` in the next step).

## Step 2: Axes, max, argmax, any, and one-hot vectors

**The idea first.** Two operations are at the centre of Q-learning: **max** (the best number in a row) and **argmax** (the *position* of the best number, which names the best action). With a 2D table you must understand `axis`.

This step has **9 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 2.1: type this

```python
table = np.array([[0.0, -2.5, 0.0, 4.0],
                  [1.0,  1.0, 0.5, 0.0]])  # 2 "cells" x 4 "actions"
```

- **Creates `table`** — type: **numpy array, shape (2, 4), holding decimal numbers**; value: `[[0., -2.5, 0., 4.], [1., 1., 0.5, 0.]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[0.0, -2.5, 0.0, 4.0]` → *list of 4 items (each a float)*: `[0.0, -2.5, 0.0, 4.0]`
2. `[[0.0, -2.5, 0.0, 4.0], [1.0, 1.0, 0.5, 0.0]]` → *list of 2 items (each a list)*: `[[0.0, -2.5, 0.0, 4.0], [1.0, 1.0, 0.5, 0.0]]`

- **Why / what it means:** A 2 x 4 array of decimals: two 'cells', four 'actions'. This is a miniature Q-table. Row 0 is `[0.0, -2.5, 0.0, 4.0]`: action 3 is the best, action 1 the worst.

In [ ]:
# Step 2.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.2: type this

```python
print(table.max(axis=1))                  # axis=1: best number in each ROW -> [4. 1.]
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (2, 4), holding decimal numbers*: `[[0., -2.5, 0., 4.], [1., 1., 0.5, 0.]]`
2. `table.max(axis=1)` → *numpy array, shape (2,), holding decimal numbers*: `[4., 1.]`

**Prints:**
```text
[4. 1.]
```

In [ ]:
# Step 2.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.3: type this

```python
print(table.argmax(axis=1))               # POSITION of the best number in each row -> [3 0]
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (2, 4), holding decimal numbers*: `[[0., -2.5, 0., 4.], [1., 1., 0.5, 0.]]`
2. `table.argmax(axis=1)` → *numpy array, shape (2,), holding whole numbers*: `[3, 0]`

**Prints:**
```text
[3 0]
```

In [ ]:
# Step 2.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.4: type this

```python
print(table[1] == table[1].max())         # which entries equal the row's maximum?
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (2, 4), holding decimal numbers*: `[[0., -2.5, 0., 4.], [1., 1., 0.5, 0.]]`
2. `table[1]` → *numpy array, shape (4,), holding decimal numbers*: `[1., 1., 0.5, 0.]`
3. `table[1].max()` → *float (decimal number)*: `1.0`
4. `table[1] == table[1].max()` → *numpy array, shape (4,), holding True/False values*: `[True, True, False, False]`

**Prints:**
```text
[ True  True False False]
```

In [ ]:
# Step 2.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.5: type this

```python
print(np.flatnonzero(table[1] == table[1].max()))   # the positions of those entries
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (2, 4), holding decimal numbers*: `[[0., -2.5, 0., 4.], [1., 1., 0.5, 0.]]`
2. `table[1]` → *numpy array, shape (4,), holding decimal numbers*: `[1., 1., 0.5, 0.]`
3. `table[1].max()` → *float (decimal number)*: `1.0`
4. `table[1] == table[1].max()` → *numpy array, shape (4,), holding True/False values*: `[True, True, False, False]`
5. `np.flatnonzero(table[1] == table[1].max())` → *numpy array, shape (2,), holding whole numbers*: `[0, 1]`

**Prints:**
```text
[0 1]
```

In [ ]:
# Step 2.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.6: type this

```python
print(table[0].any(), np.zeros(3).any())  # .any(): is at least one entry non-zero?
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (2, 4), holding decimal numbers*: `[[0., -2.5, 0., 4.], [1., 1., 0.5, 0.]]`
2. `table[0]` → *numpy array, shape (4,), holding decimal numbers*: `[0., -2.5, 0., 4.]`
3. `table[0].any()` → *bool*: `np.True_`
4. `np.zeros(3)` → *numpy array, shape (3,), holding decimal numbers*: `[0., 0., 0.]`
5. `np.zeros(3).any()` → *bool*: `np.False_`

**Prints:**
```text
True False
```

In [ ]:
# Step 2.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.7: type this

```python
print(np.zeros((3, 4)).shape)             # a fresh all-zero table
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.zeros((3, 4))` → *numpy array, shape (3, 4), holding decimal numbers*: `[[0., 0., 0., 0.], [0., 0., 0., 0.], [0., 0., 0., 0.]]`
2. `np.zeros((3, 4)).shape` → *tuple of 2 items (each an int)*: `(3, 4)`

**Prints:**
```text
(3, 4)
```

In [ ]:
# Step 2.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.8: type this

```python
print(np.eye(3))                          # identity matrix: 1s on the diagonal, 0 elsewhere
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.eye(3)` → *numpy array, shape (3, 3), holding decimal numbers*: `[[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]]`

**Prints:**
```text
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
```

In [ ]:
# Step 2.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2.9: type this

```python
print(np.eye(3)[1])                       # row 1 of it: a "one-hot" vector [0, 1, 0]
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.eye(3)` → *numpy array, shape (3, 3), holding decimal numbers*: `[[1., 0., 0.], [0., 1., 0.], [0., 0., 1.]]`
2. `np.eye(3)[1]` → *numpy array, shape (3,), holding decimal numbers*: `[0., 1., 0.]`

**Prints:**
```text
[0. 1. 0.]
```

In [ ]:
# Step 2.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 2 complete: everything it printed, and what it means

```text
[4. 1.]
[3 0]
[ True  True False False]
[0 1]
True False
(3, 4)
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0. 1. 0.]
```

`[4. 1.]` and `[3 0]`: the best per row and where it sits. `[ True True False False]` marks row 1's two tied maximums (positions 0 and 1), and `[0 1]` lists those positions. `True False` shows that the first row has a non-zero entry and the zero array does not. `(3, 4)` is the shape. The identity matrix prints as three rows each with one 1, and `np.eye(3)[1]` is `[0. 1. 0.]`, a one-hot vector for item 1.

**Try this yourself.** Predict: what does `table.max(axis=0)` give? (One answer per column.)

## Step 3: Random numbers, tuples, dictionaries, deque and global

**The idea first.** A grab-bag of Python tools used later. Randomness drives exploring; a **dictionary** and a **deque** are used by the shortest-path check; `global` lets a function change a name defined outside it.

This step has **23 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 3.1: type this

```python
import random
```

- **Creates `random`** — type: **module (a library of tools)**; value: `<module random>`

In [ ]:
# Step 3.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.2: type this

```python
from collections import deque             # a list that is fast to take items off the front of
```

- **Creates `deque`** — type: **type**; value: `<class 'collections.deque'>`

In [ ]:
# Step 3.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.3: type this

```python
random.seed(42)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(42)` → *None (nothing)*: `None`

- **Why / what it means:** Fixes the starting point of Python's random formula so the 'random' numbers repeat. The three values that follow are the first outputs of that fixed sequence.

In [ ]:
# Step 3.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.4: type this

```python
print(random.random())                    # decimal between 0 and 1
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.random()` → *float (decimal number)*: `0.6394267984578837`

**Prints:**
```text
0.6394267984578837
```

In [ ]:
# Step 3.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.5: type this

```python
print(random.randrange(4))                # whole number 0, 1, 2 or 3
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.randrange(4)` → *int (whole number)*: `0`

**Prints:**
```text
0
```

In [ ]:
# Step 3.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.6: type this

```python
print(random.choice([10, 20, 30]))        # one item picked at random
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.choice([10, 20, 30])` → *int (whole number)*: `30`

**Prints:**
```text
30
```

In [ ]:
# Step 3.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.7: type this

```python
def add(x, y):                            # a function: inputs x and y, result handed back
    return x + y
```

- **Creates** `add` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Defines a function. `x` and `y` are **parameters**, empty slots. `return x + y` says what comes back. Calling `add(row, col)` later fills the slots with the values of `row` and `col`.

In [ ]:
# Step 3.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.8: type this

```python
cell = (1, 2)                             # a tuple: a fixed pair
```

- **Creates `cell`** — type: **tuple of 2 items (each an int)**; value: `(1, 2)`

- **Why / what it means:** A **tuple** of two ints: round brackets, fixed once made. A (row, column) position is exactly the kind of thing a tuple is for.

In [ ]:
# Step 3.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.9: type this

```python
row, col = cell                           # unpacking
```

- **Creates `row`** — type: **int (whole number)**; value: `1`

- **Creates `col`** — type: **int (whole number)**; value: `2`

- **Why / what it means:** **Unpacking:** the first item of the tuple goes to `row` and the second to `col`, matched by position. Both are plain `int`s afterwards.

In [ ]:
# Step 3.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.10: type this

```python
print(add(row, col))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `row` → *int (whole number)*: `1`
2. `col` → *int (whole number)*: `2`
3. `add(row, col)` → *int (whole number)*: `3`

**Prints:**
```text
3
```

In [ ]:
# Step 3.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.11: type this

```python
distance = {(0, 0): 0}                    # a dictionary: look things up by key
```

- **Creates `distance`** — type: **dict with 1 entry**; value: `{(0, 0): 0}`

- **Why / what it means:** A **dictionary**: curly brackets with `key: value` pairs. Here the key is the tuple `(0, 0)` (a cell) and the value is `0` (steps to reach it). A dictionary lets you look up a value by its key, like a phone book. Tuples can be keys; lists cannot, because lists can change.

In [ ]:
# Step 3.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.12: type this

```python
distance[(0, 1)] = 1                      # add an entry
```

- **Changes `distance` in place** — type: **dict with 2 entries**; value now: `{(0, 0): 0, (0, 1): 1}`

- **Why / what it means:** Adds a new entry to the dictionary by assigning to a new key. The dictionary is **changed in place** and now has two entries.

In [ ]:
# Step 3.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.13: type this

```python
print((0, 1) in distance, (5, 5) in distance)   # "in" asks whether a key exists
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `distance` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
2. `(0, 1) in distance` → *bool (True/False)*: `True`
3. `(5, 5) in distance` → *bool (True/False)*: `False`

**Prints:**
```text
True False
```

In [ ]:
# Step 3.13: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.14: type this

```python
print(distance[(0, 1)])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `distance` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
2. `distance[0, 1]` → *int (whole number)*: `1`

**Prints:**
```text
1
```

In [ ]:
# Step 3.14: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.15: type this

```python
line = deque([(0, 0)])                    # a waiting line holding one item
```

- **Creates `line`** — type: **deque (a waiting line) with 1 item**; value: `[(0, 0)]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[(0, 0)]` → *list of 1 item (each a tuple)*: `[(0, 0)]`

- **Why / what it means:** A **deque** (pronounced 'deck') is a waiting line. We build it from a list holding one item, the cell `(0, 0)`. Unlike a list, taking items off the *front* is fast, which matters for the shortest-path search.

In [ ]:
# Step 3.15: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.16: type this

```python
line.append((0, 1))                       # join at the back
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `line` → *deque (a waiting line) with 1 item*: `[(0, 0)]`
2. `line.append((0, 1))` → *None (nothing)*: `None`

- **Why / what it means:** `.append` joins the back of the line. Notice the *double* brackets: the inner `(0, 1)` is the tuple, the outer pair is the function call. The deque now holds two cells.

In [ ]:
# Step 3.16: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.17: type this

```python
print(line.popleft())                     # take from the front -> (0, 0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `line` → *deque (a waiting line) with 2 items*: `[(0, 0), (0, 1)]`
2. `line.popleft()` → *tuple of 2 items (each an int)*: `(0, 0)`

**Prints:**
```text
(0, 0)
```

In [ ]:
# Step 3.17: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.18: type this

```python
print(len(line))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `line` → *deque (a waiting line) with 1 item*: `[(0, 1)]`
2. `len(line)` → *int (whole number)*: `1`

**Prints:**
```text
1
```

In [ ]:
# Step 3.18: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.19: type this

```python
counter = 0
```

- **Creates `counter`** — type: **int (whole number)**; value: `0`

- **Why / what it means:** A counter that lives *outside* any function (a **global** name).

In [ ]:
# Step 3.19: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.20: type this

```python
def bump():
    global counter                        # "global" lets a function CHANGE an outside name
    counter += 1
```

- **Creates** `bump` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** A function with no parameters. The line `global counter` tells Python that `counter` here means the *outside* name; without it, `counter += 1` would try to create a new local variable and fail. Defining it does not run it.

In [ ]:
# Step 3.20: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.21: type this

```python
bump(); bump()
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `bump()` → *None (nothing)*: `None`

- **Why / what it means:** Two statements on one line, separated by `;`. This is the first call: it runs the function body, adding 1 to the global counter, which becomes 1.

In [ ]:
# Step 3.21: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.22: type this

```python
bump()
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `bump()` → *None (nothing)*: `None`

- **Why / what it means:** The second call: the counter becomes 2.

In [ ]:
# Step 3.22: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3.23: type this

```python
print(counter)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `counter` → *int (whole number)*: `2`

**Prints:**
```text
2
```

In [ ]:
# Step 3.23: type the code from the cell above, then run this cell (Shift+Enter)


### Step 3 complete: everything it printed, and what it means

```text
0.6394267984578837
0
30
3
True False
1
(0, 0)
1
2
```

`0.6394267984578837` is the famous first number of seed 42 (the next two depend on the Python version, so small differences there are fine). `3` is `add(1, 2)`. `True False` answers the two `in` questions. `1` is `distance[(0, 1)]`. `(0, 0)` is the first item taken from the front of the line (the one that joined first), and `1` item remains. `2` is the counter after two calls.

---
# Part B: the maze world

An **environment** is the part of the system that, given where you are and what you try, tells you what happens. The agent will live inside it, and never see how it works. We build it ourselves.

## Step 4: Describing the maze as text, and loading it

**The idea first.** It is easiest to *draw* a maze as text and have code convert the drawing into data. Each string is one row. This also lets us load a different maze later by passing different text.

This step has **5 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 4.1: type this

```python
SMALL = [
    "S....",                              # S = start
    "##.#.",                              # # = wall
    ".....",                              # . = open
    ".####",                              # G = goal
    "....G",
]
```

- **Creates `SMALL`** — type: **list of 5 items (each a str)**; value: `['S....', '##.#.', '.....', '.####', '....G']`

- **Why / what it means:** `SMALL` is a **list of 5 strings**; each string is one row of the maze drawn as text. Strings are sequences of characters, so `SMALL[1][2]` means 'row 1, then character 2'. The maze is written as a picture so it is easy to read and edit.

In [ ]:
# Step 4.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.2: type this

```python
def load_maze(text):
    global MAZE_TEXT, ROWS, COLS, walls, START, GOAL      # these names live outside the function
    MAZE_TEXT = text
    ROWS = len(text)                      # number of strings = number of rows
    COLS = len(text[0])                   # length of the first string = number of columns
    walls = np.zeros((ROWS, COLS), dtype=int)
    for r in range(ROWS):
        for c in range(COLS):
            ch = text[r][c]               # character at row r, column c
            if ch == "#":
                walls[r, c] = 1
            elif ch == "S":
                START = (r, c)
            elif ch == "G":
                GOAL = (r, c)
```

- **Creates** `load_maze` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Turns the picture into data the program can use. The line `global MAZE_TEXT, ROWS, COLS, walls, START, GOAL` lists names that live *outside* the function and that we want to **assign to**. Without it, the assignments below would create local variables that vanish when the function ends, and the rest of the program would never see them.

In [ ]:
# Step 4.2: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `load_maze(SMALL)`

**What the function receives** (its inputs, called *parameters*):
- `text` — type: **list of 5 items (each a str)**; value: `['S....', '##.#.', '.....', '.####', '....G']`

**Now each line runs, top to bottom:**

```python
MAZE_TEXT = text
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
- **Why:** Remembers the drawing, so another function can restore it later. It is the same list of strings that was passed in.

```python
ROWS = len(text)                      # number of strings = number of rows
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `len(text)` → *int (whole number)*: `5`
- **Why:** `len` counts the items in the list: 5 strings means 5 rows. An `int`.

```python
COLS = len(text[0])                   # length of the first string = number of columns
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `text[0]` → *str (text, 5 characters)*: `'S....'`
3. `len(text[0])` → *int (whole number)*: `5`
- **Why:** `text[0]` is the first string; `len` of a string counts its characters: 5 columns.


#### …the same call, continued: `load_maze(SMALL)`

```python
walls = np.zeros((ROWS, COLS), dtype=int)
```
Evaluated inside-out (the main pieces):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`
3. `(ROWS, COLS)` → *tuple of 2 items (each an int)*: `(5, 5)`
4. `np.zeros((ROWS, COLS), dtype=int)` → *numpy array, shape (5, 5), holding whole numbers*: `[[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]`
- **Why:** A 5 x 5 array of whole-number zeros. `dtype=int` asks for whole numbers, not decimals. Everything starts as 'open'; the loops below put 1s where the walls are.

```python
for r in range(ROWS):
```
Evaluated inside-out (the main pieces):
1. `ROWS` → *int (whole number)*: `5`
2. `range(ROWS)` → *range*: `range(0, 5)`
- **Now `r` holds** — type: **int (whole number)**; value: `0`
- **Why:** The outer loop: `r` takes the values 0, 1, 2, 3, 4 in turn, one row per pass. Shown here for the first pass.

```python
for c in range(COLS):
```
Evaluated inside-out (the main pieces):
1. `COLS` → *int (whole number)*: `5`
2. `range(COLS)` → *range*: `range(0, 5)`
- **Now `c` holds** — type: **int (whole number)**; value: `0`
- **Why:** The inner loop, run in full for every `r`: `c` takes 0 to 4. Together the two loops visit all 25 cells, row by row.

```python
ch = text[r][c]               # character at row r, column c
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `r` → *int (whole number)*: `0`
3. `text[r]` → *str (text, 5 characters)*: `'S....'`
4. `c` → *int (whole number)*: `0`
- **Now `ch` holds** — type: **str (text, 1 characters)**; value: `'S'`
- **Why:** `text[r]` is row `r`'s string, `[c]` picks one character of it. So `ch` is a one-character `str` such as `'S'`, `'.'` or `'#'`.

```python
if ch == "#":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'S'`
2. `ch == '#'` → *bool (True/False)*: `False`
- **Why:** `==` compares and gives a `bool`. If the character is a `#` we mark a wall.


#### …the same call, continued: `load_maze(SMALL)`

```python
elif ch == "S":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'S'`
2. `ch == 'S'` → *bool (True/False)*: `True`
- **Why:** `elif` means 'otherwise, if...'. It is only tested when the previous `if` was false.

```python
START = (r, c)
```
Evaluated inside-out (the main pieces):
1. `r` → *int (whole number)*: `0`
2. `c` → *int (whole number)*: `0`
3. `(r, c)` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Why:** Stores the start position as a **tuple** `(row, column)`. Because of the `global` line, this sets the outside name `START`. (`GOAL = (r, c)` in the next branch does the same for the goal; it runs when the loop reaches the `G`, which is beyond this short trace.)

```python
for c in range(COLS):
```
- **Now `c` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
ch = text[r][c]               # character at row r, column c
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `r` → *int (whole number)*: `0`
3. `text[r]` → *str (text, 5 characters)*: `'S....'`
4. `c` → *int (whole number)*: `1`
- **Now `ch` holds** — type: **str (text, 1 characters)**; value: `'.'`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if ch == "#":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == '#'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `load_maze(SMALL)`

```python
elif ch == "S":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == 'S'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
elif ch == "G":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == 'G'` → *bool (True/False)*: `False`
- **Why:** Same test for the goal character.

```python
for c in range(COLS):
```
- **Now `c` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
ch = text[r][c]               # character at row r, column c
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `r` → *int (whole number)*: `0`
3. `text[r]` → *str (text, 5 characters)*: `'S....'`
4. `c` → *int (whole number)*: `2`
5. `text[r][c]` → *str (text, 1 characters)*: `'.'`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if ch == "#":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == '#'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
elif ch == "S":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == 'S'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `load_maze(SMALL)`

```python
elif ch == "G":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == 'G'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for c in range(COLS):
```
- **Now `c` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
ch = text[r][c]               # character at row r, column c
```
Evaluated inside-out (the main pieces):
1. `text` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `r` → *int (whole number)*: `0`
3. `text[r]` → *str (text, 5 characters)*: `'S....'`
4. `c` → *int (whole number)*: `3`
5. `text[r][c]` → *str (text, 1 characters)*: `'.'`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if ch == "#":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == '#'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
elif ch == "S":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == 'S'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
elif ch == "G":
```
Evaluated inside-out (the main pieces):
1. `ch` → *str (text, 1 characters)*: `'.'`
2. `ch == 'G'` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 4.3: type this

```python
load_maze(SMALL)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `SMALL` → *list of 5 items (each a str)*: `['S....', '##.#.', '.....', '.####', '....G']`
2. `load_maze(SMALL)` → *None (nothing)*: `None`

- **Why / what it means:** Runs the function on our maze. Afterwards the global names `ROWS`, `COLS`, `walls`, `START` and `GOAL` exist and describe it.

In [ ]:
# Step 4.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.4: type this

```python
print(walls)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `walls` → *numpy array, shape (5, 5), holding whole numbers*: `[[0, 0, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 1, 1, 1], [0, 0, 0, 0, 0]]`

**Prints:**
```text
[[0 0 0 0 0]
 [1 1 0 1 0]
 [0 0 0 0 0]
 [0 1 1 1 1]
 [0 0 0 0 0]]
```

In [ ]:
# Step 4.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4.5: type this

```python
print("start", START, "goal", GOAL, "size", ROWS, "x", COLS)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
3. `ROWS` → *int (whole number)*: `5`
4. `COLS` → *int (whole number)*: `5`

**Prints:**
```text
start (0, 0) goal (4, 4) size 5 x 5
```

In [ ]:
# Step 4.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 4 complete: everything it printed, and what it means

```text
[[0 0 0 0 0]
 [1 1 0 1 0]
 [0 0 0 0 0]
 [0 1 1 1 1]
 [0 0 0 0 0]]
start (0, 0) goal (4, 4) size 5 x 5
```

The `walls` array matches the picture: row 1 is `##.#.`, so it prints `[1 1 0 1 0]`. Rows count downward from 0 and columns rightward from 0, so `(0, 0)` is the top-left and `(4, 4)` the bottom-right. **A position is always (row, column), row first.** There are two routes to the goal: a short one through the gap at `(1, 2)`, and a longer one around the right side and back through `(2, 4)` and `(2, 3)`.

**Try this yourself.** Add a `#` somewhere to a copy of `SMALL` and call `load_maze` on it. Does the `walls` array change where you expect? (Remember to run `load_maze(SMALL)` again afterwards.)

## Step 5: Giving every cell a single number

**The idea first.** The Q-table needs one row per cell, so each `(row, col)` pair must become one whole number. We number the cells the way you read a page: left to right, top to bottom.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 5.1: type this

```python
def state_of(cell):
    return cell[0] * COLS + cell[1]       # count cells left to right, top to bottom
```

- **Creates** `state_of` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Converts a (row, column) tuple into one whole number. It is a function with one parameter, `cell`.

In [ ]:
# Step 5.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `state_of((1, 2))`

**What the function receives** (its inputs, called *parameters*):
- `cell` — type: **tuple of 2 items (each an int)**; value: `(1, 2)`

**Now each line runs, top to bottom:**

```python
return cell[0] * COLS + cell[1]       # count cells left to right, top to bottom
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(1, 2)`
2. `cell[0]` → *int (whole number)*: `1`
3. `COLS` → *int (whole number)*: `5`
4. `cell[0] * COLS` → *int (whole number)*: `5`
5. `cell[1]` → *int (whole number)*: `2`
6. `cell[0] * COLS + cell[1]` → *int (whole number)*: `7`
- **Why:** `cell[0]` is the row and `cell[1]` the column (tuples count from 0 too). Multiplying the row by the width skips whole rows, then we add the column. For `(1, 2)`: one full row of 5 cells, then 2 more: 7. This is how a page is numbered when you read left to right, top to bottom.

**The function hands back** (its *return value*) — type: **int (whole number)**; value: `7`


### Step 5.2: type this

```python
def cell_of(state):
    return divmod(state, COLS)            # divmod(a, b) -> (a // b, a % b): quotient, remainder
```

- **Creates** `cell_of` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The reverse conversion: a whole number back to a (row, column) pair.

In [ ]:
# Step 5.2: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `cell_of(7)`

**What the function receives** (its inputs, called *parameters*):
- `state` — type: **int (whole number)**; value: `7`

**Now each line runs, top to bottom:**

```python
return divmod(state, COLS)            # divmod(a, b) -> (a // b, a % b): quotient, remainder
```
Evaluated inside-out (the main pieces):
1. `state` → *int (whole number)*: `7`
2. `COLS` → *int (whole number)*: `5`
3. `divmod(state, COLS)` → *tuple of 2 items (each an int)*: `(1, 2)`
- **Why:** `divmod(a, b)` returns **two** numbers as a tuple: how many times `b` fits into `a` (the quotient), and what is left over (the remainder). For 7 and 5: it fits once with 2 left over: `(1, 2)`, which is exactly the row and column.

**The function hands back** (its *return value*) — type: **tuple of 2 items (each an int)**; value: `(1, 2)`


### Step 5.3: type this

```python
print(state_of((0, 0)), state_of((1, 2)), state_of(GOAL))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `state_of((0, 0))` → *int (whole number)*: `0`
2. `state_of((1, 2))` → *int (whole number)*: `7`
3. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
4. `state_of(GOAL)` → *int (whole number)*: `24`

**Prints:**
```text
0 7 24
```

In [ ]:
# Step 5.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.4: type this

```python
print(cell_of(7), cell_of(24))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `cell_of(7)` → *tuple of 2 items (each an int)*: `(1, 2)`
2. `cell_of(24)` → *tuple of 2 items (each an int)*: `(4, 4)`

**Prints:**
```text
(1, 2) (4, 4)
```

In [ ]:
# Step 5.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.5: type this

```python
print(divmod(17, 5))                      # 17 = 3 x 5 + 2  -> (3, 2)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `divmod(17, 5)` → *tuple of 2 items (each an int)*: `(3, 2)`

**Prints:**
```text
(3, 2)
```

In [ ]:
# Step 5.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.6: type this

```python
n_states = ROWS * COLS
```

- **Creates `n_states`** — type: **int (whole number)**; value: `25`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`

- **Why / what it means:** The number of cells: 5 x 5 = 25 (an `int`). This will be the number of **rows** in the Q-table.

In [ ]:
# Step 5.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5.7: type this

```python
print(n_states)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `n_states` → *int (whole number)*: `25`

**Prints:**
```text
25
```

In [ ]:
# Step 5.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 5 complete: everything it printed, and what it means

```text
0 7 24
(1, 2) (4, 4)
(3, 2)
25
```

`0 7 24`: the start is state 0, `(1, 2)` is state 7, and the goal `(4, 4)` is state 24. `cell_of(7)` gives back `(1, 2)` and `cell_of(24)` gives `(4, 4)`. `(3, 2)` is the quotient and remainder of 17 divided by 5. `25` is the total.

**Try this yourself.** By hand: what is `state_of((3, 0))`? And `cell_of(13)`? (Answers: 15, and `(2, 3)`.)

## Step 6: What happens when the agent moves

**The idea first.** This is the **environment's transition function**: given where you are and what you try, it returns what happens: the new position, a reward, and whether the episode is over. Choosing the reward numbers is how *you* tell the agent what you want.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 6.1: type this

```python
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]    # up, down, left, right as (row change, col change)
```

- **Creates `ACTIONS`** — type: **list of 4 items (each a tuple)**; value: `[(-1, 0), (1, 0), (0, -1), (0, 1)]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `(-1, 0)` → *tuple of 2 items (each an int)*: `(-1, 0)`
2. `(0, -1)` → *tuple of 2 items (each an int)*: `(0, -1)`

- **Why / what it means:** A **list of four tuples**. Each tuple is `(row change, column change)`. Moving up decreases the row number by 1, because row 0 is the *top*. The position in the list (0 to 3) is the action's number, and will be its column in the Q-table.

In [ ]:
# Step 6.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.2: type this

```python
NAMES = ["up", "down", "left", "right"]
STEP_REWARD = -1                          # every ordinary move costs 1
WALL_REWARD = -5                          # bumping a wall or the edge costs 5
```

- **Creates `NAMES`** — type: **list of 4 items (each a str)**; value: `['up', 'down', 'left', 'right']`

- **Why / what it means:** A list of four strings, parallel to `ACTIONS`, used only for printing.

- **Creates `STEP_REWARD`** — type: **int (whole number)**; value: `-1`

- **Why / what it means:** The reward for an ordinary move: an `int`, negative, so wandering is costly and short routes win.

- **Creates `WALL_REWARD`** — type: **int (whole number)**; value: `-5`

- **Why / what it means:** A larger penalty for bumping a wall or the edge, so the agent learns not to try impossible moves.

In [ ]:
# Step 6.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.3: type this

```python
GOAL_REWARD = 10                          # reaching the goal pays 10
```

- **Creates `GOAL_REWARD`** — type: **int (whole number)**; value: `10`

- **Why / what it means:** The prize for reaching the goal.

In [ ]:
# Step 6.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.4: type this

```python
def step(cell, action):
    d_row, d_col = ACTIONS[action]
    row = cell[0] + d_row                 # where the move WOULD put us
    col = cell[1] + d_col
    if row < 0 or row >= ROWS or col < 0 or col >= COLS or walls[row, col] == 1:
        return cell, WALL_REWARD, False   # blocked: stay where we are
    if (row, col) == GOAL:
        return (row, col), GOAL_REWARD, True     # done = True ends the episode
    return (row, col), STEP_REWARD, False
```

- **Creates** `step` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The environment's **transition function**. It receives a position (tuple) and an action number (int) and returns **three** values packed in a tuple: the new position, the reward, and a `bool` saying whether the episode is over.

In [ ]:
# Step 6.4: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `step((0, 1), 1)`

**What the function receives** (its inputs, called *parameters*):
- `cell` — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- `action` — type: **int (whole number)**; value: `1`

**Now each line runs, top to bottom:**

```python
d_row, d_col = ACTIONS[action]
```
Evaluated inside-out (the main pieces):
1. `ACTIONS` → *list of 4 items (each a tuple)*: `[(-1, 0), (1, 0), (0, -1), (0, 1)]`
2. `action` → *int (whole number)*: `1`
3. `ACTIONS[action]` → *tuple of 2 items (each an int)*: `(1, 0)`
- **Now `d_row` holds** — type: **int (whole number)**; value: `1`
- **Now `d_col` holds** — type: **int (whole number)**; value: `0`
- **Why:** `ACTIONS[action]` looks up the tuple at that position in the list; unpacking splits it into the row change and the column change.

```python
row = cell[0] + d_row                 # where the move WOULD put us
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `cell[0]` → *int (whole number)*: `0`
3. `d_row` → *int (whole number)*: `1`
- **Now `row` holds** — type: **int (whole number)**; value: `1`
- **Why:** The row the agent *would* move to. This is only a candidate; we have not yet checked whether it is allowed.

```python
col = cell[1] + d_col
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `cell[1]` → *int (whole number)*: `1`
3. `d_col` → *int (whole number)*: `0`
- **Now `col` holds** — type: **int (whole number)**; value: `1`
- **Why:** The candidate column.


#### …the same call, continued: `step((0, 1), 1)`

```python
if row < 0 or row >= ROWS or col < 0 or col >= COLS or walls[row, col] == 1:
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `1`
2. `row < 0` → *bool (True/False)*: `False`
3. `ROWS` → *int (whole number)*: `5`
4. `row >= ROWS` → *bool (True/False)*: `False`
5. `col` → *int (whole number)*: `1`
6. `col < 0` → *bool (True/False)*: `False`
7. `COLS` → *int (whole number)*: `5`
8. `col >= COLS` → *bool (True/False)*: `False`
- **Why:** A chain of questions joined by `or`; the whole thing is `True` if **any** is true. The first four check the grid edges. The last, `walls[row, col] == 1`, checks for a wall. Python stops at the first true part, so the wall lookup is never reached for an out-of-range position (which would crash). Here the move is allowed, so the whole condition is `False`.

```python
return cell, WALL_REWARD, False   # blocked: stay where we are
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `WALL_REWARD` → *int (whole number)*: `-5`
3. `(cell, WALL_REWARD, False)` → *tuple of 3 items*: `((0, 1), -5, False)`
- **Why:** The blocked case: we return the **old** `cell` (the agent does not move), the wall penalty, and `False` (the episode continues).

**The function hands back** (its *return value*) — type: **tuple of 3 items**; value: `((0, 1), -5, False)`


#### Inside the function: one concrete call, `step((0, 0), 3)`

**What the function receives** (its inputs, called *parameters*):
- `cell` — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- `action` — type: **int (whole number)**; value: `3`

**Now each line runs, top to bottom:**

```python
d_row, d_col = ACTIONS[action]
```
Evaluated inside-out (the main pieces):
1. `ACTIONS` → *list of 4 items (each a tuple)*: `[(-1, 0), (1, 0), (0, -1), (0, 1)]`
2. `action` → *int (whole number)*: `3`
3. `ACTIONS[action]` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `d_row` holds** — type: **int (whole number)**; value: `0`
- **Now `d_col` holds** — type: **int (whole number)**; value: `1`
- **Why (as in the earlier call):** `ACTIONS[action]` looks up the tuple at that position in the list; unpacking splits it into the row change and the column change.

```python
row = cell[0] + d_row                 # where the move WOULD put us
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `cell[0]` → *int (whole number)*: `0`
3. `d_row` → *int (whole number)*: `0`
- **Now `row` holds** — type: **int (whole number)**; value: `0`
- **Why (as in the earlier call):** The row the agent *would* move to. This is only a candidate; we have not yet checked whether it is allowed.

```python
col = cell[1] + d_col
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `cell[1]` → *int (whole number)*: `0`
3. `d_col` → *int (whole number)*: `1`
- **Now `col` holds** — type: **int (whole number)**; value: `1`
- **Why (as in the earlier call):** The candidate column.


#### …the same call, continued: `step((0, 0), 3)`

```python
if row < 0 or row >= ROWS or col < 0 or col >= COLS or walls[row, col] == 1:
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `0`
2. `row < 0` → *bool (True/False)*: `False`
3. `ROWS` → *int (whole number)*: `5`
4. `row >= ROWS` → *bool (True/False)*: `False`
5. `col` → *int (whole number)*: `1`
6. `col < 0` → *bool (True/False)*: `False`
7. `COLS` → *int (whole number)*: `5`
8. `col >= COLS` → *bool (True/False)*: `False`
- **Why (as in the earlier call):** A chain of questions joined by `or`; the whole thing is `True` if **any** is true. The first four check the grid edges. The last, `walls[row, col] == 1`, checks for a wall. Python stops at the first true part, so the wall lookup is never reached for an out-of-range position (which would crash). Here the move is allowed, so the whole condition is `False`.

```python
if (row, col) == GOAL:
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `0`
2. `col` → *int (whole number)*: `1`
3. `(row, col)` → *tuple of 2 items (each an int)*: `(0, 1)`
4. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
5. `(row, col) == GOAL` → *bool (True/False)*: `False`
- **Why:** Builds a tuple from the candidate and compares it with the goal tuple. Tuples compare item by item.

```python
return (row, col), STEP_REWARD, False
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `0`
2. `col` → *int (whole number)*: `1`
3. `(row, col)` → *tuple of 2 items (each an int)*: `(0, 1)`
4. `STEP_REWARD` → *int (whole number)*: `-1`
5. `((row, col), STEP_REWARD, False)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Why:** The ordinary case: the candidate becomes the new position, with the step reward and `False`. Returning three values with commas makes a tuple, which the caller can unpack.

**The function hands back** (its *return value*) — type: **tuple of 3 items**; value: `((0, 1), -1, False)`


#### Inside the function: one concrete call, `step((4, 3), 3)`

**What the function receives** (its inputs, called *parameters*):
- `cell` — type: **tuple of 2 items (each an int)**; value: `(4, 3)`
- `action` — type: **int (whole number)**; value: `3`

**Now each line runs, top to bottom:**

```python
d_row, d_col = ACTIONS[action]
```
Evaluated inside-out (the main pieces):
1. `ACTIONS` → *list of 4 items (each a tuple)*: `[(-1, 0), (1, 0), (0, -1), (0, 1)]`
2. `action` → *int (whole number)*: `3`
3. `ACTIONS[action]` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `d_row` holds** — type: **int (whole number)**; value: `0`
- **Now `d_col` holds** — type: **int (whole number)**; value: `1`
- **Why (as in the earlier call):** `ACTIONS[action]` looks up the tuple at that position in the list; unpacking splits it into the row change and the column change.

```python
row = cell[0] + d_row                 # where the move WOULD put us
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(4, 3)`
2. `cell[0]` → *int (whole number)*: `4`
3. `d_row` → *int (whole number)*: `0`
- **Now `row` holds** — type: **int (whole number)**; value: `4`
- **Why (as in the earlier call):** The row the agent *would* move to. This is only a candidate; we have not yet checked whether it is allowed.

```python
col = cell[1] + d_col
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(4, 3)`
2. `cell[1]` → *int (whole number)*: `3`
3. `d_col` → *int (whole number)*: `1`
- **Now `col` holds** — type: **int (whole number)**; value: `4`
- **Why (as in the earlier call):** The candidate column.


#### …the same call, continued: `step((4, 3), 3)`

```python
if row < 0 or row >= ROWS or col < 0 or col >= COLS or walls[row, col] == 1:
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `4`
2. `row < 0` → *bool (True/False)*: `False`
3. `ROWS` → *int (whole number)*: `5`
4. `row >= ROWS` → *bool (True/False)*: `False`
5. `col` → *int (whole number)*: `4`
6. `col < 0` → *bool (True/False)*: `False`
7. `COLS` → *int (whole number)*: `5`
8. `col >= COLS` → *bool (True/False)*: `False`
- **Why (as in the earlier call):** A chain of questions joined by `or`; the whole thing is `True` if **any** is true. The first four check the grid edges. The last, `walls[row, col] == 1`, checks for a wall. Python stops at the first true part, so the wall lookup is never reached for an out-of-range position (which would crash). Here the move is allowed, so the whole condition is `False`.

```python
if (row, col) == GOAL:
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `4`
2. `col` → *int (whole number)*: `4`
3. `(row, col)` → *tuple of 2 items (each an int)*: `(4, 4)`
4. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
5. `(row, col) == GOAL` → *bool (True/False)*: `True`
- **Why (as in the earlier call):** Builds a tuple from the candidate and compares it with the goal tuple. Tuples compare item by item.

```python
return (row, col), GOAL_REWARD, True     # done = True ends the episode
```
Evaluated inside-out (the main pieces):
1. `row` → *int (whole number)*: `4`
2. `col` → *int (whole number)*: `4`
3. `(row, col)` → *tuple of 2 items (each an int)*: `(4, 4)`
4. `GOAL_REWARD` → *int (whole number)*: `10`
5. `((row, col), GOAL_REWARD, True)` → *tuple of 3 items*: `((4, 4), 10, True)`
- **Why:** Reaching the goal returns `True` for done.

**The function hands back** (its *return value*) — type: **tuple of 3 items**; value: `((4, 4), 10, True)`


### Step 6.5: type this

```python
print(step((0, 0), 3))                    # right from the start: a normal move
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `step((0, 0), 3)` → *tuple of 3 items*: `((0, 1), -1, False)`

**Prints:**
```text
((0, 1), -1, False)
```

In [ ]:
# Step 6.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.6: type this

```python
print(step((0, 0), 0))                    # up from the top-left corner: off the grid
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `step((0, 0), 0)` → *tuple of 3 items*: `((0, 0), -5, False)`

**Prints:**
```text
((0, 0), -5, False)
```

In [ ]:
# Step 6.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.7: type this

```python
print(step((0, 1), 1))                    # down into the wall at (1, 1)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `step((0, 1), 1)` → *tuple of 3 items*: `((0, 1), -5, False)`

**Prints:**
```text
((0, 1), -5, False)
```

In [ ]:
# Step 6.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6.8: type this

```python
print(step((4, 3), 3))                    # right into the goal
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `step((4, 3), 3)` → *tuple of 3 items*: `((4, 4), 10, True)`

**Prints:**
```text
((4, 4), 10, True)
```

In [ ]:
# Step 6.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 6 complete: everything it printed, and what it means

```text
((0, 1), -1, False)
((0, 0), -5, False)
((0, 1), -5, False)
((4, 4), 10, True)
```

Right from the start gives `((0, 1), -1, False)`: a normal move. Up from the top-left corner is off the grid, so the agent stays at `(0, 0)` and gets `-5`. Down from `(0, 1)` runs into the wall at `(1, 1)`: stays at `(0, 1)`, `-5`. Right from `(4, 3)` enters the goal: `((4, 4), 10, True)`.

## Step 7: The true shortest path (only for checking our work)

**The idea first.** The learning agent never sees the maze. But *we* can compute the true answer, so we can later tell whether the agent found the best route. The method is **breadth-first search** (BFS): explore outward from the start one ring of cells at a time. The first time you reach the goal, it is by a shortest route.

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 7.1: type this

```python
def bfs():
    dist = {START: 0}                     # steps needed to reach each cell found so far
    queue = deque([START])                # cells waiting to be explored
    while queue:                          # while the line is not empty
        cell = queue.popleft()            # take the cell at the front
        if cell == GOAL:
            return dist[cell]
        for action in range(4):
            nxt, _, _ = step(cell, action)    # "_" = a value we do not need
            if nxt not in dist:           # a cell we have not reached before
                dist[nxt] = dist[cell] + 1
                queue.append(nxt)         # explore it later
    return None
```

- **Creates** `bfs` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Breadth-first search: finds the length of the shortest route by exploring outward in rings. No parameters: it uses the global maze. (If the loop ends without finding the goal, the last line returns `None`, meaning 'unreachable'.)

In [ ]:
# Step 7.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `bfs()`

**Now each line runs, top to bottom:**

```python
dist = {START: 0}                     # steps needed to reach each cell found so far
```
Evaluated inside-out (the main pieces):
1. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `dist` holds** — type: **dict with 1 entry**; value: `{(0, 0): 0}`
- **Why:** A **dictionary** from cell (tuple) to the number of steps needed to reach it. The start is 0 steps from itself.

```python
queue = deque([START])                # cells waiting to be explored
```
Evaluated inside-out (the main pieces):
1. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `[START]` → *list of 1 item (each a tuple)*: `[(0, 0)]`
- **Now `queue` holds** — type: **deque (a waiting line) with 1 item**; value: `[(0, 0)]`
- **Why:** A waiting line of cells still to explore, starting with just the start.

```python
while queue:                          # while the line is not empty
```
Evaluated inside-out (the main pieces):
1. `queue` → *deque (a waiting line) with 1 item*: `[(0, 0)]`
- **Why:** A list or deque counts as `True` when it has items in it and `False` when empty, so this repeats until nothing is left to explore.

```python
cell = queue.popleft()            # take the cell at the front
```
Evaluated inside-out (the main pieces):
1. `queue` → *deque (a waiting line) with 1 item*: `[(0, 0)]`
- **Now `queue` holds** — type: **deque (a waiting line) with 0 items**; value: `[]`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Why:** Takes the cell at the front of the line (the one waiting longest) and removes it.


#### …the same call, continued: `bfs()`

```python
if cell == GOAL:
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
3. `cell == GOAL` → *bool (True/False)*: `False`
- **Why:** Tuple comparison. The first time we pull the goal off the queue we have the shortest route, because cells are explored in order of distance.

```python
for action in range(4):
```
Evaluated inside-out (the main pieces):
1. `range(4)` → *range*: `range(0, 4)`
- **Now `action` holds** — type: **int (whole number)**; value: `0`
- **Why:** Tries all four actions from this cell.

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `action` → *int (whole number)*: `0`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 0), -5, False)`
- **Now `nxt` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Now `_` holds** — type: **bool (True/False)**; value: `False`
- **Why:** `step` returns three values; we keep only the first (the next cell). The underscores are throwaway names for values we do not need.

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `dist` → *dict with 1 entry*: `{(0, 0): 0}`
3. `nxt not in dist` → *bool (True/False)*: `False`
- **Why:** `in` on a dictionary asks 'is this a key?'. A blocked move returns the *same* cell, which is already a key, so walls and edges are skipped automatically.


#### …the same call, continued: `bfs()`

```python
for action in range(4):
```
- **Now `action` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `action` → *int (whole number)*: `1`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 0), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `dist` → *dict with 1 entry*: `{(0, 0): 0}`
3. `nxt not in dist` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for action in range(4):
```
- **Now `action` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `action` → *int (whole number)*: `2`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 0), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `bfs()`

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `dist` → *dict with 1 entry*: `{(0, 0): 0}`
3. `nxt not in dist` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for action in range(4):
```
- **Now `action` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `action` → *int (whole number)*: `3`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Now `nxt` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `dist` → *dict with 1 entry*: `{(0, 0): 0}`
3. `nxt not in dist` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `bfs()`

```python
dist[nxt] = dist[cell] + 1
```
Evaluated inside-out (the main pieces):
1. `dist` → *dict with 1 entry*: `{(0, 0): 0}`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `dist[cell]` → *int (whole number)*: `0`
4. `dist[cell] + 1` → *int (whole number)*: `1`
5. `nxt` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `dist` holds** — type: **dict with 2 entries**; value: `{(0, 0): 0, (0, 1): 1}`
- **Why:** A cell we have not reached before is one step further than the cell we came from. A new dictionary entry.

```python
queue.append(nxt)         # explore it later
```
Evaluated inside-out (the main pieces):
1. `queue` → *deque (a waiting line) with 0 items*: `[]`
2. `nxt` → *tuple of 2 items (each an int)*: `(0, 1)`
3. `queue.append(nxt)` → *None (nothing)*: `None`
- **Now `queue` holds** — type: **deque (a waiting line) with 1 item**; value: `[(0, 1)]`
- **Why:** Joins the back of the line so its neighbours are explored later.

```python
for action in range(4):
```
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
while queue:                          # while the line is not empty
```
Evaluated inside-out (the main pieces):
1. `queue` → *deque (a waiting line) with 1 item*: `[(0, 1)]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `bfs()`

```python
cell = queue.popleft()            # take the cell at the front
```
Evaluated inside-out (the main pieces):
1. `queue` → *deque (a waiting line) with 1 item*: `[(0, 1)]`
- **Now `queue` holds** — type: **deque (a waiting line) with 0 items**; value: `[]`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if cell == GOAL:
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
3. `cell == GOAL` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for action in range(4):
```
Evaluated inside-out (the main pieces):
1. `range(4)` → *range*: `range(0, 4)`
- **Now `action` holds** — type: **int (whole number)**; value: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `action` → *int (whole number)*: `0`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 1), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `bfs()`

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `dist` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
3. `nxt not in dist` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for action in range(4):
```
- **Now `action` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `action` → *int (whole number)*: `1`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 1), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `dist` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
3. `nxt not in dist` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for action in range(4):
```
- **Now `action` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `bfs()`

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `action` → *int (whole number)*: `2`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 0), -1, False)`
- **Now `nxt` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `dist` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
3. `nxt not in dist` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for action in range(4):
```
- **Now `action` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
nxt, _, _ = step(cell, action)    # "_" = a value we do not need
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `action` → *int (whole number)*: `3`
3. `step(cell, action)` → *tuple of 3 items*: `((0, 2), -1, False)`
- **Now `nxt` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 2)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `bfs()`

```python
if nxt not in dist:           # a cell we have not reached before
```
Evaluated inside-out (the main pieces):
1. `nxt` → *tuple of 2 items (each an int)*: `(0, 2)`
2. `dist` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
3. `nxt not in dist` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
dist[nxt] = dist[cell] + 1
```
Evaluated inside-out (the main pieces):
1. `dist` → *dict with 2 entries*: `{(0, 0): 0, (0, 1): 1}`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
3. `dist[cell]` → *int (whole number)*: `1`
4. `dist[cell] + 1` → *int (whole number)*: `2`
5. `nxt` → *tuple of 2 items (each an int)*: `(0, 2)`
- **Now `dist` holds** — type: **dict with 3 entries**; value: `{(0, 0): 0, (0, 1): 1, (0, 2): 2}`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
queue.append(nxt)         # explore it later
```
Evaluated inside-out (the main pieces):
1. `queue` → *deque (a waiting line) with 0 items*: `[]`
2. `nxt` → *tuple of 2 items (each an int)*: `(0, 2)`
3. `queue.append(nxt)` → *None (nothing)*: `None`
- **Now `queue` holds** — type: **deque (a waiting line) with 1 item**; value: `[(0, 2)]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 7.2: type this

```python
print(bfs())
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `bfs()` → *int (whole number)*: `12`

**Prints:**
```text
12
```

In [ ]:
# Step 7.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 7 complete: everything it printed, and what it means

```text
12
```

`12`: the shortest route is 12 steps. Remember this number. The straight-line "Manhattan" distance from `(0, 0)` to `(4, 4)` is only `4 + 4 = 8`; the walls make the true route longer, which is why this maze is interesting.

---
# Part C: Q-learning

First what the agent is trying to maximise, then the table, then the rule that fills it in.

## Step 8: The return: what the agent is really maximising

**The idea first.** The agent does not care only about the next reward; it cares about the *total from now on*, with later rewards counted slightly less (the **discount factor**, gamma). That total is the **return**. Here we compute the return of the best route by hand, because later the learned table must agree with it.

This step has **4 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 8.1: type this

```python
rewards = [-1] * 11 + [10]                # the best route: 11 ordinary steps, then the goal step
```

- **Creates `rewards`** — type: **list of 12 items (each an int)**; value: `[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 10]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[-1]` → *list of 1 item (each an int)*: `[-1]`
2. `[-1] * 11` → *list of 11 items (each an int)*: `[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]`

- **Why / what it means:** `[-1] * 11` is a list of eleven -1s; `+ [10]` joins a one-item list holding the final prize. So `rewards` is a **list of 12 ints**: the reward at each step of the best route.

In [ ]:
# Step 8.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.2: type this

```python
gamma = 0.95
total = 0.0
```

- **Creates `gamma`** — type: **float (decimal number)**; value: `0.95`

- **Why / what it means:** The discount factor, a `float`: a reward one step later is worth 95% of one now.

- **Creates `total`** — type: **float (decimal number)**; value: `0.0`

- **Why / what it means:** A running total, starting at zero. A float, because we will add decimals to it.

In [ ]:
# Step 8.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.3: type this

```python
for k, r in enumerate(rewards):           # k = 0, 1, 2, ... ; r = that step's reward
    total += (gamma ** k) * r             # a reward k steps away is worth gamma**k of its value
```

- **Creates `k`** — type: **int (whole number)**; value: `11`

- **Creates `r`** — type: **int (whole number)**; value: `10`

- **Creates `total`** — type: **float (decimal number)**; value: `-2.935997231706202`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `rewards` → *list of 12 items (each an int)*: `[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 10]`
2. `enumerate(rewards)` → *enumerate*: `<enumerate object at 0x7fde8c4b5da0>`
3. `gamma` → *float (decimal number)*: `0.95` *(evaluated 12 times; the first is shown)*
4. `k` → *int (whole number)*: `0` *(evaluated 12 times; the first is shown)*
5. `gamma ** k` → *float (decimal number)*: `1.0` *(evaluated 12 times; the first is shown)*
6. `r` → *int (whole number)*: `-1` *(evaluated 12 times; the first is shown)*
7. `gamma ** k * r` → *float (decimal number)*: `-1.0` *(evaluated 12 times; the first is shown)*

- **Why / what it means:** The loop body. `gamma ** k` is gamma raised to the power `k`: 1 for the first reward, 0.95 for the next, 0.9025 for the next, and so on. Multiplying by `r` discounts that reward, and `+=` adds it to the total. The loop runs 12 times; the first pass is shown.

In [ ]:
# Step 8.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8.4: type this

```python
print(round(total, 3))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `total` → *float (decimal number)*: `-2.935997231706202`
2. `round(total, 3)` → *float (decimal number)*: `-2.936`

**Prints:**
```text
-2.936
```

In [ ]:
# Step 8.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 8 complete: everything it printed, and what it means

```text
-2.936
```

`-2.936`. The eleven penalties add up to about `-8.62` once discounted, and the goal prize `10 x 0.95**11 = 5.69` cancels about two-thirds of that damage. Hold on to **-2.936**: it is the true value of starting at the start and walking the best route. Q-learning will rediscover this exact number without anyone telling it. A very useful way to read any Q-value: *it is the discounted return of the best plan from that situation.*

**Try this yourself.** Change `gamma` to `0.5`. What do you predict, and why is the goal's prize now almost worthless from the start cell?

## Step 9: The Q-table, and two ways to look at it

**The idea first.** **Q(s, a)** is the agent's estimate of the return from taking action `a` in cell `s` and playing well afterwards. We store it in an array: one row per cell, one column per action. Two display helpers let us *see* the table, which is how you understand it.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 9.1: type this

```python
Q = np.zeros((n_states, 4))               # 25 rows (cells) x 4 columns (actions)
```

- **Creates `Q`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `n_states` → *int (whole number)*: `25`
2. `(n_states, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`

- **Why / what it means:** The **Q-table**: an array of shape `(25, 4)`, 25 rows (one per cell) and 4 columns (one per action), all 0.0. The pair of numbers in `(n_states, 4)` is itself a tuple giving the shape.

In [ ]:
# Step 9.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.2: type this

```python
def show_values(Q):
    values = Q.max(axis=1)                # best number in each row: 25 numbers
    for r in range(ROWS):
        line = ""
        for c in range(COLS):
            if walls[r, c] == 1:
                line += "  ###  "
            else:
                line += f"{values[state_of((r, c))]:6.1f} "    # 6 wide, 1 decimal
        print(line)
```

- **Creates** `show_values` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Prints the best value in each cell as a grid. It only *displays*; it changes nothing.

In [ ]:
# Step 9.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.3: type this

```python
ARROWS = ["^", "v", "<", ">"]
```

- **Creates `ARROWS`** — type: **list of 4 items (each a str)**; value: `['^', 'v', '<', '>']`

- **Why / what it means:** A list of four one-character strings, in the same order as the actions, so `ARROWS[2]` is the arrow for 'left'.

In [ ]:
# Step 9.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.4: type this

```python
def show_policy(Q):
    for r in range(ROWS):
        line = ""
        for c in range(COLS):
            s = state_of((r, c))
            if walls[r, c] == 1:
                ch = "#"
            elif (r, c) == GOAL:
                ch = "G"
            elif not Q[s].any():          # row is all zeros: nothing learned yet
                ch = "?"
            else:
                ch = ARROWS[int(np.argmax(Q[s]))]
            line += ch + " "
        print(line)
```

- **Creates** `show_policy` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Prints an arrow for each cell's best action. `np.argmax(Q[s])` gives the position of the largest number in the row, which is the action number, and indexing `ARROWS` with it gives the arrow.

In [ ]:
# Step 9.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.5: type this

```python
print(Q.shape)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `Q.shape` → *tuple of 2 items (each an int)*: `(25, 4)`

**Prints:**
```text
(25, 4)
```

In [ ]:
# Step 9.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.6: type this

```python
show_values(Q)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `show_values(Q)` → *None (nothing)*: `None`

**Prints:**
```text
   0.0    0.0    0.0    0.0    0.0 
  ###    ###     0.0   ###     0.0 
   0.0    0.0    0.0    0.0    0.0 
   0.0   ###    ###    ###    ###  
   0.0    0.0    0.0    0.0    0.0 
```

- **Why / what it means:** Calls the function. Every value is 0.0 because nothing has been learned.

In [ ]:
# Step 9.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9.7: type this

```python
show_policy(Q)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `show_policy(Q)` → *None (nothing)*: `None`

**Prints:**
```text
? ? ? ? ? 
# # ? # ? 
? ? ? ? ? 
? # # # # 
? ? ? ? G 
```

- **Why / what it means:** Every cell shows `?`, because each row is all zeros: no preference yet.

In [ ]:
# Step 9.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 9 complete: everything it printed, and what it means

```text
(25, 4)
   0.0    0.0    0.0    0.0    0.0 
  ###    ###     0.0   ###     0.0 
   0.0    0.0    0.0    0.0    0.0 
   0.0   ###    ###    ###    ###  
   0.0    0.0    0.0    0.0    0.0 
? ? ? ? ? 
# # ? # ? 
? ? ? ? ? 
? # # # # 
? ? ? ? G 
```

`(25, 4)`: 25 rows, 4 columns. The value map is all `0.0` and the policy map is all `?` (with `#` for walls and `G` for the goal). This is the starting point, a blank page.

## Step 10: Choosing an action: explore or exploit

**The idea first.** If the agent always did what looks best now, it would never find out whether something untried is better. So with probability **epsilon** it acts randomly (**explores**), otherwise it takes the best-known action (**exploits**). This is *epsilon-greedy*.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 10.1: type this

```python
def choose_action(Q, s, epsilon):
    if random.random() < epsilon:         # explore with probability epsilon
        return random.randrange(4)
    row = Q[s]
    best = np.flatnonzero(row == row.max())    # positions of all tied-best actions
    return int(random.choice(best))       # pick one of them at random
```

- **Creates** `choose_action` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Parameters: the table `Q` (array), the cell's row number `s` (int) and `epsilon` (float). Returns an action number, 0 to 3.

In [ ]:
# Step 10.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `choose_action(np.zeros((25, 4)), 12, 0.0)`

**What the function receives** (its inputs, called *parameters*):
- `Q` — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`
- `s` — type: **int (whole number)**; value: `12`
- `epsilon` — type: **float (decimal number)**; value: `0.0`

**Now each line runs, top to bottom:**

```python
if random.random() < epsilon:         # explore with probability epsilon
```
Evaluated inside-out (the main pieces):
1. `random.random()` → *float (decimal number)*: `0.80317946927987`
2. `epsilon` → *float (decimal number)*: `0.0`
3. `random.random() < epsilon` → *bool (True/False)*: `False`
- **Why:** `random.random()` is a float in [0, 1). If it is below `epsilon` we explore. With epsilon 0.0 it is never below, so this is `False`.

```python
row = Q[s]
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `s` → *int (whole number)*: `12`
- **Now `row` holds** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 0., 0.]`
- **Why:** The four numbers for this cell: a 1D array of floats. (This is a *view* of the table, not a copy.)


#### …the same call, continued: `choose_action(np.zeros((25, 4)), 12, 0.0)`

```python
best = np.flatnonzero(row == row.max())    # positions of all tied-best actions
```
Evaluated inside-out (the main pieces):
1. `row` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
2. `row.max()` → *float (decimal number)*: `0.0`
3. `row == row.max()` → *numpy array, shape (4,), holding True/False values*: `[True, True, True, True]`
- **Now `best` holds** — type: **numpy array, shape (4,), holding whole numbers**; value: `[0, 1, 2, 3]`
- **Why:** Three steps, inside-out: `row.max()` is the largest number; `row == row.max()` compares every item with it and gives an array of four `bool`s; `np.flatnonzero` returns the *positions* of the `True`s. On an untrained row all four tie, so `best` is `[0, 1, 2, 3]`.

```python
return int(random.choice(best))       # pick one of them at random
```
Evaluated inside-out (the main pieces):
1. `best` → *numpy array, shape (4,), holding whole numbers*: `[0, 1, 2, 3]`
2. `random.choice(best)` → *numpy integer (a whole number)*: `3`
3. `int(random.choice(best))` → *int (whole number)*: `3`
- **Why:** `random.choice` picks one item from the array; `int(...)` converts numpy's integer type into a plain Python `int`. Picking randomly among ties avoids a hidden bias toward action 0.

**The function hands back** (its *return value*) — type: **int (whole number)**; value: `3`


### Step 10.2: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable randomness for the tests.

In [ ]:
# Step 10.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.3: type this

```python
Q = np.zeros((n_states, 4))
```

- **Creates `Q`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `n_states` → *int (whole number)*: `25`
2. `(n_states, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`

- **Why / what it means:** A fresh all-zero table.

In [ ]:
# Step 10.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.4: type this

```python
s = state_of((2, 2))
```

- **Creates `s`** — type: **int (whole number)**; value: `12`

- **Why / what it means:** The row number for cell `(2, 2)`: `2 x 5 + 2 = 12`.

In [ ]:
# Step 10.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.5: type this

```python
print([choose_action(Q, s, 0.0) for _ in range(8)])    # all tied: a random mix
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[choose_action(Q, s, 0.0) for _ in range(8)]` → *list of 8 items (each an int)*: `[3, 3, 2, 2, 1, 2, 0, 2]`

**Prints:**
```text
[3, 3, 2, 2, 1, 2, 0, 2]
```

In [ ]:
# Step 10.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.6: type this

```python
Q[s, 3] = 5.0
```

- **Changes `Q` in place** — type: **numpy array, shape (25, 4), holding decimal numbers**; value now: `100 numbers; 1 are non-zero; smallest 0, largest 5`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `s` → *int (whole number)*: `12`

- **Why / what it means:** Writes into one box: row 12, column 3 (the action 'right'). Now one action clearly beats the other three.

In [ ]:
# Step 10.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.7: type this

```python
print([choose_action(Q, s, 0.0) for _ in range(8)])    # one clear best: always action 3
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[choose_action(Q, s, 0.0) for _ in range(8)]` → *list of 8 items (each an int)*: `[3, 3, 3, 3, 3, 3, 3, 3]`

**Prints:**
```text
[3, 3, 3, 3, 3, 3, 3, 3]
```

In [ ]:
# Step 10.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10.8: type this

```python
print([choose_action(Q, s, 1.0) for _ in range(8)])    # always explore
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[choose_action(Q, s, 1.0) for _ in range(8)]` → *list of 8 items (each an int)*: `[0, 3, 0, 2, 2, 0, 1, 1]`

**Prints:**
```text
[0, 3, 0, 2, 2, 0, 1, 1]
```

In [ ]:
# Step 10.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 10 complete: everything it printed, and what it means

```text
[3, 3, 2, 2, 1, 2, 0, 2]
[3, 3, 3, 3, 3, 3, 3, 3]
[0, 3, 0, 2, 2, 0, 1, 1]
```

With an all-zero row, all four actions tie, so the choices are a mix of 0, 1, 2 and 3. After `Q[s, 3] = 5.0`, action 3 is the unique best, so every choice is `3`. With `epsilon = 1.0` the table is ignored and the choices are random again.

## Step 11: One update, printed: the ripple from the goal

**The idea first.** This is the whole of Q-learning. `trace_step` performs one update and prints every number, so you can see exactly what it does. Then we play four updates by hand along the bottom row.

This step has **8 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 11.1: type this

```python
ALPHA = 0.5                               # learning rate
GAMMA = 0.95                              # discount factor
```

- **Creates `ALPHA`** — type: **float (decimal number)**; value: `0.5`

- **Why / what it means:** The learning rate, a float: close half the gap between the old belief and the new target.

- **Creates `GAMMA`** — type: **float (decimal number)**; value: `0.95`

- **Why / what it means:** The discount factor.

In [ ]:
# Step 11.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.2: type this

```python
def trace_step(Q, cell, action):
    s = state_of(cell)
    next_cell, reward, done = step(cell, action)
    s_next = state_of(next_cell)
    old = Q[s, action]                    # the one box we will change
    best_next = 0.0 if done else Q[s_next].max()    # best belief about where we landed
    target = reward + GAMMA * best_next   # what the box should be, given what we saw
    new = old + ALPHA * (target - old)    # move ALPHA of the way toward it
    Q[s, action] = new                    # the ONLY write to the table
    print(f"at {cell} (row {s}) action {NAMES[action]} -> lands {next_cell} (row {s_next}), reward {reward}")
    print(f"   old {old:.3f} | best_next {best_next:.3f} | target {reward} + {GAMMA} * {best_next:.3f} = {target:.3f} | new {new:.3f}")
```

- **Creates** `trace_step` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** One Q-learning update, printed. Parameters: the table, a position (tuple) and an action (int). It changes **one box** of `Q` in place.

In [ ]:
# Step 11.2: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `trace_step(np.zeros((25, 4)), (4, 3), 3)`

**What the function receives** (its inputs, called *parameters*):
- `Q` — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`
- `cell` — type: **tuple of 2 items (each an int)**; value: `(4, 3)`
- `action` — type: **int (whole number)**; value: `3`

**Now each line runs, top to bottom:**

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(4, 3)`
- **Now `s` holds** — type: **int (whole number)**; value: `23`
- **Why:** The row number of the cell we start in (an int).

```python
next_cell, reward, done = step(cell, action)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(4, 3)`
2. `action` → *int (whole number)*: `3`
3. `step(cell, action)` → *tuple of 3 items*: `((4, 4), 10, True)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(4, 4)`
- **Now `reward` holds** — type: **int (whole number)**; value: `10`
- **Now `done` holds** — type: **bool (True/False)**; value: `True`
- **Why:** Calls the environment. It returns three values; unpacking gives the landing cell (tuple), the reward (int) and `done` (bool).

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(4, 4)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `24`
- **Why:** The row number of the cell we *landed* in.


#### …the same call, continued: `trace_step(np.zeros((25, 4)), (4, 3), 3)`

```python
old = Q[s, action]                    # the one box we will change
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `s` → *int (whole number)*: `23`
3. `action` → *int (whole number)*: `3`
- **Now `old` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** Reads one box: row `s`, column `action`. It is a numpy float. This comes from the row we **left**.

```python
best_next = 0.0 if done else Q[s_next].max()    # best belief about where we landed
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `True`
- **Now `best_next` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** If the episode is over there is no future, so 0.0. Otherwise it is the largest number in the row we **arrived at**: 'how good is it to be where I landed?'.

```python
target = reward + GAMMA * best_next   # what the box should be, given what we saw
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `10`
2. `GAMMA` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `GAMMA * best_next` → *float (decimal number)*: `0.0`
- **Now `target` holds** — type: **float (decimal number)**; value: `10.0`
- **Why:** The new opinion: what we just got, plus the discounted future. Multiplication happens before addition.

```python
new = old + ALPHA * (target - old)    # move ALPHA of the way toward it
```
Evaluated inside-out (the main pieces):
1. `old` → *float (decimal number)*: `0.0`
2. `ALPHA` → *float (decimal number)*: `0.5`
3. `target` → *float (decimal number)*: `10.0`
4. `target - old` → *float (decimal number)*: `10.0`
5. `ALPHA * (target - old)` → *float (decimal number)*: `5.0`
- **Now `new` holds** — type: **float (decimal number)**; value: `5.0`
- **Why:** `target - old` is the error; we correct a fraction ALPHA of it. With ALPHA 0.5 we move halfway.

```python
Q[s, action] = new                    # the ONLY write to the table
```
Evaluated inside-out (the main pieces):
1. `new` → *float (decimal number)*: `5.0`
2. `s` → *int (whole number)*: `23`
3. `action` → *int (whole number)*: `3`
4. `(s, action)` → *tuple of 2 items (each an int)*: `(23, 3)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest 0, largest 5`
- **Why:** The only write to the table: one box.

```python
print(f"at {cell} (row {s}) action {NAMES[action]} -> lands {next_cell} (row {s_next}), reward {reward}")
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(4, 3)`
2. `s` → *int (whole number)*: `23`
3. `NAMES` → *list of 4 items (each a str)*: `['up', 'down', 'left', 'right']`
4. `action` → *int (whole number)*: `3`
5. `NAMES[action]` → *str (text, 5 characters)*: `'right'`
6. `next_cell` → *tuple of 2 items (each an int)*: `(4, 4)`
7. `s_next` → *int (whole number)*: `24`
8. `reward` → *int (whole number)*: `10`
- **Why:** An f-string: anything in braces is replaced by its value. Prints the movement.

```python
print(f"   old {old:.3f} | best_next {best_next:.3f} | target {reward} + {GAMMA} * {best_next:.3f} = {target:.3f} | new {new:.3f}")
```
Evaluated inside-out (the main pieces):
1. `old` → *float (decimal number)*: `0.0`
2. `best_next` → *float (decimal number)*: `0.0`
3. `reward` → *int (whole number)*: `10`
4. `GAMMA` → *float (decimal number)*: `0.95`
5. `target` → *float (decimal number)*: `10.0`
6. `new` → *float (decimal number)*: `5.0`
- **Why:** `{old:.3f}` formats a decimal with 3 digits after the point.

**The function hands back** (its *return value*) — type: **None (nothing)**; value: `None`


### Step 11.3: type this

```python
Q = np.zeros((n_states, 4))
```

- **Creates `Q`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `n_states` → *int (whole number)*: `25`
2. `(n_states, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`

- **Why / what it means:** A fresh table for this demonstration.

In [ ]:
# Step 11.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.4: type this

```python
trace_step(Q, (4, 3), 3)                  # right, into the goal
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `trace_step(Q, (4, 3), 3)` → *None (nothing)*: `None`

**Prints:**
```text
at (4, 3) (row 23) action right -> lands (4, 4) (row 24), reward 10
   old 0.000 | best_next 0.000 | target 10 + 0.95 * 0.000 = 10.000 | new 5.000
```

- **Why / what it means:** Places the agent at `(4, 3)`, next to the goal, and presses right (action 3). It is the first of four updates; read the printed numbers in the output below.

In [ ]:
# Step 11.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.5: type this

```python
trace_step(Q, (4, 2), 3)                  # right, onto the cell we just updated
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest 0, largest 5`
2. `trace_step(Q, (4, 2), 3)` → *None (nothing)*: `None`

**Prints:**
```text
at (4, 2) (row 22) action right -> lands (4, 3) (row 23), reward -1
   old 0.000 | best_next 5.000 | target -1 + 0.95 * 5.000 = 3.750 | new 1.875
```

- **Why / what it means:** Moves the agent one cell left of the previous test. It lands on `(4, 3)`, which has just learned something, so `best_next` is no longer zero. That is the ripple.

In [ ]:
# Step 11.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.6: type this

```python
trace_step(Q, (4, 1), 3)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 2 are non-zero; smallest 0, largest 5`
2. `trace_step(Q, (4, 1), 3)` → *None (nothing)*: `None`

**Prints:**
```text
at (4, 1) (row 21) action right -> lands (4, 2) (row 22), reward -1
   old 0.000 | best_next 1.875 | target -1 + 0.95 * 1.875 = 0.781 | new 0.391
```

- **Why / what it means:** One more cell back: it lands on `(4, 2)`, whose value is small but positive.

In [ ]:
# Step 11.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.7: type this

```python
trace_step(Q, (4, 3), 3)                  # the goal step again
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 3 are non-zero; smallest 0, largest 5`
2. `trace_step(Q, (4, 3), 3)` → *None (nothing)*: `None`

**Prints:**
```text
at (4, 3) (row 23) action right -> lands (4, 4) (row 24), reward 10
   old 5.000 | best_next 0.000 | target 10 + 0.95 * 0.000 = 10.000 | new 7.500
```

- **Why / what it means:** The goal step again. The old value is now 5.0, so the target of 10 pulls it halfway up, to 7.5.

In [ ]:
# Step 11.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11.8: type this

```python
show_values(Q)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 3 are non-zero; smallest 0, largest 7.5`
2. `show_values(Q)` → *None (nothing)*: `None`

**Prints:**
```text
   0.0    0.0    0.0    0.0    0.0 
  ###    ###     0.0   ###     0.0 
   0.0    0.0    0.0    0.0    0.0 
   0.0   ###    ###    ###    ###  
   0.0    0.4    1.9    7.5    0.0 
```

- **Why / what it means:** Shows the whole table's best values. Only three cells have non-zero numbers: the three we updated.

In [ ]:
# Step 11.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 11 complete: everything it printed, and what it means

```text
at (4, 3) (row 23) action right -> lands (4, 4) (row 24), reward 10
   old 0.000 | best_next 0.000 | target 10 + 0.95 * 0.000 = 10.000 | new 5.000
at (4, 2) (row 22) action right -> lands (4, 3) (row 23), reward -1
   old 0.000 | best_next 5.000 | target -1 + 0.95 * 5.000 = 3.750 | new 1.875
at (4, 1) (row 21) action right -> lands (4, 2) (row 22), reward -1
   old 0.000 | best_next 1.875 | target -1 + 0.95 * 1.875 = 0.781 | new 0.391
at (4, 3) (row 23) action right -> lands (4, 4) (row 24), reward 10
   old 5.000 | best_next 0.000 | target 10 + 0.95 * 0.000 = 10.000 | new 7.500
   0.0    0.0    0.0    0.0    0.0 
  ###    ###     0.0   ###     0.0 
   0.0    0.0    0.0    0.0    0.0 
   0.0   ###    ###    ###    ###  
   0.0    0.4    1.9    7.5    0.0 
```

Read the four updates in order. (1) At `(4, 3)`, right, into the goal: reward 10, no future. Target 10, new `0.5 x 10 =` **5.0**. (2) At `(4, 2)`, right, landing on `(4, 3)`: reward -1; `best_next` is now **5.0** (what step 1 just learned). Target `-1 + 0.95 x 5 = 3.75`; new `0.5 x 3.75 =` **1.875**. (3) At `(4, 1)`: `best_next` 1.875, target 0.781, new **0.391**. (4) Back at `(4, 3)` to the goal again: old 5.0, target 10, new `5 + 0.5 x 5 =` **7.5**. The value map's bottom row is `0.0 0.4 1.9 7.5 0.0`. **That is the ripple.** Step 2's cell had no idea where the goal was; it learned only because the cell it landed in had already learned. Good news flows *backward* from the goal, one cell per update. The goal cell itself stays `0.0`, because we never leave it, so its row is never updated.

**Try this yourself.** Predict the new value if you now run `trace_step(Q, (4, 2), 3)` a second time. (old 1.875, best_next 7.5, target `-1 + 0.95 x 7.5 = 6.125`, new `1.875 + 0.5 x (6.125 - 1.875) = 4.0`.)

## Step 12: How the table learns that walls are bad

**The idea first.** The same update handles bad news just as well as good news. We press "down" from `(2, 2)`, where there is a wall below, twice.

This step has **5 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 12.1: type this

```python
Q = np.zeros((n_states, 4))
```

- **Creates `Q`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `n_states` → *int (whole number)*: `25`
2. `(n_states, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`

- **Why / what it means:** A fresh table.

In [ ]:
# Step 12.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.2: type this

```python
trace_step(Q, (2, 2), 1)                  # "down" from (2, 2): into a wall
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `trace_step(Q, (2, 2), 1)` → *None (nothing)*: `None`

**Prints:**
```text
at (2, 2) (row 12) action down -> lands (2, 2) (row 12), reward -5
   old 0.000 | best_next 0.000 | target -5 + 0.95 * 0.000 = -5.000 | new -2.500
```

- **Why / what it means:** From `(2, 2)` press down (action 1). There is a wall below, so `step` returns the same cell with the wall penalty. The update then uses `best_next` from the *same* row, which is all zeros, so the target is just -5.

In [ ]:
# Step 12.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.3: type this

```python
print(Q[state_of((2, 2))])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
2. `state_of((2, 2))` → *int (whole number)*: `12`
3. `Q[state_of((2, 2))]` → *numpy array, shape (4,), holding decimal numbers*: `[0., -2.5, 0., 0.]`

**Prints:**
```text
[ 0.  -2.5  0.   0. ]
```

In [ ]:
# Step 12.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.4: type this

```python
trace_step(Q, (2, 2), 1)                  # the same bump again
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
2. `trace_step(Q, (2, 2), 1)` → *None (nothing)*: `None`

**Prints:**
```text
at (2, 2) (row 12) action down -> lands (2, 2) (row 12), reward -5
   old -2.500 | best_next 0.000 | target -5 + 0.95 * 0.000 = -5.000 | new -3.750
```

- **Why / what it means:** The same bump again. The old value is now -2.5, the target is still -5, so we move halfway again: to -3.75.

In [ ]:
# Step 12.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12.5: type this

```python
print(Q[state_of((2, 2))])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -3.75, largest 0`
2. `state_of((2, 2))` → *int (whole number)*: `12`
3. `Q[state_of((2, 2))]` → *numpy array, shape (4,), holding decimal numbers*: `[0., -3.75, 0., 0.]`

**Prints:**
```text
[ 0.   -3.75  0.    0.  ]
```

In [ ]:
# Step 12.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 12 complete: everything it printed, and what it means

```text
at (2, 2) (row 12) action down -> lands (2, 2) (row 12), reward -5
   old 0.000 | best_next 0.000 | target -5 + 0.95 * 0.000 = -5.000 | new -2.500
[ 0.  -2.5  0.   0. ]
at (2, 2) (row 12) action down -> lands (2, 2) (row 12), reward -5
   old -2.500 | best_next 0.000 | target -5 + 0.95 * 0.000 = -5.000 | new -3.750
[ 0.   -3.75  0.    0.  ]
```

First press: old 0, target `-5 + 0 = -5`, new `0.5 x -5 =` **-2.5**. The row is `[0, -2.5, 0, 0]`: only the "down" box changed. Second press: old -2.5, target again -5, new `-2.5 + 0.5 x (-5 + 2.5) =` **-3.75**, creeping toward -5. **Why this teaches the agent to avoid walls:** the three other boxes are still 0, and 0 is *better* than -3.75, so `argmax` now prefers the untried actions. Bad news pushes the agent toward alternatives. (Notice that because every real reward here is negative, a fresh table of zeros quietly encourages exploring: untried boxes look better than tried ones. Experiment 1 returns to this.)

## Step 13: The training loop

**The idea first.** One **episode** is one attempt from the start to the goal (or a step limit). `train` runs many of them, applying the update thousands of times, while epsilon shrinks so the agent explores a lot early and exploits later.

This step has **6 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 13.1: type this

```python
def train(episodes, alpha=0.5, gamma=0.95, eps_start=1.0, eps_min=0.05,
          eps_decay=0.99, max_steps=200, Q=None, stop_when_optimal=False):
    if Q is None:                         # no table given: start a fresh one
        Q = np.zeros((ROWS * COLS, 4))
    shortest = bfs()
    epsilon = eps_start
    history = []                          # steps taken in each episode
    for episode in range(episodes):
        cell = START
        for t in range(max_steps):
            s = state_of(cell)
            a = choose_action(Q, s, epsilon)
            next_cell, reward, done = step(cell, a)
            s_next = state_of(next_cell)

            best_next = 0.0 if done else Q[s_next].max()
            target = reward + gamma * best_next
            Q[s, a] += alpha * (target - Q[s, a])     # the whole algorithm

            cell = next_cell
            if done:
                break
        epsilon = max(eps_min, epsilon * eps_decay)   # explore a little less each episode
        history.append(t + 1)
        if stop_when_optimal:             # used by the experiments later
            p = greedy_path(Q)
            if p is not None and len(p) - 1 == shortest:
                break
    return Q, history
```

- **Creates** `train` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The full algorithm. Parameters after `episodes` have **default values**, so `train(300)` works. `Q=None` means 'make a new table unless one is given'. It ends with `return Q, history`: **two** values, the table and a list of episode lengths.

In [ ]:
# Step 13.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `train(1, max_steps=3)`

**What the function receives** (its inputs, called *parameters*):
- `episodes` — type: **int (whole number)**; value: `1`
- `alpha` — type: **float (decimal number)**; value: `0.5`
- `gamma` — type: **float (decimal number)**; value: `0.95`
- `eps_start` — type: **float (decimal number)**; value: `1.0`
- `eps_min` — type: **float (decimal number)**; value: `0.05`
- `eps_decay` — type: **float (decimal number)**; value: `0.99`
- `max_steps` — type: **int (whole number)**; value: `3`
- `Q` — type: **None (nothing)**; value: `None`
- `stop_when_optimal` — type: **bool (True/False)**; value: `False`

**Now each line runs, top to bottom:**

```python
if Q is None:                         # no table given: start a fresh one
```
Evaluated inside-out (the main pieces):
1. `Q` → *None (nothing)*: `None`
2. `Q is None` → *bool (True/False)*: `True`
- **Why:** `is None` tests whether no table was passed in.

```python
Q = np.zeros((ROWS * COLS, 4))
```
Evaluated inside-out (the main pieces):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`
3. `ROWS * COLS` → *int (whole number)*: `25`
4. `(ROWS * COLS, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`
- **Why:** A new, empty table sized for the current maze.

```python
shortest = bfs()
```
- **Now `shortest` holds** — type: **int (whole number)**; value: `12`
- **Why:** The true answer, used only when `stop_when_optimal` is on.

```python
epsilon = eps_start
```
- **Now `epsilon` holds** — type: **float (decimal number)**; value: `1.0`
- **Why:** Epsilon starts at 1.0: explore always.


#### …the same call, continued: `train(1, max_steps=3)`

```python
history = []                          # steps taken in each episode
```
- **Now `history` holds** — type: **list of 0 items**; value: `[]`
- **Why:** An empty list that will collect each episode's length.

```python
for episode in range(episodes):
```
Evaluated inside-out (the main pieces):
1. `episodes` → *int (whole number)*: `1`
2. `range(episodes)` → *range*: `range(0, 1)`
- **Now `episode` holds** — type: **int (whole number)**; value: `0`
- **Why:** The outer loop: one pass per episode.

```python
cell = START
```
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Why:** Every episode begins at the start cell (a tuple).

```python
for t in range(max_steps):
```
Evaluated inside-out (the main pieces):
1. `max_steps` → *int (whole number)*: `3`
2. `range(max_steps)` → *range*: `range(0, 3)`
- **Now `t` holds** — type: **int (whole number)**; value: `0`
- **Why:** The inner loop: at most 200 steps, so the agent cannot wander forever. `t` counts from 0.

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `s` holds** — type: **int (whole number)**; value: `0`
- **Why:** Row number for the current cell.

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `s` → *int (whole number)*: `0`
3. `epsilon` → *float (decimal number)*: `1.0`
- **Now `a` holds** — type: **int (whole number)**; value: `2`
- **Why:** Epsilon-greedy choice from Step 10.

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `2`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Now `reward` holds** — type: **int (whole number)**; value: `-5`
- **Now `done` holds** — type: **bool (True/False)**; value: `False`
- **Why:** The environment responds.

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `0`
- **Why:** Row number of where we landed.

```python
best_next = 0.0 if done else Q[s_next].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
3. `s_next` → *int (whole number)*: `0`
4. `Q[s_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
- **Now `best_next` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** Future value of the landing cell, or 0 at the goal.

```python
target = reward + gamma * best_next
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `gamma` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
- **Now `target` holds** — type: **float (decimal number)**; value: `-5.0`
- **Why:** The target, as in Step 11.

```python
Q[s, a] += alpha * (target - Q[s, a])     # the whole algorithm
```
Evaluated inside-out (the main pieces):
1. `s` → *int (whole number)*: `0`
2. `a` → *int (whole number)*: `2`
3. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 2)`
4. `alpha` → *float (decimal number)*: `0.5`
5. `target` → *float (decimal number)*: `-5.0`
6. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
7. `Q[s, a]` → *float (decimal number)*: `0.0`
8. `target - Q[s, a]` → *float (decimal number)*: `-5.0`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
- **Why:** **The whole algorithm in one line.** `+=` adds to the existing box, so this is `Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])`.

```python
cell = next_cell
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Why:** Moves the agent: the landing cell becomes the current cell.

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- **Why:** If we reached the goal, `break` leaves the inner loop and the episode ends.


#### …the same call, continued: `train(1, max_steps=3)`

```python
for t in range(max_steps):
```
- **Now `t` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `state_of(cell)` → *int (whole number)*: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
2. `s` → *int (whole number)*: `0`
3. `epsilon` → *float (decimal number)*: `1.0`
- **Now `a` holds** — type: **int (whole number)**; value: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `0`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `state_of(next_cell)` → *int (whole number)*: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if done else Q[s_next].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
3. `s_next` → *int (whole number)*: `0`
4. `Q[s_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., -2.5, 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
6. `0.0 if done else Q[s_next].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
target = reward + gamma * best_next
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `gamma` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
5. `reward + gamma * best_next` → *float (decimal number)*: `-5.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[s, a] += alpha * (target - Q[s, a])     # the whole algorithm
```
Evaluated inside-out (the main pieces):
1. `s` → *int (whole number)*: `0`
2. `a` → *int (whole number)*: `0`
3. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 0)`
4. `alpha` → *float (decimal number)*: `0.5`
5. `target` → *float (decimal number)*: `-5.0`
6. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
7. `Q[s, a]` → *float (decimal number)*: `0.0`
8. `target - Q[s, a]` → *float (decimal number)*: `-5.0`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 2 are non-zero; smallest -2.5, largest 0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
cell = next_cell
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for t in range(max_steps):
```
- **Now `t` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `state_of(cell)` → *int (whole number)*: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 2 are non-zero; smallest -2.5, largest 0`
2. `s` → *int (whole number)*: `0`
3. `epsilon` → *float (decimal number)*: `1.0`
- **Now `a` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `train(1, max_steps=3)`

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `3`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- **Now `reward` holds** — type: **int (whole number)**; value: `-1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if done else Q[s_next].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 2 are non-zero; smallest -2.5, largest 0`
3. `s_next` → *int (whole number)*: `1`
4. `Q[s_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 13.2: type this

```python
def greedy_path(Q, max_len=50):
    cell = START
    path = [cell]
    for _ in range(max_len):
        a = int(np.argmax(Q[state_of(cell)]))
        cell, _, done = step(cell, a)
        path.append(cell)
        if done:
            return path
    return None                           # never reached the goal (a loop)
```

- **Creates** `greedy_path` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Follows the table's best actions from the start, recording each cell, until the goal is reached. `max_len` stops it looping forever if the table is badly trained.

In [ ]:
# Step 13.2: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `greedy_path(Q, max_len=3)`

**What the function receives** (its inputs, called *parameters*):
- `Q` — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
- `max_len` — type: **int (whole number)**; value: `3`

**Now each line runs, top to bottom:**

```python
cell = START
```
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Why:** Starts the walk at the beginning.

```python
path = [cell]
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `path` holds** — type: **list of 1 item (each a tuple)**; value: `[(0, 0)]`
- **Why:** A list holding the cells visited, starting with the start.

```python
for _ in range(max_len):
```
Evaluated inside-out (the main pieces):
1. `max_len` → *int (whole number)*: `3`
2. `range(max_len)` → *range*: `range(0, 3)`
- **Now `_` holds** — type: **int (whole number)**; value: `0`
- **Why:** At most `max_len` moves; `_` is a throwaway counter name.

```python
a = int(np.argmax(Q[state_of(cell)]))
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `state_of(cell)` → *int (whole number)*: `0`
4. `Q[state_of(cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[-7.7892, -7.7892, -7.7892, -2.936]`
5. `np.argmax(Q[state_of(cell)])` → *numpy integer (a whole number)*: `3`
- **Now `a` holds** — type: **int (whole number)**; value: `3`
- **Why:** Row for this cell, position of its best number: the best action. No randomness.


#### …the same call, continued: `greedy_path(Q, max_len=3)`

```python
cell, _, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `3`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- **Now `_` holds** — type: **int (whole number)**; value: `-1`
- **Now `done` holds** — type: **bool (True/False)**; value: `False`
- **Why:** Moves; keeps the new cell and `done`, ignores the reward.

```python
path.append(cell)
```
Evaluated inside-out (the main pieces):
1. `path` → *list of 1 item (each a tuple)*: `[(0, 0)]`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
3. `path.append(cell)` → *None (nothing)*: `None`
- **Now `path` holds** — type: **list of 2 items (each a tuple)**; value: `[(0, 0), (0, 1)]`
- **Why:** Adds the new cell to the end of the list.

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- **Why:** On reaching the goal the function returns the path. (If the walk never gets there, the last line returns `None`: the table sent the agent in a loop.)

```python
for _ in range(max_len):
```
- **Now `_` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = int(np.argmax(Q[state_of(cell)]))
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
3. `state_of(cell)` → *int (whole number)*: `1`
4. `Q[state_of(cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[-6.936, -6.936, -3.7892, -2.0379]`
5. `np.argmax(Q[state_of(cell)])` → *numpy integer (a whole number)*: `3`
6. `int(np.argmax(Q[state_of(cell)]))` → *int (whole number)*: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
cell, _, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `a` → *int (whole number)*: `3`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 2), -1, False)`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 2)`
- **Now `_` holds** — type: **int (whole number)**; value: `-1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `greedy_path(Q, max_len=3)`

```python
path.append(cell)
```
Evaluated inside-out (the main pieces):
1. `path` → *list of 2 items (each a tuple)*: `[(0, 0), (0, 1)]`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 2)`
3. `path.append(cell)` → *None (nothing)*: `None`
- **Now `path` holds** — type: **list of 3 items (each a tuple)**; value: `[(0, 0), (0, 1), (0, 2)]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for _ in range(max_len):
```
- **Now `_` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = int(np.argmax(Q[state_of(cell)]))
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 2)`
3. `state_of(cell)` → *int (whole number)*: `2`
4. `Q[state_of(cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[-6.0379, -1.0925, -2.936, -2.936]`
5. `np.argmax(Q[state_of(cell)])` → *numpy integer (a whole number)*: `1`
- **Now `a` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
cell, _, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 2)`
2. `a` → *int (whole number)*: `1`
3. `step(cell, a)` → *tuple of 3 items*: `((1, 2), -1, False)`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(1, 2)`
- **Now `_` holds** — type: **int (whole number)**; value: `-1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 13.3: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable randomness.

In [ ]:
# Step 13.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 13.4: type this

```python
Q, history = train(300)
```

- **Creates `Q`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`

- **Creates `history`** — type: **list of 300 items (each an int)**; value: `[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38, 87, 96, 41, 77, 134, 199, 40, 67, 57, 95, 86, 79, 42, 46, 71, 31, 39, ...`

- **Why / what it means:** Runs 300 episodes and unpacks the two returned values: `Q` becomes the learned table (array) and `history` a list of 300 ints.

In [ ]:
# Step 13.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 13.5: type this

```python
print(history[:12])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `history` → *list of 300 items (each an int)*: `[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38, 87, 96, 41, 77, 134, 199, 40, 67, 57, 95, 86, 79, 42, 46, 71, 31, 39, ...`
2. `history[:12]` → *list of 12 items (each an int)*: `[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38]`

**Prints:**
```text
[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38]
```

In [ ]:
# Step 13.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 13.6: type this

```python
for i in range(0, 300, 50):
    print("episodes", i, "to", i + 49, ": average steps", round(sum(history[i:i + 50]) / 50, 1))
```

- **Creates `i`** — type: **int (whole number)**; value: `250`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `range(0, 300, 50)` → *range*: `range(0, 300, 50)`
2. `i` → *int (whole number)*: `0` *(evaluated 6 times; the first is shown)*
3. `i + 49` → *int (whole number)*: `49` *(evaluated 6 times; the first is shown)*
4. `history` → *list of 300 items (each an int)*: `[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38, 87, 96, 41, 77, 134, 199, 40, 67, 57, 95, 86, 79, 42, 46, 71, 31, 39, ...` *(evaluated 6 times; the first is shown)*
5. `history[i:i + 50]` → *list of 50 items (each an int)*: `[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38, 87, 96, 41, 77, 134, 199, 40, 67, 57, 95, 86, 79, 42, 46, 71, 31, 39, ...` *(evaluated 6 times; the first is shown)*
6. `sum(history[i:i + 50])` → *int (whole number)*: `3560` *(evaluated 6 times; the first is shown)*
7. `sum(history[i:i + 50]) / 50` → *float (decimal number)*: `71.2` *(evaluated 6 times; the first is shown)*
8. `round(sum(history[i:i + 50]) / 50, 1)` → *float (decimal number)*: `71.2` *(evaluated 6 times; the first is shown)*

**Prints:**
```text
episodes 0 to 49 : average steps 71.2
episodes 50 to 99 : average steps 23.5
episodes 100 to 149 : average steps 17.7
episodes 150 to 199 : average steps 15.2
episodes 200 to 249 : average steps 13.9
episodes 250 to 299 : average steps 13.0
```

- **Why / what it means:** Counts 0, 50, 100, ... 250 (step size 50). `history[i:i + 50]` is a slice of 50 episode lengths; `sum(...) / 50` is their average.

In [ ]:
# Step 13.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 13 complete: everything it printed, and what it means

```text
[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38]
episodes 0 to 49 : average steps 71.2
episodes 50 to 99 : average steps 23.5
episodes 100 to 149 : average steps 17.7
episodes 150 to 199 : average steps 15.2
episodes 200 to 249 : average steps 13.9
episodes 250 to 299 : average steps 13.0
```

The first twelve episodes took `[200, 103, 200, 200, 121, ...]` steps: mostly random wandering (200 means it timed out). The block averages then fall from **71.2** steps to 23.5, 17.7, 15.2, 13.9 and **13.0**. They approach the shortest path of 12 but do not reach it, because epsilon never drops below 0.05, so about one move in twenty is still random. **A perfect table can still produce imperfect episodes.**

## Step 14: Reading the learned table

**The idea first.** Now look at what the agent learned: the value map, the arrow map, the path it follows, and one very important number.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 14.1: type this

```python
show_values(Q)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `show_values(Q)` → *None (nothing)*: `None`

**Prints:**
```text
  -2.9   -2.0   -1.1   -2.0   -2.9 
  ###    ###    -0.1   ###    -2.1 
   3.2    2.1    1.0   -0.1   -1.1 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0    0.0 
```

- **Why / what it means:** The best value per cell of the table learned in Step 13.

In [ ]:
# Step 14.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.2: type this

```python
print()
```

In [ ]:
# Step 14.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.3: type this

```python
show_policy(Q)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `show_policy(Q)` → *None (nothing)*: `None`

**Prints:**
```text
> > v < < 
# # v # v 
v < < < < 
v # # # # 
> > > > G 
```

- **Why / what it means:** The best action per cell, as arrows.

In [ ]:
# Step 14.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.4: type this

```python
path = greedy_path(Q)
```

- **Creates `path`** — type: **list of 13 items (each a tuple)**; value: `[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`

- **Why / what it means:** Follows those arrows from the start. The result is a **list of 13 tuples**, the cells visited, so 12 moves.

In [ ]:
# Step 14.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.5: type this

```python
print(path)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `path` → *list of 13 items (each a tuple)*: `[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]`

**Prints:**
```text
[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
```

In [ ]:
# Step 14.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.6: type this

```python
print("learned path length:", len(path) - 1, " shortest possible:", bfs())
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `path` → *list of 13 items (each a tuple)*: `[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]`
2. `len(path)` → *int (whole number)*: `13`
3. `len(path) - 1` → *int (whole number)*: `12`
4. `bfs()` → *int (whole number)*: `12`

**Prints:**
```text
learned path length: 12  shortest possible: 12
```

In [ ]:
# Step 14.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14.7: type this

```python
print("Q at start, best action:", round(Q[state_of(START)].max(), 3))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `state_of(START)` → *int (whole number)*: `0`
4. `Q[state_of(START)]` → *numpy array, shape (4,), holding decimal numbers*: `[-7.7892, -7.7892, -7.7892, -2.936]`
5. `Q[state_of(START)].max()` → *float (decimal number)*: `-2.935997231706204`
6. `round(Q[state_of(START)].max(), 3)` → *float (decimal number)*: `-2.936`

**Prints:**
```text
Q at start, best action: -2.936
```

In [ ]:
# Step 14.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 14 complete: everything it printed, and what it means

```text
  -2.9   -2.0   -1.1   -2.0   -2.9 
  ###    ###    -0.1   ###    -2.1 
   3.2    2.1    1.0   -0.1   -1.1 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0    0.0 
> > v < < 
# # v # v 
v < < < < 
v # # # # 
> > > > G 
[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]
learned path length: 12  shortest possible: 12
Q at start, best action: -2.936
```

The values rise steadily along the route: `-2.9, -2.0, -1.1, -0.1` down through the gap, then `1.0, 2.1, 3.2, 4.4, 5.7, 7.1, 8.5, 10.0` to the goal. **Check the rule by hand:** a cell's value should be `-1 + 0.95 x (value of the next cell on its best route)`. For `(4, 2)`: `-1 + 0.95 x 10.0 = 8.5` ✓. For `(4, 1)`: `-1 + 0.95 x 8.5 = 7.075` ✓ (shown as 7.1). When every number agrees with its neighbour like this, the table has **converged** (this self-consistency is called the Bellman equation). The arrows lead from every visited cell back to the route: the dead-end cells `(0, 3)` and `(0, 4)` point left, away from the dead end. The path has length **12**, equal to the BFS answer. And the final line prints **-2.936**, *exactly the number you computed by hand in Step 8*. The agent was never told about discounting or routes; its table converged to the true value of the best plan.

## Step 15: Watching the wave spread

**The idea first.** Learning is a wave moving backward from the goal. To see it, we train the *same* table in stages and print it after each.

This step has **2 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 15.1: type this

```python
Q = np.zeros((n_states, 4))
```

- **Creates `Q`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `n_states` → *int (whole number)*: `25`
2. `(n_states, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`

- **Why / what it means:** A new, untrained table that we will train in stages.

In [ ]:
# Step 15.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15.2: type this

```python
for total in (1, 5, 20, 100):
    random.seed(total)
    train(total, Q=Q, eps_start=0.3, eps_decay=1.0)      # keep training the SAME table
    print("after another", total, "episodes")
    show_values(Q)
    print()
```

- **Creates `total`** — type: **int (whole number)**; value: `100`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `total` → *int (whole number)*: `1` *(evaluated 4 times; the first is shown)*
2. `random.seed(total)` → *None (nothing)*: `None` *(evaluated 4 times; the first is shown)*
3. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero` *(evaluated 4 times; the first is shown)*
4. `train(total, Q=Q, eps_start=0.3, eps_decay=1.0)` → *tuple of 2 items*: `(array([[-2.5 , -2.5 , -2.5 , -2.16652373], [-5.21949248, -5.16617187, -2.09991406, -2.11346289], [-4.2190625 , -2.00859375, -2...` *(evaluated 4 times; the first is shown)*
5. `show_values(Q)` → *None (nothing)*: `None` *(evaluated 4 times; the first is shown)*

**Prints:**
```text
after another 1 episodes
  -2.2   -2.1   -2.0   -2.0   -1.8 
  ###    ###    -1.9   ###    -1.2 
  -1.6   -1.8   -1.8   -1.8   -1.6 
  -1.2   ###    ###    ###    ###  
  -0.9   -0.8    0.0    5.0    0.0 

after another 5 episodes
  -5.2   -5.0   -4.7   -4.5   -4.4 
  ###    ###    -4.3   ###    -4.4 
  -3.2   -3.5   -3.9   -4.1   -3.8 
  -1.9   ###    ###    ###    ###  
   0.4    4.1    7.8    9
...
```

- **Why / what it means:** Loops over the four stage sizes. Inside, `train(total, Q=Q, ...)` passes **our existing table** in: because arrays are changed in place, `Q` keeps its progress between stages. `eps_start=0.3` and `eps_decay=1.0` hold epsilon at 0.3 (multiplying by 1.0 changes nothing).

In [ ]:
# Step 15.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 15 complete: everything it printed, and what it means

```text
after another 1 episodes
  -2.2   -2.1   -2.0   -2.0   -1.8 
  ###    ###    -1.9   ###    -1.2 
  -1.6   -1.8   -1.8   -1.8   -1.6 
  -1.2   ###    ###    ###    ###  
  -0.9   -0.8    0.0    5.0    0.0 

after another 5 episodes
  -5.2   -5.0   -4.7   -4.5   -4.4 
  ###    ###    -4.3   ###    -4.4 
  -3.2   -3.5   -3.9   -4.1   -3.8 
  -1.9   ###    ###    ###    ###  
   0.4    4.1    7.8    9.8    0.0 

after another 20 episodes
  -4.1   -2.6   -1.3   -4.1   -5.9 
  ###    ###    -0.2   ###    -5.7 
   3.2    2.0    0.9   -2.7   -5.6 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0    0.0 

after another 100 episodes
  -2.9   -2.0   -1.1   -2.0   -5.9 
  ###    ###    -0.1   ###    -5.7 
   3.2    2.1    1.0   -0.1   -5.6 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0    0.0 
```

**After 1 episode:** the cell beside the goal, `(4, 3)`, already holds `5.0`; the bottom row elsewhere is small negative numbers (`(4, 2)` is still `0.0`). **After 5 more:** the bottom row reads `0.4, 4.1, 7.8, 9.8`; the goal's value is spreading backward along it. **After 20 more:** the bottom row and left column are essentially their final values from Step 14. **After 100 more:** the top row near the start is `-2.9, -2.0, -1.1`, final. The cells in the dead end on the right (`-5.9`, `-5.7`, `-5.6`) are still *stale*: a box is only updated when the agent visits it, and with epsilon 0.3 it spends little time there. **The table is correct only where the wave has passed.**

---
# Part D: experiments

**Change one thing, write down a prediction, then run.** Each line reports: how many episodes it took before the table's greedy path equalled the true shortest path (for 5 different random seeds, with `None` meaning it never managed it in 2000 episodes), the mean of those, and the average total number of moves made while learning (a measure of how costly the learning was).

## Step 16: A helper that measures how fast a setting learns

**The idea first.** To compare settings fairly we need a measurement. This helper trains the agent from scratch with 5 different seeds and reports how many episodes it took for the table to produce the true shortest path.

This step has **6 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 16.1: type this

```python
def experiment(label, trainer=None, seeds=range(5), episodes=2000, rewards=None, **settings):
    global STEP_REWARD, WALL_REWARD, GOAL_REWARD
    trainer = trainer or train
    saved = (STEP_REWARD, WALL_REWARD, GOAL_REWARD)
    try:
        if rewards is not None:
            STEP_REWARD, WALL_REWARD, GOAL_REWARD = rewards
        shortest = bfs()
        solved_at, costs = [], []
        for seed in seeds:
            random.seed(seed)
            Q_run, hist = trainer(episodes, stop_when_optimal=True, **settings)
            path = greedy_path(Q_run)
            ok = path is not None and len(path) - 1 == shortest
            solved_at.append(len(hist) if ok else None)
            costs.append(sum(hist))
    finally:
        STEP_REWARD, WALL_REWARD, GOAL_REWARD = saved
    good = [x for x in solved_at if x is not None]
    mean_solved = round(sum(good) / len(good), 1) if good else None
    print(f"{label:22s} episodes to shortest path: {solved_at}  mean {mean_solved}  avg total steps {round(sum(costs) / len(costs))}")
```

- **Creates** `experiment` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Trains from scratch several times with chosen settings and prints one summary row. `trainer=None` lets us pass a different learning function; `**settings` collects extra named inputs like `alpha=0.01` and passes them on to the trainer.

In [ ]:
# Step 16.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.2: type this

```python
experiment("baseline")
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('baseline')` → *None (nothing)*: `None`

**Prints:**
```text
baseline               episodes to shortest path: [5, 6, 8, 7, 7]  mean 6.6  avg total steps 1128
```

- **Why / what it means:** No overrides: every setting takes its default.

In [ ]:
# Step 16.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.3: type this

```python
experiment("no exploring", eps_start=0.0, eps_min=0.0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('no exploring', eps_start=0.0, eps_min=0.0)` → *None (nothing)*: `None`

**Prints:**
```text
no exploring           episodes to shortest path: [10, 13, 10, 12, 14]  mean 11.8  avg total steps 489
```

- **Why / what it means:** Epsilon starts at 0 and stays there.

In [ ]:
# Step 16.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.4: type this

```python
experiment("alpha=0.01", alpha=0.01)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('alpha=0.01', alpha=0.01)` → *None (nothing)*: `None`

**Prints:**
```text
alpha=0.01             episodes to shortest path: [142, 213, 7, 105, 226]  mean 138.6  avg total steps 11611
```

- **Why / what it means:** A hundred times smaller learning rate than 1.0, fifty times smaller than the default 0.5.

In [ ]:
# Step 16.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.5: type this

```python
experiment("gamma=0.0", gamma=0.0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('gamma=0.0', gamma=0.0)` → *None (nothing)*: `None`

**Prints:**
```text
gamma=0.0              episodes to shortest path: [None, None, None, None, None]  mean None  avg total steps 282957
```

- **Why / what it means:** A discount factor of zero: only the immediate reward counts.

In [ ]:
# Step 16.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16.6: type this

```python
experiment("wall bump costs 0", rewards=(-1, 0, 10))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `(-1, 0, 10)` → *tuple of 3 items (each an int)*: `(-1, 0, 10)`
2. `experiment('wall bump costs 0', rewards=(-1, 0, 10))` → *None (nothing)*: `None`

**Prints:**
```text
wall bump costs 0      episodes to shortest path: [None, None, None, None, None]  mean None  avg total steps 379748
```

- **Why / what it means:** `rewards=(-1, 0, 10)` is a tuple overriding the three reward constants: step -1, wall 0, goal +10.

In [ ]:
# Step 16.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 16 complete: everything it printed, and what it means

```text
baseline               episodes to shortest path: [5, 6, 8, 7, 7]  mean 6.6  avg total steps 1128
no exploring           episodes to shortest path: [10, 13, 10, 12, 14]  mean 11.8  avg total steps 489
alpha=0.01             episodes to shortest path: [142, 213, 7, 105, 226]  mean 138.6  avg total steps 11611
gamma=0.0              episodes to shortest path: [None, None, None, None, None]  mean None  avg total steps 282957
wall bump costs 0      episodes to shortest path: [None, None, None, None, None]  mean None  avg total steps 379748
```

**Baseline:** the shortest path is found after `[5, 6, 8, 7, 7]` episodes, about **6.6** on average, for about 1128 moves. (Only about seven episodes, even though Step 13's average episode length took 300 episodes to settle near 13. Those are different questions: the *table* becomes right quickly, while the *behaviour* stays noisy because epsilon is still high.)

**No exploring: 11.8 episodes, but only 489 moves.** Slower to find the path, but much cheaper, because it never wastes moves on random choices. It works only because the rewards here are *negative*: tried boxes drop below 0 while untried ones sit at 0, so the agent keeps trying untried actions on its own (this is called optimistic initialisation). In CartPole all the rewards were positive, and `epsilon = 0` made the agent get stuck.

**Alpha 0.01: 138.6 episodes**, twenty times slower, since each update moves a box only 1% of the way.

**Gamma 0: never.** With `gamma = 0` the target is just the reward, so only the cell next to the goal learns anything; no information ripples backward and the path is never found.

**Wall bump costs 0: never.** The total of about 380,000 moves over 2000 episodes means about 190 moves per episode, nearly always hitting the 200-move limit. A normal move costs -1 but bumping a wall costs 0, so *standing still beats moving*, and the agent learned to push into walls. **The agent maximises the reward you give it, not what you meant.** This is the most common bug in reinforcement-learning projects.

**Try this yourself.** Predict, then run, one more: `experiment("alpha=1.0", alpha=1.0)`. In CartPole this failed. Will it here? (It works, about as fast as the baseline, because this maze is deterministic and every cell has its own row, so the newest sample is always exactly right.)

---
# Part E: beyond a table that learns by trial and error

Q-learning is one way to find the numbers in the table. Here are three others, and each one teaches you something about Q-learning itself.

## Step 17: Value iteration: computing the answer directly

**The idea first.** We have been *learning* the table by trial and error. But the maze is small and we know exactly how it behaves, so we can compute the table directly. This method is **value iteration**. It "cheats" by consulting the environment's rules, which a real learner would not be allowed to do, but it gives us the perfect answer to compare against.

This step has **7 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 17.1: type this

```python
def value_iteration(gamma=0.95, max_sweeps=200):
    Q = np.zeros((ROWS * COLS, 4))
    for sweep in range(max_sweeps):
        biggest_change = 0.0
        for s in range(ROWS * COLS):
            cell = cell_of(s)
            if walls[cell] == 1 or cell == GOAL:   # nothing to compute in walls or the goal
                continue
            for a in range(4):
                next_cell, reward, done = step(cell, a)     # we ASK the environment: cheating!
                best_next = 0.0 if done else Q[state_of(next_cell)].max()
                new = reward + gamma * best_next            # no alpha: write the full answer
                biggest_change = max(biggest_change, abs(new - Q[s, a]))
                Q[s, a] = new
        if biggest_change < 1e-9:         # nothing changed any more: finished
            return Q, sweep + 1
    return Q, max_sweeps
```

- **Creates** `value_iteration` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Computes the exact Q-table by repeated sweeps, using the environment's rules. Parameters: `gamma` and the maximum number of sweeps.

In [ ]:
# Step 17.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `value_iteration(max_sweeps=1)`

**What the function receives** (its inputs, called *parameters*):
- `gamma` — type: **float (decimal number)**; value: `0.95`
- `max_sweeps` — type: **int (whole number)**; value: `1`

**Now each line runs, top to bottom:**

```python
Q = np.zeros((ROWS * COLS, 4))
```
Evaluated inside-out (the main pieces):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`
3. `ROWS * COLS` → *int (whole number)*: `25`
4. `(ROWS * COLS, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`
- **Why:** Starts blank, like the learner.

```python
for sweep in range(max_sweeps):
```
Evaluated inside-out (the main pieces):
1. `max_sweeps` → *int (whole number)*: `1`
2. `range(max_sweeps)` → *range*: `range(0, 1)`
- **Now `sweep` holds** — type: **int (whole number)**; value: `0`
- **Why:** Each pass is one **sweep** over the whole table.

```python
biggest_change = 0.0
```
- **Now `biggest_change` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** Resets the record of how much the table changed in this sweep.

```python
for s in range(ROWS * COLS):
```
Evaluated inside-out (the main pieces):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`
3. `ROWS * COLS` → *int (whole number)*: `25`
4. `range(ROWS * COLS)` → *range*: `range(0, 25)`
- **Now `s` holds** — type: **int (whole number)**; value: `0`
- **Why:** Visits every cell number.


#### …the same call, continued: `value_iteration(max_sweeps=1)`

```python
cell = cell_of(s)
```
Evaluated inside-out (the main pieces):
1. `s` → *int (whole number)*: `0`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Why:** Converts the row number back to a (row, column) tuple.

```python
if walls[cell] == 1 or cell == GOAL:   # nothing to compute in walls or the goal
```
Evaluated inside-out (the main pieces):
1. `walls` → *numpy array, shape (5, 5), holding whole numbers*: `[[0, 0, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 1, 1, 1], [0, 0, 0, 0, 0]]`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `walls[cell]` → *numpy integer (a whole number)*: `0`
4. `walls[cell] == 1` → *bool*: `np.False_`
5. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
6. `cell == GOAL` → *bool (True/False)*: `False`
7. `walls[cell] == 1 or cell == GOAL` → *bool (True/False)*: `False`
- **Why:** Skips walls and the goal. `walls[cell]` works because a tuple can index an array.

```python
for a in range(4):
```
Evaluated inside-out (the main pieces):
1. `range(4)` → *range*: `range(0, 4)`
- **Now `a` holds** — type: **int (whole number)**; value: `0`
- **Why:** All four actions in this cell.

```python
next_cell, reward, done = step(cell, a)     # we ASK the environment: cheating!
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `0`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Now `reward` holds** — type: **int (whole number)**; value: `-5`
- **Now `done` holds** — type: **bool (True/False)**; value: `False`
- **Why:** **The cheat:** we ask the environment what *would* happen. A learning agent cannot do this; it must actually try.

```python
best_next = 0.0 if done else Q[state_of(next_cell)].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
3. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
4. `state_of(next_cell)` → *int (whole number)*: `0`
5. `Q[state_of(next_cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
6. `Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
- **Now `best_next` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** Best value of the landing cell, or 0 at the goal.

```python
new = reward + gamma * best_next            # no alpha: write the full answer
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `gamma` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
- **Now `new` holds** — type: **float (decimal number)**; value: `-5.0`
- **Why:** No `alpha`: we write the full answer, because the rules are exact and not noisy.


#### …the same call, continued: `value_iteration(max_sweeps=1)`

```python
biggest_change = max(biggest_change, abs(new - Q[s, a]))
```
Evaluated inside-out (the main pieces):
1. `biggest_change` → *float (decimal number)*: `0.0`
2. `new` → *float (decimal number)*: `-5.0`
3. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
4. `s` → *int (whole number)*: `0`
5. `a` → *int (whole number)*: `0`
6. `Q[s, a]` → *float (decimal number)*: `0.0`
7. `new - Q[s, a]` → *float (decimal number)*: `-5.0`
8. `abs(new - Q[s, a])` → *float (decimal number)*: `5.0`
- **Now `biggest_change` holds** — type: **float (decimal number)**; value: `5.0`
- **Why:** Keeps the largest change seen. `abs` removes the sign.

```python
Q[s, a] = new
```
Evaluated inside-out (the main pieces):
1. `new` → *float (decimal number)*: `-5.0`
2. `s` → *int (whole number)*: `0`
3. `a` → *int (whole number)*: `0`
4. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest -5, largest 0`
- **Why:** Overwrites the box.

```python
for a in range(4):
```
- **Now `a` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)     # we ASK the environment: cheating!
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `1`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if done else Q[state_of(next_cell)].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -5, largest 0`
3. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
4. `state_of(next_cell)` → *int (whole number)*: `0`
5. `Q[state_of(next_cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[-5., 0., 0., 0.]`
6. `Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
7. `0.0 if done else Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
new = reward + gamma * best_next            # no alpha: write the full answer
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `gamma` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
5. `reward + gamma * best_next` → *float (decimal number)*: `-5.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `value_iteration(max_sweeps=1)`

```python
biggest_change = max(biggest_change, abs(new - Q[s, a]))
```
Evaluated inside-out (the main pieces):
1. `biggest_change` → *float (decimal number)*: `5.0`
2. `new` → *float (decimal number)*: `-5.0`
3. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -5, largest 0`
4. `s` → *int (whole number)*: `0`
5. `a` → *int (whole number)*: `1`
6. `Q[s, a]` → *float (decimal number)*: `0.0`
7. `new - Q[s, a]` → *float (decimal number)*: `-5.0`
8. `abs(new - Q[s, a])` → *float (decimal number)*: `5.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[s, a] = new
```
Evaluated inside-out (the main pieces):
1. `new` → *float (decimal number)*: `-5.0`
2. `s` → *int (whole number)*: `0`
3. `a` → *int (whole number)*: `1`
4. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 2 are non-zero; smallest -5, largest 0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for a in range(4):
```
- **Now `a` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)     # we ASK the environment: cheating!
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `2`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if done else Q[state_of(next_cell)].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 2 are non-zero; smallest -5, largest 0`
3. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
4. `state_of(next_cell)` → *int (whole number)*: `0`
5. `Q[state_of(next_cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[-5., -5., 0., 0.]`
6. `Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
7. `0.0 if done else Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
new = reward + gamma * best_next            # no alpha: write the full answer
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `gamma` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
5. `reward + gamma * best_next` → *float (decimal number)*: `-5.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `value_iteration(max_sweeps=1)`

```python
biggest_change = max(biggest_change, abs(new - Q[s, a]))
```
Evaluated inside-out (the main pieces):
1. `biggest_change` → *float (decimal number)*: `5.0`
2. `new` → *float (decimal number)*: `-5.0`
3. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 2 are non-zero; smallest -5, largest 0`
4. `s` → *int (whole number)*: `0`
5. `a` → *int (whole number)*: `2`
6. `Q[s, a]` → *float (decimal number)*: `0.0`
7. `new - Q[s, a]` → *float (decimal number)*: `-5.0`
8. `abs(new - Q[s, a])` → *float (decimal number)*: `5.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[s, a] = new
```
Evaluated inside-out (the main pieces):
1. `new` → *float (decimal number)*: `-5.0`
2. `s` → *int (whole number)*: `0`
3. `a` → *int (whole number)*: `2`
4. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 2)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 3 are non-zero; smallest -5, largest 0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for a in range(4):
```
- **Now `a` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)     # we ASK the environment: cheating!
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `3`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- **Now `reward` holds** — type: **int (whole number)**; value: `-1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if done else Q[state_of(next_cell)].max()
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 3 are non-zero; smallest -5, largest 0`
3. `next_cell` → *tuple of 2 items (each an int)*: `(0, 1)`
4. `state_of(next_cell)` → *int (whole number)*: `1`
5. `Q[state_of(next_cell)]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
6. `Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
7. `0.0 if done else Q[state_of(next_cell)].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
new = reward + gamma * best_next            # no alpha: write the full answer
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-1`
2. `gamma` → *float (decimal number)*: `0.95`
3. `best_next` → *float (decimal number)*: `0.0`
4. `gamma * best_next` → *float (decimal number)*: `0.0`
- **Now `new` holds** — type: **float (decimal number)**; value: `-1.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `value_iteration(max_sweeps=1)`

```python
biggest_change = max(biggest_change, abs(new - Q[s, a]))
```
Evaluated inside-out (the main pieces):
1. `biggest_change` → *float (decimal number)*: `5.0`
2. `new` → *float (decimal number)*: `-1.0`
3. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 3 are non-zero; smallest -5, largest 0`
4. `s` → *int (whole number)*: `0`
5. `a` → *int (whole number)*: `3`
6. `Q[s, a]` → *float (decimal number)*: `0.0`
7. `new - Q[s, a]` → *float (decimal number)*: `-1.0`
8. `abs(new - Q[s, a])` → *float (decimal number)*: `1.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[s, a] = new
```
Evaluated inside-out (the main pieces):
1. `new` → *float (decimal number)*: `-1.0`
2. `s` → *int (whole number)*: `0`
3. `a` → *int (whole number)*: `3`
4. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 3)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 4 are non-zero; smallest -5, largest 0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for a in range(4):
```
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for s in range(ROWS * COLS):
```
- **Now `s` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
cell = cell_of(s)
```
Evaluated inside-out (the main pieces):
1. `s` → *int (whole number)*: `1`
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if walls[cell] == 1 or cell == GOAL:   # nothing to compute in walls or the goal
```
Evaluated inside-out (the main pieces):
1. `walls` → *numpy array, shape (5, 5), holding whole numbers*: `[[0, 0, 0, 0, 0], [1, 1, 0, 1, 0], [0, 0, 0, 0, 0], [0, 1, 1, 1, 1], [0, 0, 0, 0, 0]]`
2. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
3. `walls[cell]` → *numpy integer (a whole number)*: `0`
4. `walls[cell] == 1` → *bool*: `np.False_`
5. `GOAL` → *tuple of 2 items (each an int)*: `(4, 4)`
6. `cell == GOAL` → *bool (True/False)*: `False`
7. `walls[cell] == 1 or cell == GOAL` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `value_iteration(max_sweeps=1)`

```python
for a in range(4):
```
Evaluated inside-out (the main pieces):
1. `range(4)` → *range*: `range(0, 4)`
- **Now `a` holds** — type: **int (whole number)**; value: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 17.2: type this

```python
Q_exact, sweeps = value_iteration()
```

- **Creates `Q_exact`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 68 are non-zero; smallest -7.789, largest 10`

- **Creates `sweeps`** — type: **int (whole number)**; value: `12`

- **Why / what it means:** Runs until nothing changes, and unpacks the table and the number of sweeps used.

In [ ]:
# Step 17.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.3: type this

```python
print("sweeps needed:", sweeps)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `sweeps` → *int (whole number)*: `12`

**Prints:**
```text
sweeps needed: 12
```

In [ ]:
# Step 17.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.4: type this

```python
show_values(Q_exact)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_exact` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -7.789, largest 10`
2. `show_values(Q_exact)` → *None (nothing)*: `None`

**Prints:**
```text
  -2.9   -2.0   -1.1   -2.0   -2.9 
  ###    ###    -0.1   ###    -2.0 
   3.2    2.1    1.0   -0.1   -1.1 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0    0.0 
```

- **Why / what it means:** The exact value map.

In [ ]:
# Step 17.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.5: type this

```python
print()
```

In [ ]:
# Step 17.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.6: type this

```python
show_policy(Q_exact)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_exact` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -7.789, largest 10`
2. `show_policy(Q_exact)` → *None (nothing)*: `None`

**Prints:**
```text
> > v < v 
# # v # v 
v < < < < 
v # # # # 
> > > > G 
```

- **Why / what it means:** The exact best action per cell.

In [ ]:
# Step 17.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17.7: type this

```python
print(round(Q_exact[state_of(START)].max(), 3), "vs the return computed by hand in M5")
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_exact` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -7.789, largest 10`
2. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `state_of(START)` → *int (whole number)*: `0`
4. `Q_exact[state_of(START)]` → *numpy array, shape (4,), holding decimal numbers*: `[-7.7892, -7.7892, -7.7892, -2.936]`
5. `Q_exact[state_of(START)].max()` → *float (decimal number)*: `-2.935997231706202`
6. `round(Q_exact[state_of(START)].max(), 3)` → *float (decimal number)*: `-2.936`

**Prints:**
```text
-2.936 vs the return computed by hand in M5
```

In [ ]:
# Step 17.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 17 complete: everything it printed, and what it means

```text
sweeps needed: 12
  -2.9   -2.0   -1.1   -2.0   -2.9 
  ###    ###    -0.1   ###    -2.0 
   3.2    2.1    1.0   -0.1   -1.1 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0    0.0 
> > v < v 
# # v # v 
v < < < < 
v # # # # 
> > > > G 
-2.936 vs the return computed by hand in M5
```

**12 sweeps** gave the exact table. (Information travels one cell per sweep, and the farthest cell is 12 steps from the goal.) The value map is the same as the learned one on the route: `-2.9, -2.0, -1.1, ..., 10.0`. The start's best value prints **-2.936**: the *same number three ways*: computed by hand (Step 8), found by value iteration, and learned by Q-learning (Step 14). One small difference in the arrows: at `(0, 4)` value iteration points down while the learned table pointed left. That is not an error: both routes from there cost the same, so it is an exact tie and `argmax` simply picks the first. **Q-learning is a way of solving the same equations by sampling, when you are not allowed to look at the rules.**

## Step 18: How close did the learner get?

**The idea first.** Now we can measure the learner against the exact answer.

This step has **9 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 18.1: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Repeatable randomness.

In [ ]:
# Step 18.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.2: type this

```python
Q_learned, _ = train(300)
```

- **Creates `Q_learned`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`

- **Creates `_`** — type: **list of 300 items (each an int)**; value: `[200, 103, 200, 200, 121, 123, 80, 122, 100, 48, 63, 38, 87, 96, 41, 77, 134, 199, 40, 67, 57, 95, 86, 79, 42, 46, 71, 31, 39, ...`

- **Why / what it means:** Trains the learner again; we keep the table and discard the episode lengths (`_`).

In [ ]:
# Step 18.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.3: type this

```python
visited = Q_learned != 0                  # True where the agent ever updated the box
```

- **Creates `visited`** — type: **numpy array, shape (25, 4), holding True/False values**; value: `100 numbers; 68 are non-zero; smallest 0, largest 1`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_learned` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`

- **Why / what it means:** `!=` means 'not equal'. Comparing a whole array gives an array of `bool`s of the same shape: `True` wherever the agent ever changed a box.

In [ ]:
# Step 18.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.4: type this

```python
gap = np.abs(Q_learned - Q_exact)         # difference, box by box
```

- **Creates `gap`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 65 are non-zero; smallest 0, largest 3.088`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_learned` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `Q_exact` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -7.789, largest 10`
3. `Q_learned - Q_exact` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 65 are non-zero; smallest -3.088, largest 0`

- **Why / what it means:** Subtracts the two tables box by box (both shape `(25, 4)`) and takes absolute values: how far the learner is from the exact answer in each box.

In [ ]:
# Step 18.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.5: type this

```python
print("boxes the agent updated:", visited.sum(), "of", visited.size)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `visited` → *numpy array, shape (25, 4), holding True/False values*: `100 numbers; 68 are non-zero; smallest 0, largest 1`
2. `visited.sum()` → *numpy integer (a whole number)*: `68`
3. `visited.size` → *int (whole number)*: `100`

**Prints:**
```text
boxes the agent updated: 68 of 100
```

In [ ]:
# Step 18.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.6: type this

```python
print("biggest gap on those boxes:", round(gap[visited].max(), 3))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `gap` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 65 are non-zero; smallest 0, largest 3.088`
2. `visited` → *numpy array, shape (25, 4), holding True/False values*: `100 numbers; 68 are non-zero; smallest 0, largest 1`
3. `gap[visited]` → *numpy array, shape (68,), holding decimal numbers*: `68 numbers; 65 are non-zero; smallest 0, largest 3.088`
4. `gap[visited].max()` → *float (decimal number)*: `3.0879301100041`
5. `round(gap[visited].max(), 3)` → *float (decimal number)*: `3.088`

**Prints:**
```text
biggest gap on those boxes: 3.088
```

In [ ]:
# Step 18.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.7: type this

```python
print("average gap on those boxes:", round(gap[visited].mean(), 3))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `gap` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 65 are non-zero; smallest 0, largest 3.088`
2. `visited` → *numpy array, shape (25, 4), holding True/False values*: `100 numbers; 68 are non-zero; smallest 0, largest 1`
3. `gap[visited]` → *numpy array, shape (68,), holding decimal numbers*: `68 numbers; 65 are non-zero; smallest 0, largest 3.088`
4. `gap[visited].mean()` → *float (decimal number)*: `0.16197738885376947`
5. `round(gap[visited].mean(), 3)` → *float (decimal number)*: `0.162`

**Prints:**
```text
average gap on those boxes: 0.162
```

In [ ]:
# Step 18.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.8: type this

```python
print(np.round(Q_learned[state_of(START)], 2))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_learned` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -10.88, largest 10`
2. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `state_of(START)` → *int (whole number)*: `0`
4. `Q_learned[state_of(START)]` → *numpy array, shape (4,), holding decimal numbers*: `[-7.7892, -7.7892, -7.7892, -2.936]`
5. `np.round(Q_learned[state_of(START)], 2)` → *numpy array, shape (4,), holding decimal numbers*: `[-7.79, -7.79, -7.79, -2.94]`

**Prints:**
```text
[-7.79 -7.79 -7.79 -2.94]
```

In [ ]:
# Step 18.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18.9: type this

```python
print(np.round(Q_exact[state_of(START)], 2))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_exact` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 68 are non-zero; smallest -7.789, largest 10`
2. `START` → *tuple of 2 items (each an int)*: `(0, 0)`
3. `state_of(START)` → *int (whole number)*: `0`
4. `Q_exact[state_of(START)]` → *numpy array, shape (4,), holding decimal numbers*: `[-7.7892, -7.7892, -7.7892, -2.936]`
5. `np.round(Q_exact[state_of(START)], 2)` → *numpy array, shape (4,), holding decimal numbers*: `[-7.79, -7.79, -7.79, -2.94]`

**Prints:**
```text
[-7.79 -7.79 -7.79 -2.94]
```

In [ ]:
# Step 18.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 18 complete: everything it printed, and what it means

```text
boxes the agent updated: 68 of 100
biggest gap on those boxes: 3.088
average gap on those boxes: 0.162
[-7.79 -7.79 -7.79 -2.94]
[-7.79 -7.79 -7.79 -2.94]
```

The agent updated 68 of the 100 boxes. The **biggest gap is 3.088** but the **average gap is only 0.162**: most boxes are nearly exact and a few rarely visited ones are stale (like the dead-end cells in Step 15). The start cell's row is **identical** in both tables (`-7.79 -7.79 -7.79 -2.94`). Look at the first three numbers: up, down and left from the start all bump into something, and a bump gives -5 then continues at the start's value: `-5 + 0.95 x (-2.94) = -7.79`. The table obeys the rule even for the bad moves.

## Step 19: Dyna-Q: learning from imagined experience

**The idea first.** In Q-learning every real move is used for one update, then forgotten. **Dyna-Q** remembers what happened (a *model* of the world: "from this cell, this action gave that reward and led there") and, after each real move, replays several remembered moves as extra updates. It is like thinking about past experiences instead of just moving on.

This step has **4 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 19.1: type this

```python
def train_dyna(episodes, planning_steps=10, alpha=0.5, gamma=0.95, eps_start=1.0,
               eps_min=0.05, eps_decay=0.99, max_steps=200, stop_when_optimal=False):
    Q = np.zeros((ROWS * COLS, 4))
    model = {}                            # remembers: (state, action) -> (reward, next state, done)
    shortest = bfs()
    epsilon = eps_start
    history = []
    for episode in range(episodes):
        cell = START
        for t in range(max_steps):
            s = state_of(cell)
            a = choose_action(Q, s, epsilon)
            next_cell, reward, done = step(cell, a)
            s_next = state_of(next_cell)

            best_next = 0.0 if done else Q[s_next].max()          # 1) the normal real update
            Q[s, a] += alpha * (reward + gamma * best_next - Q[s, a])

            model[(s, a)] = (reward, s_next, done)                # 2) remember what happened

            for _ in range(planning_steps):                       # 3) replay remembered moves
                ps, pa = random.choice(list(model.keys()))
                pr, ps_next, pdone = model[(ps, pa)]
                pbest = 0.0 if pdone else Q[ps_next].max()
                Q[ps, pa] += alpha * (pr + gamma * pbest - Q[ps, pa])

            cell = next_cell
            if done:
                break
        epsilon = max(eps_min, epsilon * eps_decay)
        history.append(t + 1)
        if stop_when_optimal:
            p = greedy_path(Q)
            if p is not None and len(p) - 1 == shortest:
                break
    return Q, history
```

- **Creates** `train_dyna` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Like `train`, plus a **model** (a memory of what happened) and extra 'planning' updates.

In [ ]:
# Step 19.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `train_dyna(1, planning_steps=1, max_steps=2)`

**What the function receives** (its inputs, called *parameters*):
- `episodes` — type: **int (whole number)**; value: `1`
- `planning_steps` — type: **int (whole number)**; value: `1`
- `alpha` — type: **float (decimal number)**; value: `0.5`
- `gamma` — type: **float (decimal number)**; value: `0.95`
- `eps_start` — type: **float (decimal number)**; value: `1.0`
- `eps_min` — type: **float (decimal number)**; value: `0.05`
- `eps_decay` — type: **float (decimal number)**; value: `0.99`
- `max_steps` — type: **int (whole number)**; value: `2`
- `stop_when_optimal` — type: **bool (True/False)**; value: `False`

**Now each line runs, top to bottom:**

```python
Q = np.zeros((ROWS * COLS, 4))
```
Evaluated inside-out (the main pieces):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`
3. `ROWS * COLS` → *int (whole number)*: `25`
4. `(ROWS * COLS, 4)` → *tuple of 2 items (each an int)*: `(25, 4)`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers, all zero`
- **Why:** The learner's table.

```python
model = {}                            # remembers: (state, action) -> (reward, next state, done)
```
- **Now `model` holds** — type: **dict with 0 entries**; value: `{}`
- **Why:** An empty dictionary. Its keys will be `(state, action)` pairs and its values `(reward, next state, done)`.

```python
shortest = bfs()
```
- **Now `shortest` holds** — type: **int (whole number)**; value: `12`
- **Why:** For the stopping test.

```python
epsilon = eps_start
```
- **Now `epsilon` holds** — type: **float (decimal number)**; value: `1.0`
- **Why:** As before.


#### …the same call, continued: `train_dyna(1, planning_steps=1, max_steps=2)`

```python
history = []
```
- **Now `history` holds** — type: **list of 0 items**; value: `[]`
- **Why:** As before.

```python
for episode in range(episodes):
```
Evaluated inside-out (the main pieces):
1. `episodes` → *int (whole number)*: `1`
2. `range(episodes)` → *range*: `range(0, 1)`
- **Now `episode` holds** — type: **int (whole number)**; value: `0`
- **Why:** One pass per episode.

```python
cell = START
```
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Why:** Begin at the start.

```python
for t in range(max_steps):
```
Evaluated inside-out (the main pieces):
1. `max_steps` → *int (whole number)*: `2`
2. `range(max_steps)` → *range*: `range(0, 2)`
- **Now `t` holds** — type: **int (whole number)**; value: `0`
- **Why:** One pass per step.

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `s` holds** — type: **int (whole number)**; value: `0`
- **Why:** Current row.

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
2. `s` → *int (whole number)*: `0`
3. `epsilon` → *float (decimal number)*: `1.0`
- **Now `a` holds** — type: **int (whole number)**; value: `2`
- **Why:** Epsilon-greedy action.

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `2`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Now `reward` holds** — type: **int (whole number)**; value: `-5`
- **Now `done` holds** — type: **bool (True/False)**; value: `False`
- **Why:** A real step in the environment.

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `0`
- **Why:** Landing row.

```python
best_next = 0.0 if done else Q[s_next].max()          # 1) the normal real update
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers, all zero`
3. `s_next` → *int (whole number)*: `0`
4. `Q[s_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
- **Now `best_next` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** Part 1: the usual real update starts here.

```python
Q[s, a] += alpha * (reward + gamma * best_next - Q[s, a])
```
Evaluated inside-out (the main pieces):
1. `s` → *int (whole number)*: `0`
2. `a` → *int (whole number)*: `2`
3. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 2)`
4. `alpha` → *float (decimal number)*: `0.5`
5. `reward` → *int (whole number)*: `-5`
6. `gamma` → *float (decimal number)*: `0.95`
7. `best_next` → *float (decimal number)*: `0.0`
8. `gamma * best_next` → *float (decimal number)*: `0.0`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
- **Why:** The same update as always, with the target written inline.

```python
model[(s, a)] = (reward, s_next, done)                # 2) remember what happened
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `s_next` → *int (whole number)*: `0`
3. `done` → *bool (True/False)*: `False`
4. `(reward, s_next, done)` → *tuple of 3 items*: `(-5, 0, False)`
5. `s` → *int (whole number)*: `0`
6. `a` → *int (whole number)*: `2`
7. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 2)`
- **Now `model` holds** — type: **dict with 1 entry**; value: `{(0, 2): (-5, 0, False)}`
- **Why:** Part 2: remember what happened. The key is a tuple of two ints; the value is a tuple of three things. In this deterministic maze the same move always gives the same result, so one memory per box is exact.

```python
for _ in range(planning_steps):                       # 3) replay remembered moves
```
Evaluated inside-out (the main pieces):
1. `planning_steps` → *int (whole number)*: `1`
2. `range(planning_steps)` → *range*: `range(0, 1)`
- **Now `_` holds** — type: **int (whole number)**; value: `0`
- **Why:** Part 3: repeat a few extra updates *without moving*.

```python
ps, pa = random.choice(list(model.keys()))
```
Evaluated inside-out (the main pieces):
1. `model` → *dict with 1 entry*: `{(0, 2): (-5, 0, False)}`
2. `model.keys()` → *dict_keys*: `dict_keys([(0, 2)])`
3. `list(model.keys())` → *list of 1 item (each a tuple)*: `[(0, 2)]`
4. `random.choice(list(model.keys()))` → *tuple of 2 items (each an int)*: `(0, 2)`
- **Now `ps` holds** — type: **int (whole number)**; value: `0`
- **Now `pa` holds** — type: **int (whole number)**; value: `2`
- **Why:** `model.keys()` lists the remembered (state, action) pairs; `list(...)` makes it indexable; `random.choice` picks one; unpacking splits the pair.


#### …the same call, continued: `train_dyna(1, planning_steps=1, max_steps=2)`

```python
pr, ps_next, pdone = model[(ps, pa)]
```
Evaluated inside-out (the main pieces):
1. `model` → *dict with 1 entry*: `{(0, 2): (-5, 0, False)}`
2. `ps` → *int (whole number)*: `0`
3. `pa` → *int (whole number)*: `2`
4. `model[ps, pa]` → *tuple of 3 items*: `(-5, 0, False)`
- **Now `pr` holds** — type: **int (whole number)**; value: `-5`
- **Now `ps_next` holds** — type: **int (whole number)**; value: `0`
- **Now `pdone` holds** — type: **bool (True/False)**; value: `False`
- **Why:** Looks up what happened then: reward, next state, done.

```python
pbest = 0.0 if pdone else Q[ps_next].max()
```
Evaluated inside-out (the main pieces):
1. `pdone` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -2.5, largest 0`
3. `ps_next` → *int (whole number)*: `0`
4. `Q[ps_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., -2.5, 0.]`
5. `Q[ps_next].max()` → *float (decimal number)*: `0.0`
- **Now `pbest` holds** — type: **float (decimal number)**; value: `0.0`
- **Why:** The best next value according to the *current* table.

```python
Q[ps, pa] += alpha * (pr + gamma * pbest - Q[ps, pa])
```
Evaluated inside-out (the main pieces):
1. `ps` → *int (whole number)*: `0`
2. `pa` → *int (whole number)*: `2`
3. `(ps, pa)` → *tuple of 2 items (each an int)*: `(0, 2)`
4. `alpha` → *float (decimal number)*: `0.5`
5. `pr` → *int (whole number)*: `-5`
6. `gamma` → *float (decimal number)*: `0.95`
7. `pbest` → *float (decimal number)*: `0.0`
8. `gamma * pbest` → *float (decimal number)*: `0.0`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest -3.75, largest 0`
- **Why:** The same update, on an imagined repeat of an old move.

```python
for _ in range(planning_steps):                       # 3) replay remembered moves
```
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
cell = next_cell
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Why:** Move for real.

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- **Why:** Leave the step loop at the goal.

```python
for t in range(max_steps):
```
- **Now `t` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `state_of(cell)` → *int (whole number)*: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = choose_action(Q, s, epsilon)
```
Evaluated inside-out (the main pieces):
1. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -3.75, largest 0`
2. `s` → *int (whole number)*: `0`
3. `epsilon` → *float (decimal number)*: `1.0`
4. `choose_action(Q, s, epsilon)` → *int (whole number)*: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `2`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 0), -5, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `state_of(next_cell)` → *int (whole number)*: `0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
best_next = 0.0 if done else Q[s_next].max()          # 1) the normal real update
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -3.75, largest 0`
3. `s_next` → *int (whole number)*: `0`
4. `Q[s_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., -3.75, 0.]`
5. `Q[s_next].max()` → *float (decimal number)*: `0.0`
6. `0.0 if done else Q[s_next].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[s, a] += alpha * (reward + gamma * best_next - Q[s, a])
```
Evaluated inside-out (the main pieces):
1. `s` → *int (whole number)*: `0`
2. `a` → *int (whole number)*: `2`
3. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 2)`
4. `alpha` → *float (decimal number)*: `0.5`
5. `reward` → *int (whole number)*: `-5`
6. `gamma` → *float (decimal number)*: `0.95`
7. `best_next` → *float (decimal number)*: `0.0`
8. `gamma * best_next` → *float (decimal number)*: `0.0`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest -4.375, largest 0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `train_dyna(1, planning_steps=1, max_steps=2)`

```python
model[(s, a)] = (reward, s_next, done)                # 2) remember what happened
```
Evaluated inside-out (the main pieces):
1. `reward` → *int (whole number)*: `-5`
2. `s_next` → *int (whole number)*: `0`
3. `done` → *bool (True/False)*: `False`
4. `(reward, s_next, done)` → *tuple of 3 items*: `(-5, 0, False)`
5. `s` → *int (whole number)*: `0`
6. `a` → *int (whole number)*: `2`
7. `(s, a)` → *tuple of 2 items (each an int)*: `(0, 2)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for _ in range(planning_steps):                       # 3) replay remembered moves
```
Evaluated inside-out (the main pieces):
1. `planning_steps` → *int (whole number)*: `1`
2. `range(planning_steps)` → *range*: `range(0, 1)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
ps, pa = random.choice(list(model.keys()))
```
Evaluated inside-out (the main pieces):
1. `model` → *dict with 1 entry*: `{(0, 2): (-5, 0, False)}`
2. `model.keys()` → *dict_keys*: `dict_keys([(0, 2)])`
3. `list(model.keys())` → *list of 1 item (each a tuple)*: `[(0, 2)]`
4. `random.choice(list(model.keys()))` → *tuple of 2 items (each an int)*: `(0, 2)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
pr, ps_next, pdone = model[(ps, pa)]
```
Evaluated inside-out (the main pieces):
1. `model` → *dict with 1 entry*: `{(0, 2): (-5, 0, False)}`
2. `ps` → *int (whole number)*: `0`
3. `pa` → *int (whole number)*: `2`
4. `model[ps, pa]` → *tuple of 3 items*: `(-5, 0, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
pbest = 0.0 if pdone else Q[ps_next].max()
```
Evaluated inside-out (the main pieces):
1. `pdone` → *bool (True/False)*: `False`
2. `Q` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 1 are non-zero; smallest -4.375, largest 0`
3. `ps_next` → *int (whole number)*: `0`
4. `Q[ps_next]` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., -4.375, 0.]`
5. `Q[ps_next].max()` → *float (decimal number)*: `0.0`
6. `0.0 if pdone else Q[ps_next].max()` → *float (decimal number)*: `0.0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
Q[ps, pa] += alpha * (pr + gamma * pbest - Q[ps, pa])
```
Evaluated inside-out (the main pieces):
1. `ps` → *int (whole number)*: `0`
2. `pa` → *int (whole number)*: `2`
3. `(ps, pa)` → *tuple of 2 items (each an int)*: `(0, 2)`
4. `alpha` → *float (decimal number)*: `0.5`
5. `pr` → *int (whole number)*: `-5`
6. `gamma` → *float (decimal number)*: `0.95`
7. `pbest` → *float (decimal number)*: `0.0`
8. `gamma * pbest` → *float (decimal number)*: `0.0`
- **Now `Q` holds** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 1 are non-zero; smallest -4.688, largest 0`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for _ in range(planning_steps):                       # 3) replay remembered moves
```
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 19.2: type this

```python
experiment("no planning (plain)", trainer=train_dyna, planning_steps=0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('no planning (plain)', trainer=train_dyna, planning_steps=0)` → *None (nothing)*: `None`

**Prints:**
```text
no planning (plain)    episodes to shortest path: [5, 6, 8, 7, 7]  mean 6.6  avg total steps 1128
```

- **Why / what it means:** `planning_steps=0`: no extra updates, so this should match the plain learner exactly. It is a correctness check.

In [ ]:
# Step 19.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.3: type this

```python
experiment("planning_steps=5", trainer=train_dyna, planning_steps=5)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('planning_steps=5', trainer=train_dyna, planning_steps=5)` → *None (nothing)*: `None`

**Prints:**
```text
planning_steps=5       episodes to shortest path: [4, 3, 3, 7, 3]  mean 4.0  avg total steps 668
```

- **Why / what it means:** Five imagined updates after every real move.

In [ ]:
# Step 19.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19.4: type this

```python
experiment("planning_steps=20", trainer=train_dyna, planning_steps=20)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `experiment('planning_steps=20', trainer=train_dyna, planning_steps=20)` → *None (nothing)*: `None`

**Prints:**
```text
planning_steps=20      episodes to shortest path: [3, 6, 3, 4, 2]  mean 3.6  avg total steps 622
```

- **Why / what it means:** Twenty.

In [ ]:
# Step 19.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 19 complete: everything it printed, and what it means

```text
no planning (plain)    episodes to shortest path: [5, 6, 8, 7, 7]  mean 6.6  avg total steps 1128
planning_steps=5       episodes to shortest path: [4, 3, 3, 7, 3]  mean 4.0  avg total steps 668
planning_steps=20      episodes to shortest path: [3, 6, 3, 4, 2]  mean 3.6  avg total steps 622
```

With `planning_steps=0` it reproduces the baseline exactly (6.6 episodes, 1128 moves), a good check that the code is correct. With 5 planning updates it needs **4.0 episodes and 668 moves**, and with 20 it needs **3.6 episodes and 622 moves**. Remembered experience is being squeezed for extra information. The trade-off is computation: each real move now costs up to 21 updates. Remembering transitions is exact only in a deterministic world; in a random one you would store averages.

## Step 20: Numpy for neural networks: matrices, broadcasting, fancy indexing

**The idea first.** A neural network is, at heart, a few matrix multiplications with simple functions between them. Before building one, we need five more numpy tools.

This step has **17 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 20.1: type this

```python
A_ = np.array([[1, 2],
               [3, 4],
               [5, 6]])                   # shape (3, 2)
```

- **Creates `A_`** — type: **numpy array, shape (3, 2), holding whole numbers**; value: `[[1, 2], [3, 4], [5, 6]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[[1, 2], [3, 4], [5, 6]]` → *list of 3 items (each a list)*: `[[1, 2], [3, 4], [5, 6]]`

- **Why / what it means:** A 3 x 2 array of whole numbers (three rows of two).

In [ ]:
# Step 20.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.2: type this

```python
B_ = np.array([[1, 0, 2, 1],
               [0, 1, 1, 3]])             # shape (2, 4)
```

- **Creates `B_`** — type: **numpy array, shape (2, 4), holding whole numbers**; value: `[[1, 0, 2, 1], [0, 1, 1, 3]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[[1, 0, 2, 1], [0, 1, 1, 3]]` → *list of 2 items (each a list)*: `[[1, 0, 2, 1], [0, 1, 1, 3]]`

- **Why / what it means:** A 2 x 4 array.

In [ ]:
# Step 20.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.3: type this

```python
print((A_ @ B_).shape)                    # @ is matrix multiplication: (3,2) @ (2,4) -> (3,4)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `A_` → *numpy array, shape (3, 2), holding whole numbers*: `[[1, 2], [3, 4], [5, 6]]`
2. `B_` → *numpy array, shape (2, 4), holding whole numbers*: `[[1, 0, 2, 1], [0, 1, 1, 3]]`
3. `A_ @ B_` → *numpy array, shape (3, 4), holding whole numbers*: `[[1, 2, 4, 7], [3, 4, 10, 15], [5, 6, 16, 23]]`
4. `(A_ @ B_).shape` → *tuple of 2 items (each an int)*: `(3, 4)`

**Prints:**
```text
(3, 4)
```

In [ ]:
# Step 20.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.4: type this

```python
print(A_ @ B_)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `A_` → *numpy array, shape (3, 2), holding whole numbers*: `[[1, 2], [3, 4], [5, 6]]`
2. `B_` → *numpy array, shape (2, 4), holding whole numbers*: `[[1, 0, 2, 1], [0, 1, 1, 3]]`
3. `A_ @ B_` → *numpy array, shape (3, 4), holding whole numbers*: `[[1, 2, 4, 7], [3, 4, 10, 15], [5, 6, 16, 23]]`

**Prints:**
```text
[[ 1  2  4  7]
 [ 3  4 10 15]
 [ 5  6 16 23]]
```

In [ ]:
# Step 20.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.5: type this

```python
print(A_.T)                               # .T flips rows and columns (transpose)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `A_` → *numpy array, shape (3, 2), holding whole numbers*: `[[1, 2], [3, 4], [5, 6]]`
2. `A_.T` → *numpy array, shape (2, 3), holding whole numbers*: `[[1, 3, 5], [2, 4, 6]]`

**Prints:**
```text
[[1 3 5]
 [2 4 6]]
```

In [ ]:
# Step 20.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.6: type this

```python
bias = np.array([10, 20])
```

- **Creates `bias`** — type: **numpy array, shape (2,), holding whole numbers**; value: `[10, 20]`

- **Why / what it means:** A one-dimensional array of two numbers.

In [ ]:
# Step 20.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.7: type this

```python
print(A_ + bias)                          # "broadcasting": the 2 numbers are added to EVERY row
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `A_` → *numpy array, shape (3, 2), holding whole numbers*: `[[1, 2], [3, 4], [5, 6]]`
2. `bias` → *numpy array, shape (2,), holding whole numbers*: `[10, 20]`
3. `A_ + bias` → *numpy array, shape (3, 2), holding whole numbers*: `[[11, 22], [13, 24], [15, 26]]`

**Prints:**
```text
[[11 22]
 [13 24]
 [15 26]]
```

In [ ]:
# Step 20.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.8: type this

```python
table = np.array([[5., 6., 7., 8.],
                  [1., 2., 3., 4.],
                  [9., 8., 7., 6.]])
```

- **Creates `table`** — type: **numpy array, shape (3, 4), holding decimal numbers**; value: `[[5., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[[5.0, 6.0, 7.0, 8.0], [1.0, 2.0, 3.0, 4.0], [9.0, 8.0, 7.0, 6.0]]` → *list of 3 items (each a list)*: `[[5.0, 6.0, 7.0, 8.0], [1.0, 2.0, 3.0, 4.0], [9.0, 8.0, 7.0, 6.0]]`

- **Why / what it means:** A 3 x 4 array of decimals: like a small Q-table with three cells and four actions.

In [ ]:
# Step 20.8: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.9: type this

```python
rows = np.arange(3)                       # [0 1 2]
```

- **Creates `rows`** — type: **numpy array, shape (3,), holding whole numbers**; value: `[0, 1, 2]`

- **Why / what it means:** `np.arange(3)` is an array `[0 1 2]`, like `range(3)` but as an array. Used to say 'row 0, row 1, row 2'.

In [ ]:
# Step 20.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.10: type this

```python
cols = np.array([2, 0, 3])
```

- **Creates `cols`** — type: **numpy array, shape (3,), holding whole numbers**; value: `[2, 0, 3]`

- **Why / what it means:** One column number for each row: row 0 wants column 2, row 1 wants column 0, row 2 wants column 3.

In [ ]:
# Step 20.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.11: type this

```python
print(table[rows, cols])                  # pairs up: table[0,2], table[1,0], table[2,3]
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (3, 4), holding decimal numbers*: `[[5., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`
2. `rows` → *numpy array, shape (3,), holding whole numbers*: `[0, 1, 2]`
3. `cols` → *numpy array, shape (3,), holding whole numbers*: `[2, 0, 3]`
4. `table[rows, cols]` → *numpy array, shape (3,), holding decimal numbers*: `[7., 1., 6.]`

**Prints:**
```text
[7. 1. 6.]
```

In [ ]:
# Step 20.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.12: type this

```python
print(np.maximum(0, np.array([-2, 3, -1, 5])))   # item-by-item max(0, x): negatives become 0
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[-2, 3, -1, 5]` → *list of 4 items (each an int)*: `[-2, 3, -1, 5]`
2. `np.array([-2, 3, -1, 5])` → *numpy array, shape (4,), holding whole numbers*: `[-2, 3, -1, 5]`
3. `np.maximum(0, np.array([-2, 3, -1, 5]))` → *numpy array, shape (4,), holding whole numbers*: `[0, 3, 0, 5]`

**Prints:**
```text
[0 3 0 5]
```

In [ ]:
# Step 20.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.13: type this

```python
print(np.zeros_like(table).shape)         # zeros with the same shape as another array
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (3, 4), holding decimal numbers*: `[[5., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`
2. `np.zeros_like(table)` → *numpy array, shape (3, 4), holding decimal numbers*: `[[0., 0., 0., 0.], [0., 0., 0., 0.], [0., 0., 0., 0.]]`
3. `np.zeros_like(table).shape` → *tuple of 2 items (each an int)*: `(3, 4)`

**Prints:**
```text
(3, 4)
```

In [ ]:
# Step 20.13: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.14: type this

```python
print(np.random.randn(2, 3).shape)        # random numbers from a bell curve, in a 2x3 array
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.random.randn(2, 3)` → *numpy array, shape (2, 3), holding decimal numbers*: `[[1.5539, 0.8792, 0.2042], [0.2692, -0.321, -2.0007]]`
2. `np.random.randn(2, 3).shape` → *tuple of 2 items (each an int)*: `(2, 3)`

**Prints:**
```text
(2, 3)
```

In [ ]:
# Step 20.14: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.15: type this

```python
copied = table.copy()                     # a real, separate copy
```

- **Creates `copied`** — type: **numpy array, shape (3, 4), holding decimal numbers**; value: `[[5., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (3, 4), holding decimal numbers*: `[[5., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`

- **Why / what it means:** A genuine second array with the same contents. Plain assignment (`copied = table`) would give a second *name* for the same array.

In [ ]:
# Step 20.15: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.16: type this

```python
copied[0, 0] = 99
```

- **Changes `copied` in place** — type: **numpy array, shape (3, 4), holding decimal numbers**; value now: `[[99., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`

- **Why / what it means:** Changes the copy only; the original keeps its 5.0.

In [ ]:
# Step 20.16: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20.17: type this

```python
print(table[0, 0], copied[0, 0])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `table` → *numpy array, shape (3, 4), holding decimal numbers*: `[[5., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`
2. `table[0, 0]` → *float (decimal number)*: `5.0`
3. `copied` → *numpy array, shape (3, 4), holding decimal numbers*: `[[99., 6., 7., 8.], [1., 2., 3., 4.], [9., 8., 7., 6.]]`
4. `copied[0, 0]` → *float (decimal number)*: `99.0`

**Prints:**
```text
5.0 99.0
```

In [ ]:
# Step 20.17: type the code from the cell above, then run this cell (Shift+Enter)


### Step 20 complete: everything it printed, and what it means

```text
(3, 4)
[[ 1  2  4  7]
 [ 3  4 10 15]
 [ 5  6 16 23]]
[[1 3 5]
 [2 4 6]]
[[11 22]
 [13 24]
 [15 26]]
[7. 1. 6.]
[0 3 0 5]
(3, 4)
(2, 3)
5.0 99.0
```

`(3, 4)`, then the product: top-left `1`, top-right `7` (row `[1, 2]` against column `[1, 3]`: `1 + 6 = 7`). The transpose is `[[1 3 5] [2 4 6]]`. Adding the bias gives `[[11 22] [13 24] [15 26]]`: 10 added to column 0 and 20 to column 1 in every row. The fancy index gives `[7. 1. 6.]`. `np.maximum` turns `[-2, 3, -1, 5]` into `[0 3 0 5]`. The copy test prints `5.0 99.0`: changing the copy left the original alone.

**Try this yourself.** By hand: what is `A_ @ B_`'s item in row 2, column 3? (Row `[5, 6]` against column `[1, 3]`: `5 + 18 = 23`. The output shows 23.)

## Step 21: A neural network that outputs Q values

**The idea first.** Replace the table with a function. Give the network a description of the cell; it outputs four numbers, the Q values for the four actions. Training will adjust the network's internal numbers (its **weights**) so those outputs match what a table would hold. Our network has two layers.

This step has **13 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 21.1: type this

```python
np.random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** numpy has its **own** random generator, separate from Python's `random`, so it needs its own seed.

In [ ]:
# Step 21.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.2: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Seed for Python's generator.

In [ ]:
# Step 21.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.3: type this

```python
N_IN, N_HIDDEN, N_OUT = ROWS * COLS, 32, 4
```

- **Creates `N_IN`** — type: **int (whole number)**; value: `25`

- **Creates `N_HIDDEN`** — type: **int (whole number)**; value: `32`

- **Creates `N_OUT`** — type: **int (whole number)**; value: `4`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `ROWS` → *int (whole number)*: `5`
2. `COLS` → *int (whole number)*: `5`
3. `ROWS * COLS` → *int (whole number)*: `25`

- **Why / what it means:** Three sizes assigned at once: 25 inputs (one per cell), 32 hidden neurons, 4 outputs (one Q value per action). The right side is a tuple of three ints, unpacked into three names.

In [ ]:
# Step 21.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.4: type this

```python
W1 = np.random.randn(N_IN, N_HIDDEN) * 0.3      # layer 1 weights: 25 x 32
```

- **Creates `W1`** — type: **numpy array, shape (25, 32), holding decimal numbers**; value: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_IN` → *int (whole number)*: `25`
2. `N_HIDDEN` → *int (whole number)*: `32`
3. `np.random.randn(N_IN, N_HIDDEN)` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -3.046, largest 2.696`

- **Why / what it means:** `randn` makes an array of random numbers from a bell curve (most near 0), shape `(25, 32)`. Multiplying by 0.3 makes them small. These are the **weights** of layer 1: the numbers training will adjust. Starting them random (not all equal) lets different neurons learn different things.

In [ ]:
# Step 21.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.5: type this

```python
b1 = np.zeros(N_HIDDEN)                          # layer 1 biases
```

- **Creates `b1`** — type: **numpy array, shape (32,), holding decimal numbers**; value: `32 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_HIDDEN` → *int (whole number)*: `32`

- **Why / what it means:** The **biases** of layer 1: one number per neuron, starting at zero.

In [ ]:
# Step 21.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.6: type this

```python
W2 = np.random.randn(N_HIDDEN, N_OUT) * 0.3     # layer 2 weights: 32 x 4
```

- **Creates `W2`** — type: **numpy array, shape (32, 4), holding decimal numbers**; value: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_HIDDEN` → *int (whole number)*: `32`
2. `N_OUT` → *int (whole number)*: `4`
3. `np.random.randn(N_HIDDEN, N_OUT)` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -2.583, largest 2.594`

- **Why / what it means:** Layer 2 weights, shape `(32, 4)`: from 32 hidden neurons to 4 outputs.

In [ ]:
# Step 21.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.7: type this

```python
b2 = np.zeros(N_OUT)
```

- **Creates `b2`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 0., 0.]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_OUT` → *int (whole number)*: `4`

- **Why / what it means:** Layer 2 biases.

In [ ]:
# Step 21.7: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.8: type this

```python
def forward(X):
    z1 = X @ W1 + b1                      # @ is matrix multiplication
    h = np.maximum(0, z1)                 # ReLU: negative numbers become 0
    q = h @ W2 + b2                       # 4 outputs per input: the Q values
    return z1, h, q
```

- **Creates** `forward` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Pushes a **batch** of inputs through the network. `X` has one row per input. It returns three arrays so the training code can reuse the intermediate values.

In [ ]:
# Step 21.8: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `forward(np.eye(25)[[23]])`

**What the function receives** (its inputs, called *parameters*):
- `X` — type: **numpy array, shape (1, 25), holding decimal numbers**; value: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`

**Now each line runs, top to bottom:**

```python
z1 = X @ W1 + b1                      # @ is matrix multiplication
```
Evaluated inside-out (the main pieces):
1. `X` → *numpy array, shape (1, 25), holding decimal numbers*: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`
2. `W1` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
3. `X @ W1` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 32 are non-zero; smallest -0.5473, largest 0.5265`
4. `b1` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers, all zero`
- **Now `z1` holds** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 32 are non-zero; smallest -0.5473, largest 0.5265`
- **Why:** `@` is matrix multiplication: a `(B, 25)` array times a `(25, 32)` array gives `(B, 32)`: every neuron's weighted sum of the inputs. Then `+ b1` adds each neuron's bias (broadcasting it across the batch).


#### …the same call, continued: `forward(np.eye(25)[[23]])`

```python
h = np.maximum(0, z1)                 # ReLU: negative numbers become 0
```
Evaluated inside-out (the main pieces):
1. `z1` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 32 are non-zero; smallest -0.5473, largest 0.5265`
- **Now `h` holds** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`
- **Why:** **ReLU**: replace every negative number with 0. This nonlinearity is what lets two layers do more than one.

```python
q = h @ W2 + b2                       # 4 outputs per input: the Q values
```
Evaluated inside-out (the main pieces):
1. `h` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`
2. `W2` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
3. `h @ W2` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
4. `b2` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
- **Now `q` holds** — type: **numpy array, shape (1, 4), holding decimal numbers**; value: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
- **Why:** `(B, 32) @ (32, 4)` gives `(B, 4)`: four Q values for each input.

```python
return z1, h, q
```
Evaluated inside-out (the main pieces):
1. `z1` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 32 are non-zero; smallest -0.5473, largest 0.5265`
2. `h` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`
3. `q` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
4. `(z1, h, q)` → *tuple of 3 items (each a numpy array)*: `(array([[ 0.16920256, -0.38536569, -0.03130305, -0.29640058, -0.35328869, -0.34205889, 0.52649585, -0.03989653, -0.22971066, 0....`
- **Why:** Returns all three as a tuple.

**The function hands back** (its *return value*) — type: **tuple of 3 items (each a numpy array)**; value: `(array([[ 0.16920256, -0.38536569, -0.03130305, -0.29640058, -0.35328869, -0.34205889, 0.52649585, -0.03989653, -0.22971066, 0....`


### Step 21.9: type this

```python
x = np.eye(N_IN)[state_of((4, 3))]        # one-hot input for cell (4, 3)
```

- **Creates `x`** — type: **numpy array, shape (25,), holding decimal numbers**; value: `[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_IN` → *int (whole number)*: `25`
2. `np.eye(N_IN)` → *numpy array, shape (25, 25), holding decimal numbers*: `625 numbers; 25 are non-zero; smallest 0, largest 1`
3. `state_of((4, 3))` → *int (whole number)*: `23`

- **Why / what it means:** `np.eye(25)` is the 25 x 25 identity matrix; indexing it with 23 takes row 23: a **one-hot vector** with a single 1 at position 23. This is how we tell the network *which cell the agent is in*.

In [ ]:
# Step 21.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.10: type this

```python
print(x.shape, x.sum(), int(np.argmax(x)))
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `x` → *numpy array, shape (25,), holding decimal numbers*: `[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]`
2. `x.shape` → *tuple of 1 item (each an int)*: `(25,)`
3. `x.sum()` → *float (decimal number)*: `1.0`
4. `np.argmax(x)` → *numpy integer (a whole number)*: `23`
5. `int(np.argmax(x))` → *int (whole number)*: `23`

**Prints:**
```text
(25,) 1.0 23
```

In [ ]:
# Step 21.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.11: type this

```python
_, _, q = forward(x.reshape(1, -1))       # a batch containing one input
```

- **Creates `_`** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`

- **Creates `q`** — type: **numpy array, shape (1, 4), holding decimal numbers**; value: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `x` → *numpy array, shape (25,), holding decimal numbers*: `[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]`
2. `x.reshape(1, -1)` → *numpy array, shape (1, 25), holding decimal numbers*: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`

- **Why / what it means:** The network expects a *batch*, so `reshape(1, -1)` turns the 25-vector into a `(1, 25)` array (one row; `-1` means 'work out the length'). We keep only the third returned value, the output `q`.

In [ ]:
# Step 21.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.12: type this

```python
print(q.shape)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `q` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
2. `q.shape` → *tuple of 2 items (each an int)*: `(1, 4)`

**Prints:**
```text
(1, 4)
```

In [ ]:
# Step 21.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21.13: type this

```python
print(np.round(q, 3))                     # untrained: small meaningless numbers
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `q` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
2. `np.round(q, 3)` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.33, -0.06, 0.116, 0.063]]`

**Prints:**
```text
[[-0.33  -0.06   0.116  0.063]]
```

In [ ]:
# Step 21.13: type the code from the cell above, then run this cell (Shift+Enter)


### Step 21 complete: everything it printed, and what it means

```text
(25,) 1.0 23
(1, 4)
[[-0.33  -0.06   0.116  0.063]]
```

`(25,) 1.0 23`: a 25-long vector, one 1, at position 23. The output has shape `(1, 4)`: a batch of one input, four Q values. Because the weights are random and untrained, the numbers (`-0.33, -0.06, 0.116, 0.063`) are meaningless. Training must change them. Note that **with a one-hot input, this network is a table in disguise**: each cell's row of the input selects its own slice of the weights. The point is that everything else (loss, gradients, replay memory, target network) is identical to what you would use when the input is richer, such as raw CartPole numbers or an image, where the network *can* generalise between similar inputs and a table cannot.

## Step 22: Teaching the network: loss and gradients

**The idea first.** Training means: measure how wrong the output is (the **loss**), work out how each weight affected that error (the **gradient**), and nudge each weight a little in the direction that reduces it (**gradient descent**). Working out the gradients layer by layer, backward, is called **backpropagation**. We write it explicitly so nothing is magic.

This step has **5 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 22.1: type this

```python
def gradient_step(X, A, Y, lr):
    global W1, b1, W2, b2
    z1, h, q = forward(X)
    B = len(A)
    rows = np.arange(B)
    pred = q[rows, A]                     # the Q value of the action actually taken, per sample
    error = pred - Y                      # prediction minus target
    loss = np.mean(error ** 2)

    dq = np.zeros_like(q)                 # gradient of the loss with respect to the outputs
    dq[rows, A] = 2 * error / B           # only the taken action's output gets a gradient
    dW2 = h.T @ dq
    db2 = dq.sum(axis=0)
    dh = dq @ W2.T                        # push the gradient back through layer 2
    dz1 = dh * (z1 > 0)                   # ReLU passes gradient only where z1 was positive
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0)

    W1 -= lr * dW1                        # step downhill
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    return loss
```

- **Creates** `gradient_step` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** One step of training: measure the error, work out how each weight contributed, and nudge every weight to reduce it. `global W1, b1, W2, b2` allows the function to change these outside arrays.

In [ ]:
# Step 22.1: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `gradient_step(np.eye(N_IN)[[23]], np.array([3]), np.array([10.0]), 0.01)`

**What the function receives** (its inputs, called *parameters*):
- `X` — type: **numpy array, shape (1, 25), holding decimal numbers**; value: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`
- `A` — type: **numpy array, shape (1,), holding whole numbers**; value: `[3]`
- `Y` — type: **numpy array, shape (1,), holding decimal numbers**; value: `[10.]`
- `lr` — type: **float (decimal number)**; value: `0.01`

**Now each line runs, top to bottom:**

```python
z1, h, q = forward(X)
```
Evaluated inside-out (the main pieces):
1. `X` → *numpy array, shape (1, 25), holding decimal numbers*: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`
2. `forward(X)` → *tuple of 3 items (each a numpy array)*: `(array([[ 0.16920256, -0.38536569, -0.03130305, -0.29640058, -0.35328869, -0.34205889, 0.52649585, -0.03989653, -0.22971066, 0....`
- **Now `z1` holds** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 32 are non-zero; smallest -0.5473, largest 0.5265`
- **Now `h` holds** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`
- **Now `q` holds** — type: **numpy array, shape (1, 4), holding decimal numbers**; value: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
- **Why:** Run the network forward on the batch. `q` is shape `(1, 4)` for one input.


#### …the same call, continued: `gradient_step(np.eye(N_IN)[[23]], np.array([3]), np.array([10.0]), 0.01)`

```python
B = len(A)
```
Evaluated inside-out (the main pieces):
1. `A` → *numpy array, shape (1,), holding whole numbers*: `[3]`
- **Now `B` holds** — type: **int (whole number)**; value: `1`
- **Why:** The batch size: the number of rows.

```python
rows = np.arange(B)
```
Evaluated inside-out (the main pieces):
1. `B` → *int (whole number)*: `1`
- **Now `rows` holds** — type: **numpy array, shape (1,), holding whole numbers**; value: `[0]`
- **Why:** `[0 ... B-1]`, the row numbers.

```python
pred = q[rows, A]                     # the Q value of the action actually taken, per sample
```
Evaluated inside-out (the main pieces):
1. `q` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
2. `rows` → *numpy array, shape (1,), holding whole numbers*: `[0]`
3. `A` → *numpy array, shape (1,), holding whole numbers*: `[3]`
- **Now `pred` holds** — type: **numpy array, shape (1,), holding decimal numbers**; value: `[0.0631]`
- **Why:** **Fancy indexing:** pairs `rows` with `A` to pick, for each input, the Q value of the action actually taken (`q[0, 3]` here). A 1D array of `B` floats.

```python
error = pred - Y                      # prediction minus target
```
Evaluated inside-out (the main pieces):
1. `pred` → *numpy array, shape (1,), holding decimal numbers*: `[0.0631]`
2. `Y` → *numpy array, shape (1,), holding decimal numbers*: `[10.]`
- **Now `error` holds** — type: **numpy array, shape (1,), holding decimal numbers**; value: `[-9.9369]`
- **Why:** How far each prediction is from its target. Positive means too high; negative, too low.

```python
loss = np.mean(error ** 2)
```
Evaluated inside-out (the main pieces):
1. `error` → *numpy array, shape (1,), holding decimal numbers*: `[-9.9369]`
2. `error ** 2` → *numpy array, shape (1,), holding decimal numbers*: `[98.7426]`
- **Now `loss` holds** — type: **float (decimal number)**; value: `98.74259315430623`
- **Why:** The **loss**: the average of the squared errors. Squaring makes every error positive and punishes big errors more. Training tries to make this small.

```python
dq = np.zeros_like(q)                 # gradient of the loss with respect to the outputs
```
Evaluated inside-out (the main pieces):
1. `q` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.3304, -0.0597, 0.1158, 0.0631]]`
- **Now `dq` holds** — type: **numpy array, shape (1, 4), holding decimal numbers**; value: `[[0., 0., 0., 0.]]`
- **Why:** An array of zeros shaped like `q`. It will hold the gradient: how the loss changes if each output changes.

```python
dq[rows, A] = 2 * error / B           # only the taken action's output gets a gradient
```
Evaluated inside-out (the main pieces):
1. `error` → *numpy array, shape (1,), holding decimal numbers*: `[-9.9369]`
2. `2 * error` → *numpy array, shape (1,), holding decimal numbers*: `[-19.8739]`
3. `B` → *int (whole number)*: `1`
4. `2 * error / B` → *numpy array, shape (1,), holding decimal numbers*: `[-19.8739]`
5. `rows` → *numpy array, shape (1,), holding whole numbers*: `[0]`
6. `A` → *numpy array, shape (1,), holding whole numbers*: `[3]`
7. `(rows, A)` → *tuple of 2 items (each a numpy array)*: `(array([0]), array([3]))`
- **Now `dq` holds** — type: **numpy array, shape (1, 4), holding decimal numbers**; value: `[[0., 0., 0., -19.8739]]`
- **Why:** Calculus: the loss is `mean(error^2)`, so its slope with respect to one prediction is `2 x error / B`. Only the taken action's output affected the loss, so only those positions get a non-zero gradient.

```python
dW2 = h.T @ dq
```
Evaluated inside-out (the main pieces):
1. `h` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`
2. `h.T` → *numpy array, shape (32, 1), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest 0, largest 0.5265`
3. `dq` → *numpy array, shape (1, 4), holding decimal numbers*: `[[0., 0., 0., -19.8739]]`
- **Now `dW2` holds** — type: **numpy array, shape (32, 4), holding decimal numbers**; value: `128 numbers; 17 are non-zero; smallest -10.46, largest 0`
- **Why:** Gradient for layer 2's weights. `h.T` is the transpose, shape `(32, B)`; times `(B, 4)` gives `(32, 4)`, the same shape as `W2`. Each weight's gradient says: if this weight went up a little, how would the loss change?


#### …the same call, continued: `gradient_step(np.eye(N_IN)[[23]], np.array([3]), np.array([10.0]), 0.01)`

```python
db2 = dq.sum(axis=0)
```
Evaluated inside-out (the main pieces):
1. `dq` → *numpy array, shape (1, 4), holding decimal numbers*: `[[0., 0., 0., -19.8739]]`
- **Now `db2` holds** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 0., -19.8739]`
- **Why:** A bias affects every item in the batch, so its gradient is the sum down the batch.

```python
dh = dq @ W2.T                        # push the gradient back through layer 2
```
Evaluated inside-out (the main pieces):
1. `dq` → *numpy array, shape (1, 4), holding decimal numbers*: `[[0., 0., 0., -19.8739]]`
2. `W2` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
3. `W2.T` → *numpy array, shape (4, 32), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
- **Now `dh` holds** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 32 are non-zero; smallest -13.27, largest 10.7`
- **Why:** Sends the gradient **backward** through layer 2 to the hidden neurons.

```python
dz1 = dh * (z1 > 0)                   # ReLU passes gradient only where z1 was positive
```
Evaluated inside-out (the main pieces):
1. `dh` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 32 are non-zero; smallest -13.27, largest 10.7`
2. `z1` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 32 are non-zero; smallest -0.5473, largest 0.5265`
3. `z1 > 0` → *numpy array, shape (1, 32), holding True/False values*: `32 numbers; 17 are non-zero; smallest 0, largest 1`
- **Now `dz1` holds** — type: **numpy array, shape (1, 32), holding decimal numbers**; value: `32 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
- **Why:** ReLU passed values through only where `z1` was positive, so the gradient flows back only there. `(z1 > 0)` is a `bool` array, which multiplies as 1 or 0.

```python
dW1 = X.T @ dz1
```
Evaluated inside-out (the main pieces):
1. `X` → *numpy array, shape (1, 25), holding decimal numbers*: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`
2. `X.T` → *numpy array, shape (25, 1), holding decimal numbers*: `[[0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.], [0.]...`
3. `dz1` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
- **Now `dW1` holds** — type: **numpy array, shape (25, 32), holding decimal numbers**; value: `800 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
- **Why:** Gradient for layer 1's weights, shape `(25, 32)`.

```python
db1 = dz1.sum(axis=0)
```
Evaluated inside-out (the main pieces):
1. `dz1` → *numpy array, shape (1, 32), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
- **Now `db1` holds** — type: **numpy array, shape (32,), holding decimal numbers**; value: `32 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
- **Why:** Gradient for layer 1's biases.

```python
W1 -= lr * dW1                        # step downhill
```
Evaluated inside-out (the main pieces):
1. `lr` → *float (decimal number)*: `0.01`
2. `dW1` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
3. `lr * dW1` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 17 are non-zero; smallest -0.1327, largest 0.107`
- **Why:** **Gradient descent:** move each weight a small step *against* its gradient (downhill for the loss). `lr` sets the step size. `-=` changes the array in place.

```python
b1 -= lr * db1
```
Evaluated inside-out (the main pieces):
1. `lr` → *float (decimal number)*: `0.01`
2. `db1` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest -13.27, largest 10.7`
3. `lr * db1` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers; 17 are non-zero; smallest -0.1327, largest 0.107`
- **Why:** Same for the biases.

```python
W2 -= lr * dW2
```
Evaluated inside-out (the main pieces):
1. `lr` → *float (decimal number)*: `0.01`
2. `dW2` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 17 are non-zero; smallest -10.46, largest 0`
3. `lr * dW2` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 17 are non-zero; smallest -0.1046, largest 0`
- **Why:** Same for layer 2.


#### …the same call, continued: `gradient_step(np.eye(N_IN)[[23]], np.array([3]), np.array([10.0]), 0.01)`

```python
b2 -= lr * db2
```
Evaluated inside-out (the main pieces):
1. `lr` → *float (decimal number)*: `0.01`
2. `db2` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., -19.8739]`
3. `lr * db2` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., -0.1987]`
- **Why:** Same.

```python
return loss
```
Evaluated inside-out (the main pieces):
1. `loss` → *float (decimal number)*: `98.74259315430623`
- **Why:** Returns the loss so we can watch it fall.

**The function hands back** (its *return value*) — type: **float (decimal number)**; value: `98.74259315430623`


### Step 22.2: type this

```python
# test: teach the network that action 3 from cell (4, 3) is worth 10
X = np.eye(N_IN)[[state_of((4, 3))]]
```

- **Creates `X`** — type: **numpy array, shape (1, 25), holding decimal numbers**; value: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_IN` → *int (whole number)*: `25`
2. `np.eye(N_IN)` → *numpy array, shape (25, 25), holding decimal numbers*: `625 numbers; 25 are non-zero; smallest 0, largest 1`
3. `state_of((4, 3))` → *int (whole number)*: `23`
4. `[state_of((4, 3))]` → *list of 1 item (each an int)*: `[23]`

- **Why / what it means:** One-hot input for cell 23, as a batch of one. The double brackets make it shape `(1, 25)`.

In [ ]:
# Step 22.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 22.3: type this

```python
A = np.array([3])
```

- **Creates `A`** — type: **numpy array, shape (1,), holding whole numbers**; value: `[3]`

- **Why / what it means:** The action taken: 3 (right).

In [ ]:
# Step 22.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 22.4: type this

```python
Y = np.array([10.0])
```

- **Creates `Y`** — type: **numpy array, shape (1,), holding decimal numbers**; value: `[10.]`

- **Why / what it means:** The target we want that action's Q value to have.

In [ ]:
# Step 22.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 22.5: type this

```python
for i in range(6):
    loss = gradient_step(X, A, Y, lr=0.01)
    print(i, round(loss, 4), round(forward(X)[2][0, 3], 3))
```

- **Creates `i`** — type: **int (whole number)**; value: `5`

- **Creates `loss`** — type: **float (decimal number)**; value: `18.35198958439252`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `range(6)` → *range*: `range(0, 6)`
2. `X` → *numpy array, shape (1, 25), holding decimal numbers*: `[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.]]` *(evaluated 6 times; the first is shown)*
3. `A` → *numpy array, shape (1,), holding whole numbers*: `[3]` *(evaluated 6 times; the first is shown)*
4. `Y` → *numpy array, shape (1,), holding decimal numbers*: `[10.]` *(evaluated 6 times; the first is shown)*
5. `gradient_step(X, A, Y, lr=0.01)` → *float (decimal number)*: `98.74259315430623` *(evaluated 6 times; the first is shown)*
6. `i` → *int (whole number)*: `0` *(evaluated 6 times; the first is shown)*
7. `loss` → *float (decimal number)*: `98.74259315430623` *(evaluated 6 times; the first is shown)*
8. `round(loss, 4)` → *float (decimal number)*: `98.7426` *(evaluated 6 times; the first is shown)*
9. `forward(X)` → *tuple of 3 items (each a numpy array)*: `(array([[ 0.1225526 , -0.38536569, -0.03130305, -0.29640058, -0.35328869, -0.34205889, 0.7919245 , -0.03989653, -0.22971066, 0....` *(evaluated 6 times; the first is shown)*
10. `forward(X)[2]` → *numpy array, shape (1, 4), holding decimal numbers*: `[[-0.2965, -0.1936, 0.1675, 1.0473]]` *(evaluated 6 times; the first is shown)*
11. `forward(X)[2][0, 3]` → *float (decimal number)*: `1.047349943390512` *(evaluated 6 times; the first is shown)*
12. `round(forward(X)[2][0, 3], 3)` → *float (decimal number)*: `1.047` *(evaluated 6 times; the first is shown)*

**Prints:**
```text
0 98.7426 1.047
1 80.1499 1.975
2 64.4026 3.04
3 48.4371 4.312
4 32.352 5.716
5 18.352 7.082
```

- **Why / what it means:** Six training steps on that single example. Each pass: `gradient_step` returns the loss, then we print it and the network's current prediction. `forward(X)[2][0, 3]` is the third returned value, row 0, column 3.

In [ ]:
# Step 22.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 22 complete: everything it printed, and what it means

```text
0 98.7426 1.047
1 80.1499 1.975
2 64.4026 3.04
3 48.4371 4.312
4 32.352 5.716
5 18.352 7.082
```

The loss starts at **98.7**. That is `(10 - 0.063)^2`, the squared gap between the target 10 and the untrained output `0.063` you saw in Step 21. After each gradient step the loss falls (`98.7, 80.1, 64.4, 48.4, 32.4, 18.4`) and the prediction climbs toward the target (`1.05, 1.98, 3.04, ..., 7.08`). That is learning. Why `lr=0.01`? Larger steps overshoot: with `0.05`, the loss bounced up and down instead of falling steadily. Choosing the learning rate is a trade-off here too.

**Try this yourself.** Raise `lr` to `0.05` for the test loop and run it. What happens to the loss? Why?

## Step 23: Deep Q-learning in miniature: replay memory and a target network

**The idea first.** Now put it together: Q-learning where the Q function is the network. Two ideas make this stable, and both are in every deep Q-learning program.

**Replay memory:** store every experience `(state, action, reward, next state, done)` and train on a *random handful* of old ones each step, instead of only the latest. Consecutive experiences are strongly related, and training on them in order makes the network chase its own tail; random sampling breaks that, and each experience is reused many times.

**Target network:** the targets `reward + gamma x best next` are computed from the network itself, so every update moves its own targets. To steady this, targets come from a *frozen copy* of the weights that is refreshed only every `target_every` steps.

This step has **15 small pieces**. For each one: read the explanation, type the code shown into the empty code cell right below it, and run it.

### Step 23.1: type this

```python
np.random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `np.random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Restart numpy's generator so the network starts the same as before.

In [ ]:
# Step 23.1: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.2: type this

```python
random.seed(0)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `random.seed(0)` → *None (nothing)*: `None`

- **Why / what it means:** Restart Python's generator.

In [ ]:
# Step 23.2: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.3: type this

```python
W1 = np.random.randn(N_IN, N_HIDDEN) * 0.3
```

- **Creates `W1`** — type: **numpy array, shape (25, 32), holding decimal numbers**; value: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_IN` → *int (whole number)*: `25`
2. `N_HIDDEN` → *int (whole number)*: `32`
3. `np.random.randn(N_IN, N_HIDDEN)` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -3.046, largest 2.696`

- **Why / what it means:** A fresh, untrained network: layer 1 weights.

In [ ]:
# Step 23.3: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.4: type this

```python
b1 = np.zeros(N_HIDDEN)
```

- **Creates `b1`** — type: **numpy array, shape (32,), holding decimal numbers**; value: `32 numbers, all zero`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_HIDDEN` → *int (whole number)*: `32`

- **Why / what it means:** Layer 1 biases.

In [ ]:
# Step 23.4: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.5: type this

```python
W2 = np.random.randn(N_HIDDEN, N_OUT) * 0.3
```

- **Creates `W2`** — type: **numpy array, shape (32, 4), holding decimal numbers**; value: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_HIDDEN` → *int (whole number)*: `32`
2. `N_OUT` → *int (whole number)*: `4`
3. `np.random.randn(N_HIDDEN, N_OUT)` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -2.583, largest 2.594`

- **Why / what it means:** Layer 2 weights.

In [ ]:
# Step 23.5: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.6: type this

```python
b2 = np.zeros(N_OUT)
```

- **Creates `b2`** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 0., 0.]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `N_OUT` → *int (whole number)*: `4`

- **Why / what it means:** Layer 2 biases.

In [ ]:
# Step 23.6: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.7: type this

```python
def network_table():
    return forward(np.eye(N_IN))[2]       # feed every one-hot state: a 25 x 4 table of outputs
```

- **Creates** `network_table` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** Feeds every possible one-hot state through the network at once to produce a `(25, 4)` array: the network's own Q-table, which lets us reuse `show_values` and `greedy_path`.

In [ ]:
# Step 23.7: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `network_table()`

**Now each line runs, top to bottom:**

```python
return forward(np.eye(N_IN))[2]       # feed every one-hot state: a 25 x 4 table of outputs
```
Evaluated inside-out (the main pieces):
1. `N_IN` → *int (whole number)*: `25`
2. `np.eye(N_IN)` → *numpy array, shape (25, 25), holding decimal numbers*: `625 numbers; 25 are non-zero; smallest 0, largest 1`
3. `forward(np.eye(N_IN))` → *tuple of 3 items (each a numpy array)*: `(array([[ 5.29215704e-01, 1.20047163e-01, 2.93621395e-01, 6.72267960e-01, 5.60267397e-01, -2.93183364e-01, 2.85026525e-01, -4.5...`
4. `forward(np.eye(N_IN))[2]` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 100 are non-zero; smallest -0.6446, largest 0.5834`
- **Why:** `np.eye(25)` is *all 25* one-hot inputs as a batch; `forward` returns three things and `[2]` takes the third, the outputs.

**The function hands back** (its *return value*) — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 100 are non-zero; smallest -0.6446, largest 0.5834`


### Step 23.8: type this

```python
def train_network(episodes, lr=0.02, gamma=0.95, eps_start=1.0, eps_min=0.05,
                  eps_decay=0.99, batch_size=32, target_every=50, max_steps=200):
    global W1, b1, W2, b2
    memory = deque(maxlen=2000)           # replay memory: forgets the oldest when full
    epsilon = eps_start
    steps_taken = 0
    old_weights = None
    history = []
    for episode in range(episodes):
        cell = START
        for t in range(max_steps):
            s = state_of(cell)
            if random.random() < epsilon:
                a = random.randrange(4)
            else:
                a = int(np.argmax(forward(np.eye(N_IN)[[s]])[2][0]))
            next_cell, reward, done = step(cell, a)
            s_next = state_of(next_cell)
            memory.append((s, a, reward, s_next, done))      # store the experience
            steps_taken += 1

            if steps_taken % target_every == 1:              # refresh the frozen "target" copy
                old_weights = (W1.copy(), b1.copy(), W2.copy(), b2.copy())

            if len(memory) >= batch_size:
                batch = random.sample(memory, batch_size)    # a random handful of old experiences
                S_ = np.array([m[0] for m in batch])
                A_ = np.array([m[1] for m in batch])
                R_ = np.array([m[2] for m in batch], dtype=float)
                N_ = np.array([m[3] for m in batch])
                D_ = np.array([m[4] for m in batch], dtype=float)

                tw1, tb1, tw2, tb2 = old_weights             # targets come from the frozen copy
                q_next = np.maximum(0, np.eye(N_IN)[N_] @ tw1 + tb1) @ tw2 + tb2
                targets = R_ + gamma * q_next.max(axis=1) * (1 - D_)
                gradient_step(np.eye(N_IN)[S_], A_, targets, lr)

            cell = next_cell
            if done:
                break
        epsilon = max(eps_min, epsilon * eps_decay)
        history.append(t + 1)
    return history
```

- **Creates** `train_network` — type: **function (a stored recipe)**. Defining a function only stores the recipe; its lines do not run until something *calls* it.

- **Why / what it means:** The training loop of deep Q-learning. `memory` is the replay memory; `old_weights` is the frozen copy used for targets.

In [ ]:
# Step 23.8: type the code from the cell above, then run this cell (Shift+Enter)


#### Inside the function: one concrete call, `train_network(1, batch_size=2, max_steps=3, target_every=2)`

**What the function receives** (its inputs, called *parameters*):
- `episodes` — type: **int (whole number)**; value: `1`
- `lr` — type: **float (decimal number)**; value: `0.02`
- `gamma` — type: **float (decimal number)**; value: `0.95`
- `eps_start` — type: **float (decimal number)**; value: `1.0`
- `eps_min` — type: **float (decimal number)**; value: `0.05`
- `eps_decay` — type: **float (decimal number)**; value: `0.99`
- `batch_size` — type: **int (whole number)**; value: `2`
- `target_every` — type: **int (whole number)**; value: `2`
- `max_steps` — type: **int (whole number)**; value: `3`

**Now each line runs, top to bottom:**

```python
memory = deque(maxlen=2000)           # replay memory: forgets the oldest when full
```
- **Now `memory` holds** — type: **deque (a waiting line) with 0 items**; value: `[]`
- **Why:** A waiting line with a **maximum length**: once 2000 experiences are stored, adding a new one silently drops the oldest.

```python
epsilon = eps_start
```
- **Now `epsilon` holds** — type: **float (decimal number)**; value: `1.0`
- **Why:** Epsilon starts at 1.0.

```python
steps_taken = 0
```
- **Now `steps_taken` holds** — type: **int (whole number)**; value: `0`
- **Why:** Counts every step ever taken, to time the target refresh.

```python
old_weights = None
```
- **Now `old_weights` holds** — type: **None (nothing)**; value: `None`
- **Why:** The frozen copy does not exist yet.

```python
history = []
```
- **Now `history` holds** — type: **list of 0 items**; value: `[]`
- **Why:** Collects episode lengths.


#### …the same call, continued: `train_network(1, batch_size=2, max_steps=3, target_every=2)`

```python
for episode in range(episodes):
```
Evaluated inside-out (the main pieces):
1. `episodes` → *int (whole number)*: `1`
2. `range(episodes)` → *range*: `range(0, 1)`
- **Now `episode` holds** — type: **int (whole number)**; value: `0`
- **Why:** One pass per episode.

```python
cell = START
```
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 0)`
- **Why:** Begin at the start.

```python
for t in range(max_steps):
```
Evaluated inside-out (the main pieces):
1. `max_steps` → *int (whole number)*: `3`
2. `range(max_steps)` → *range*: `range(0, 3)`
- **Now `t` holds** — type: **int (whole number)**; value: `0`
- **Why:** One pass per step.

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
- **Now `s` holds** — type: **int (whole number)**; value: `0`
- **Why:** Row number of the current cell.

```python
if random.random() < epsilon:
```
Evaluated inside-out (the main pieces):
1. `random.random()` → *float (decimal number)*: `0.8444218515250481`
2. `epsilon` → *float (decimal number)*: `1.0`
3. `random.random() < epsilon` → *bool (True/False)*: `True`
- **Why:** Explore with probability epsilon.

```python
a = random.randrange(4)
```
- **Now `a` holds** — type: **int (whole number)**; value: `3`
- **Why:** A random action. (Otherwise the code, not shown in this first pass, uses the network's best output.)

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 0)`
2. `a` → *int (whole number)*: `3`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- **Now `reward` holds** — type: **int (whole number)**; value: `-1`
- **Now `done` holds** — type: **bool (True/False)**; value: `False`
- **Why:** The environment responds.

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `1`
- **Why:** Landing row.

```python
memory.append((s, a, reward, s_next, done))      # store the experience
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 0 items*: `[]`
2. `s` → *int (whole number)*: `0`
3. `a` → *int (whole number)*: `3`
4. `reward` → *int (whole number)*: `-1`
5. `s_next` → *int (whole number)*: `1`
6. `done` → *bool (True/False)*: `False`
7. `(s, a, reward, s_next, done)` → *tuple of 5 items*: `(0, 3, -1, 1, False)`
8. `memory.append((s, a, reward, s_next, done))` → *None (nothing)*: `None`
- **Now `memory` holds** — type: **deque (a waiting line) with 1 item**; value: `[(0, 3, -1, 1, False)]`
- **Why:** Stores one **experience** as a tuple of five items. Later, training samples random experiences from here.

```python
steps_taken += 1
```
- **Now `steps_taken` holds** — type: **int (whole number)**; value: `1`
- **Why:** One more step overall.

```python
if steps_taken % target_every == 1:              # refresh the frozen "target" copy
```
Evaluated inside-out (the main pieces):
1. `steps_taken` → *int (whole number)*: `1`
2. `target_every` → *int (whole number)*: `2`
3. `steps_taken % target_every` → *int (whole number)*: `1`
4. `steps_taken % target_every == 1` → *bool (True/False)*: `True`
- **Why:** `%` is the remainder after division. This is true on the first step and every `target_every` steps after that: time to refresh the frozen copy.

```python
old_weights = (W1.copy(), b1.copy(), W2.copy(), b2.copy())
```
Evaluated inside-out (the main pieces):
1. `W1` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
2. `W1.copy()` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
3. `b1` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers, all zero`
4. `b1.copy()` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers, all zero`
5. `W2` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
6. `W2.copy()` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
7. `b2` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
8. `b2.copy()` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., 0.]`
- **Now `old_weights` holds** — type: **tuple of 4 items (each a numpy array)**; value: `(array([[ 5.29215704e-01, 1.20047163e-01, 2.93621395e-01, 6.72267960e-01, 5.60267397e-01, -2.93183364e-01, 2.85026525e-01, -4.5...`
- **Why:** A tuple of four **separate copies** of the weight arrays. The copies stay fixed while the real weights keep changing; targets will be computed from them.

```python
if len(memory) >= batch_size:
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 1 item*: `[(0, 3, -1, 1, False)]`
2. `len(memory)` → *int (whole number)*: `1`
3. `batch_size` → *int (whole number)*: `2`
4. `len(memory) >= batch_size` → *bool (True/False)*: `False`
- **Why:** Only train once the memory holds at least one batch.


#### …the same call, continued: `train_network(1, batch_size=2, max_steps=3, target_every=2)`

```python
cell = next_cell
```
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- **Why:** Move on.

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- **Why:** Leave the step loop at the goal.

```python
for t in range(max_steps):
```
- **Now `t` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `s` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if random.random() < epsilon:
```
Evaluated inside-out (the main pieces):
1. `random.random()` → *float (decimal number)*: `0.04048437818077755`
2. `epsilon` → *float (decimal number)*: `1.0`
3. `random.random() < epsilon` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = random.randrange(4)
```
Evaluated inside-out (the main pieces):
1. `random.randrange(4)` → *int (whole number)*: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 1)`
2. `a` → *int (whole number)*: `3`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 2), -1, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 2)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 2)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
memory.append((s, a, reward, s_next, done))      # store the experience
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 1 item*: `[(0, 3, -1, 1, False)]`
2. `s` → *int (whole number)*: `1`
3. `a` → *int (whole number)*: `3`
4. `reward` → *int (whole number)*: `-1`
5. `s_next` → *int (whole number)*: `2`
6. `done` → *bool (True/False)*: `False`
7. `(s, a, reward, s_next, done)` → *tuple of 5 items*: `(1, 3, -1, 2, False)`
8. `memory.append((s, a, reward, s_next, done))` → *None (nothing)*: `None`
- **Now `memory` holds** — type: **deque (a waiting line) with 2 items**; value: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False)]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
steps_taken += 1
```
- **Now `steps_taken` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if steps_taken % target_every == 1:              # refresh the frozen "target" copy
```
Evaluated inside-out (the main pieces):
1. `steps_taken` → *int (whole number)*: `2`
2. `target_every` → *int (whole number)*: `2`
3. `steps_taken % target_every` → *int (whole number)*: `0`
4. `steps_taken % target_every == 1` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if len(memory) >= batch_size:
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 2 items*: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False)]`
2. `len(memory)` → *int (whole number)*: `2`
3. `batch_size` → *int (whole number)*: `2`
4. `len(memory) >= batch_size` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
batch = random.sample(memory, batch_size)    # a random handful of old experiences
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 2 items*: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False)]`
2. `batch_size` → *int (whole number)*: `2`
- **Now `batch` holds** — type: **list of 2 items (each a tuple)**; value: `[(1, 3, -1, 2, False), (0, 3, -1, 1, False)]`
- **Why:** `random.sample` picks `batch_size` *different* experiences at random. Training on random old experiences, not just the latest, breaks the strong correlation between consecutive steps.


#### …the same call, continued: `train_network(1, batch_size=2, max_steps=3, target_every=2)`

```python
S_ = np.array([m[0] for m in batch])
```
Evaluated inside-out (the main pieces):
1. `[m[0] for m in batch]` → *list of 2 items (each an int)*: `[1, 0]`
- **Now `S_` holds** — type: **numpy array, shape (2,), holding whole numbers**; value: `[1, 0]`
- **Why:** A list comprehension collects item 0 (the state) of every experience in the batch; `np.array` makes it an array of whole numbers.

```python
A_ = np.array([m[1] for m in batch])
```
Evaluated inside-out (the main pieces):
1. `[m[1] for m in batch]` → *list of 2 items (each an int)*: `[3, 3]`
- **Now `A_` holds** — type: **numpy array, shape (2,), holding whole numbers**; value: `[3, 3]`
- **Why:** The actions taken.

```python
R_ = np.array([m[2] for m in batch], dtype=float)
```
Evaluated inside-out (the main pieces):
1. `[m[2] for m in batch]` → *list of 2 items (each an int)*: `[-1, -1]`
- **Now `R_` holds** — type: **numpy array, shape (2,), holding decimal numbers**; value: `[-1., -1.]`
- **Why:** The rewards, as decimals.

```python
N_ = np.array([m[3] for m in batch])
```
Evaluated inside-out (the main pieces):
1. `[m[3] for m in batch]` → *list of 2 items (each an int)*: `[2, 1]`
- **Now `N_` holds** — type: **numpy array, shape (2,), holding whole numbers**; value: `[2, 1]`
- **Why:** The next states.

```python
D_ = np.array([m[4] for m in batch], dtype=float)
```
Evaluated inside-out (the main pieces):
1. `[m[4] for m in batch]` → *list of 2 items (each a bool)*: `[False, False]`
- **Now `D_` holds** — type: **numpy array, shape (2,), holding decimal numbers**; value: `[0., 0.]`
- **Why:** The `done` flags as 1.0 (finished) or 0.0 (not).

```python
tw1, tb1, tw2, tb2 = old_weights             # targets come from the frozen copy
```
Evaluated inside-out (the main pieces):
1. `old_weights` → *tuple of 4 items (each a numpy array)*: `(array([[ 5.29215704e-01, 1.20047163e-01, 2.93621395e-01, 6.72267960e-01, 5.60267397e-01, -2.93183364e-01, 2.85026525e-01, -4.5...`
- **Now `tw1` holds** — type: **numpy array, shape (25, 32), holding decimal numbers**; value: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
- **Now `tb1` holds** — type: **numpy array, shape (32,), holding decimal numbers**; value: `32 numbers, all zero`
- **Now `tw2` holds** — type: **numpy array, shape (32, 4), holding decimal numbers**; value: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
- **Now `tb2` holds** — type: **numpy array, shape (4,), holding decimal numbers**; value: `[0., 0., 0., 0.]`
- **Why:** Unpacks the frozen copy into four names.

```python
q_next = np.maximum(0, np.eye(N_IN)[N_] @ tw1 + tb1) @ tw2 + tb2
```
Evaluated inside-out (the main pieces):
1. `N_IN` → *int (whole number)*: `25`
2. `np.eye(N_IN)` → *numpy array, shape (25, 25), holding decimal numbers*: `625 numbers; 25 are non-zero; smallest 0, largest 1`
3. `N_` → *numpy array, shape (2,), holding whole numbers*: `[2, 1]`
4. `np.eye(N_IN)[N_]` → *numpy array, shape (2, 25), holding decimal numbers*: `50 numbers; 2 are non-zero; smallest 0, largest 1`
5. `tw1` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
6. `np.eye(N_IN)[N_] @ tw1` → *numpy array, shape (2, 32), holding decimal numbers*: `64 numbers; 64 are non-zero; smallest -0.5942, largest 0.5852`
7. `tb1` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers, all zero`
8. `np.eye(N_IN)[N_] @ tw1 + tb1` → *numpy array, shape (2, 32), holding decimal numbers*: `64 numbers; 64 are non-zero; smallest -0.5942, largest 0.5852`
- **Now `q_next` holds** — type: **numpy array, shape (2, 4), holding decimal numbers**; value: `[[-0.0478, -0.1418, 0.2747, 0.0826], [0.0744, 0.0919, -0.0718, -0.1074]]`
- **Why:** The network's forward pass written out, but using the **frozen** weights: one-hot inputs for the next states, layer 1, ReLU, layer 2. The result has shape `(batch, 4)`: Q values of every action in each next state.

```python
targets = R_ + gamma * q_next.max(axis=1) * (1 - D_)
```
Evaluated inside-out (the main pieces):
1. `R_` → *numpy array, shape (2,), holding decimal numbers*: `[-1., -1.]`
2. `gamma` → *float (decimal number)*: `0.95`
3. `q_next` → *numpy array, shape (2, 4), holding decimal numbers*: `[[-0.0478, -0.1418, 0.2747, 0.0826], [0.0744, 0.0919, -0.0718, -0.1074]]`
4. `q_next.max(axis=1)` → *numpy array, shape (2,), holding decimal numbers*: `[0.2747, 0.0919]`
5. `gamma * q_next.max(axis=1)` → *numpy array, shape (2,), holding decimal numbers*: `[0.261, 0.0873]`
6. `D_` → *numpy array, shape (2,), holding decimal numbers*: `[0., 0.]`
7. `1 - D_` → *numpy array, shape (2,), holding decimal numbers*: `[1., 1.]`
8. `gamma * q_next.max(axis=1) * (1 - D_)` → *numpy array, shape (2,), holding decimal numbers*: `[0.261, 0.0873]`
- **Now `targets` holds** — type: **numpy array, shape (2,), holding decimal numbers**; value: `[-0.739, -0.9127]`
- **Why:** The target for each experience: reward plus discounted best next value. `max(axis=1)` takes the best of each row; `(1 - D_)` is 0 for finished episodes, removing the future after the goal.

```python
gradient_step(np.eye(N_IN)[S_], A_, targets, lr)
```
Evaluated inside-out (the main pieces):
1. `N_IN` → *int (whole number)*: `25`
2. `np.eye(N_IN)` → *numpy array, shape (25, 25), holding decimal numbers*: `625 numbers; 25 are non-zero; smallest 0, largest 1`
3. `S_` → *numpy array, shape (2,), holding whole numbers*: `[1, 0]`
4. `np.eye(N_IN)[S_]` → *numpy array, shape (2, 25), holding decimal numbers*: `50 numbers; 2 are non-zero; smallest 0, largest 1`
5. `A_` → *numpy array, shape (2,), holding whole numbers*: `[3, 3]`
6. `targets` → *numpy array, shape (2,), holding decimal numbers*: `[-0.739, -0.9127]`
7. `lr` → *float (decimal number)*: `0.02`
8. `gradient_step(np.eye(N_IN)[S_], A_, targets, lr)` → *float (decimal number)*: `1.1473607185308774`
- **Why:** Trains the network toward these targets, using the function from Step 22.

```python
cell = next_cell
```
- **Now `cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 2)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if done:
```
Evaluated inside-out (the main pieces):
1. `done` → *bool (True/False)*: `False`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
for t in range(max_steps):
```
- **Now `t` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s = state_of(cell)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 2)`
- **Now `s` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*


#### …the same call, continued: `train_network(1, batch_size=2, max_steps=3, target_every=2)`

```python
if random.random() < epsilon:
```
Evaluated inside-out (the main pieces):
1. `random.random()` → *float (decimal number)*: `0.9677999949201714`
2. `epsilon` → *float (decimal number)*: `1.0`
3. `random.random() < epsilon` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
a = random.randrange(4)
```
- **Now `a` holds** — type: **int (whole number)**; value: `2`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
next_cell, reward, done = step(cell, a)
```
Evaluated inside-out (the main pieces):
1. `cell` → *tuple of 2 items (each an int)*: `(0, 2)`
2. `a` → *int (whole number)*: `2`
3. `step(cell, a)` → *tuple of 3 items*: `((0, 1), -1, False)`
- **Now `next_cell` holds** — type: **tuple of 2 items (each an int)**; value: `(0, 1)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
s_next = state_of(next_cell)
```
Evaluated inside-out (the main pieces):
1. `next_cell` → *tuple of 2 items (each an int)*: `(0, 1)`
- **Now `s_next` holds** — type: **int (whole number)**; value: `1`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
memory.append((s, a, reward, s_next, done))      # store the experience
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 2 items*: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False)]`
2. `s` → *int (whole number)*: `2`
3. `a` → *int (whole number)*: `2`
4. `reward` → *int (whole number)*: `-1`
5. `s_next` → *int (whole number)*: `1`
6. `done` → *bool (True/False)*: `False`
7. `(s, a, reward, s_next, done)` → *tuple of 5 items*: `(2, 2, -1, 1, False)`
8. `memory.append((s, a, reward, s_next, done))` → *None (nothing)*: `None`
- **Now `memory` holds** — type: **deque (a waiting line) with 3 items**; value: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False), (2, 2, -1, 1, False)]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
steps_taken += 1
```
- **Now `steps_taken` holds** — type: **int (whole number)**; value: `3`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if steps_taken % target_every == 1:              # refresh the frozen "target" copy
```
Evaluated inside-out (the main pieces):
1. `steps_taken` → *int (whole number)*: `3`
2. `target_every` → *int (whole number)*: `2`
3. `steps_taken % target_every` → *int (whole number)*: `1`
4. `steps_taken % target_every == 1` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
old_weights = (W1.copy(), b1.copy(), W2.copy(), b2.copy())
```
Evaluated inside-out (the main pieces):
1. `W1` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
2. `W1.copy()` → *numpy array, shape (25, 32), holding decimal numbers*: `800 numbers; 800 are non-zero; smallest -0.9138, largest 0.8089`
3. `b1` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers; 27 are non-zero; smallest -0.02092, largest 0.009787`
4. `b1.copy()` → *numpy array, shape (32,), holding decimal numbers*: `32 numbers; 27 are non-zero; smallest -0.02092, largest 0.009787`
5. `W2` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
6. `W2.copy()` → *numpy array, shape (32, 4), holding decimal numbers*: `128 numbers; 128 are non-zero; smallest -0.7748, largest 0.7783`
7. `b2` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., -0.0402]`
8. `b2.copy()` → *numpy array, shape (4,), holding decimal numbers*: `[0., 0., 0., -0.0402]`
- **Now `old_weights` holds** — type: **tuple of 4 items (each a numpy array)**; value: `(array([[ 5.32447634e-01, 1.20263542e-01, 2.88292452e-01, 6.57927483e-01, 5.60077721e-01, -2.93183364e-01, 2.66637511e-01, -4.5...`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
if len(memory) >= batch_size:
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 3 items*: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False), (2, 2, -1, 1, False)]`
2. `len(memory)` → *int (whole number)*: `3`
3. `batch_size` → *int (whole number)*: `2`
4. `len(memory) >= batch_size` → *bool (True/False)*: `True`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
batch = random.sample(memory, batch_size)    # a random handful of old experiences
```
Evaluated inside-out (the main pieces):
1. `memory` → *deque (a waiting line) with 3 items*: `[(0, 3, -1, 1, False), (1, 3, -1, 2, False), (2, 2, -1, 1, False)]`
2. `batch_size` → *int (whole number)*: `2`
- **Now `batch` holds** — type: **list of 2 items (each a tuple)**; value: `[(2, 2, -1, 1, False), (0, 3, -1, 1, False)]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
S_ = np.array([m[0] for m in batch])
```
Evaluated inside-out (the main pieces):
1. `[m[0] for m in batch]` → *list of 2 items (each an int)*: `[2, 0]`
- **Now `S_` holds** — type: **numpy array, shape (2,), holding whole numbers**; value: `[2, 0]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
A_ = np.array([m[1] for m in batch])
```
Evaluated inside-out (the main pieces):
1. `[m[1] for m in batch]` → *list of 2 items (each an int)*: `[2, 3]`
- **Now `A_` holds** — type: **numpy array, shape (2,), holding whole numbers**; value: `[2, 3]`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

```python
R_ = np.array([m[2] for m in batch], dtype=float)
```
- **Now `m` holds** — type: **tuple of 5 items**; value: `(2, 2, -1, 1, False)`
- *(Same line as one you have already seen, now with new values. Compare them with last time.)*

*(The trace stops here to keep it short; the function keeps going in the same way.)*


### Step 23.9: type this

```python
history_nn = train_network(400)
```

- **Creates `history_nn`** — type: **list of 400 items (each an int)**; value: `[200, 200, 200, 200, 200, 200, 185, 200, 200, 200, 200, 38, 61, 200, 88, 74, 63, 50, 76, 50, 58, 51, 107, 56, 106, 70, 46, 34, ...`

- **Why / what it means:** Trains for 400 episodes. The result is a list of episode lengths.

In [ ]:
# Step 23.9: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.10: type this

```python
print([round(sum(history_nn[i:i + 50]) / 50, 1) for i in range(0, 400, 50)])
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `[round(sum(history_nn[i:i + 50]) / 50, 1) for i in range(0, 400, 50)]` → *list of 8 items (each a float)*: `[85.1, 24.0, 17.5, 14.6, 13.6, 13.0, 12.6, 12.7]`

**Prints:**
```text
[85.1, 24.0, 17.5, 14.6, 13.6, 13.0, 12.6, 12.7]
```

In [ ]:
# Step 23.10: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.11: type this

```python
Q_net = network_table()
```

- **Creates `Q_net`** — type: **numpy array, shape (25, 4), holding decimal numbers**; value: `100 numbers; 100 are non-zero; smallest -7.789, largest 10`

- **Why / what it means:** Reads the trained network out as a `(25, 4)` table.

In [ ]:
# Step 23.11: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.12: type this

```python
show_values(Q_net)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_net` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 100 are non-zero; smallest -7.789, largest 10`
2. `show_values(Q_net)` → *None (nothing)*: `None`

**Prints:**
```text
  -2.9   -2.0   -1.1   -2.0   -2.9 
  ###    ###    -0.1   ###    -2.0 
   3.2    2.1    1.0   -0.1   -1.1 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0   -0.2 
```

- **Why / what it means:** The network's value map.

In [ ]:
# Step 23.12: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.13: type this

```python
show_policy(Q_net)
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_net` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 100 are non-zero; smallest -7.789, largest 10`
2. `show_policy(Q_net)` → *None (nothing)*: `None`

**Prints:**
```text
> > v < v 
# # v # v 
v < < < < 
v # # # # 
> > > > G 
```

- **Why / what it means:** The network's arrows.

In [ ]:
# Step 23.13: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.14: type this

```python
path = greedy_path(Q_net)
```

- **Creates `path`** — type: **list of 13 items (each a tuple)**; value: `[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]`

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `Q_net` → *numpy array, shape (25, 4), holding decimal numbers*: `100 numbers; 100 are non-zero; smallest -7.789, largest 10`

- **Why / what it means:** Follows the network's arrows from the start.

In [ ]:
# Step 23.14: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23.15: type this

```python
print("network's path length:", None if path is None else len(path) - 1, " shortest:", bfs())
```

**Evaluated inside-out** (each piece, in the order the computer works it out):
1. `path` → *list of 13 items (each a tuple)*: `[(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3), (4, 4)]`
2. `path is None` → *bool (True/False)*: `False`
3. `len(path)` → *int (whole number)*: `13`
4. `len(path) - 1` → *int (whole number)*: `12`
5. `None if path is None else len(path) - 1` → *int (whole number)*: `12`
6. `bfs()` → *int (whole number)*: `12`

**Prints:**
```text
network's path length: 12  shortest: 12
```

In [ ]:
# Step 23.15: type the code from the cell above, then run this cell (Shift+Enter)


### Step 23 complete: everything it printed, and what it means

```text
[85.1, 24.0, 17.5, 14.6, 13.6, 13.0, 12.6, 12.7]
  -2.9   -2.0   -1.1   -2.0   -2.9 
  ###    ###    -0.1   ###    -2.0 
   3.2    2.1    1.0   -0.1   -1.1 
   4.4   ###    ###    ###    ###  
   5.7    7.1    8.5   10.0   -0.2 
> > v < v 
# # v # v 
v < < < < 
v # # # # 
> > > > G 
network's path length: 12  shortest: 12
```

This takes about 15 to 30 seconds. The average episode length falls: `85.1, 24.0, 17.5, 14.6, 13.6, 13.0, 12.6, 12.7`, matching the table method's curve. The network's value map is **the same map** as the exact table (`-2.9, -2.0, -1.1, -0.1 ... 5.7, 7.1, 8.5, 10.0`), and the greedy path has length **12**, equal to the true shortest. (The goal cell shows `-0.2`: the network was never trained on the goal's row because nothing is learned from standing on the goal, so that output is leftover random noise. Ignore it.)

**What you have just built is the structure of deep Q-learning:** environment, replay memory, a network predicting Q values, targets from a frozen copy, gradient descent on the squared error. Frameworks like Keras or PyTorch compute the gradients for you and offer bigger layers, but the loop is the one you typed. For CartPole you would change only the input to the four raw state numbers and the output to two, with no buckets needed.

Honest caveat: with a one-hot input, the network does no better than the table, and it is slower. The payoff comes when inputs are rich and similar inputs should get similar answers, which is where tables become impossible.

**Try this yourself.** Change `target_every=50` to `target_every=1` (a new frozen copy every step, which is no freezing at all) and `hidden` or `lr`. Does training still converge? Make a prediction first.

---
# Where the wider field goes from here

| Method | What it learns | In this notebook |
|---|---|---|
| **Tabular Q-learning** | A table of values by trial and error | Steps 9 to 16 |
| **Value iteration / dynamic programming** | The same table, by direct calculation (needs the rules) | Step 17 |
| **Dyna-Q (planning)** | The table, plus a model used for extra imagined updates | Step 19 |
| **Deep Q-learning (DQN)** | The Q function as a neural network, with replay memory and target network | Steps 20 to 23 |
| **Policy gradient, actor-critic, PPO** | A policy network directly, with or without a value function | A natural next step |

**Good next reading (free):** *Reinforcement Learning: An Introduction* by Sutton and Barto (the standard textbook, available free online) for tabular methods, planning (Dyna) and the maths; OpenAI's "Spinning Up in Deep RL" guide; the Gymnasium documentation. The companion **CartPole DIY** notebook uses these same ideas on a continuous problem and ends with a method that solves it using four numbers.

**Ideas for your own project:**
1. Load a bigger or randomly generated maze with `load_maze` and see how the number of episodes needed grows.
2. Make moves random: with 20% probability the agent slips to a random neighbouring direction. Which of `alpha = 0.1` and `alpha = 1.0` works better now, and why?
3. Add a penalty for revisiting a cell. Does the agent still find the shortest path?
4. Give the network coordinates `(row / 5, col / 5)` as input instead of a one-hot vector. Does it still learn? Does it generalise between neighbouring cells?
5. Replace Dyna's random planning with *prioritised* planning: replay the transitions with the biggest recent error first.

# Cheat sheet: the numpy and Python you used

| You wrote | It means |
|---|---|
| `np.array([[...], [...]])` | make a 2D array from nested lists |
| `a.shape`, `a.reshape(r, c)` | size in each dimension; same items, new shape |
| `a[2, 1]`, `a[(2, 1)]`, `a[1]`, `a[:, 0]` | one item; a whole row; a whole column |
| `np.zeros((r, c))`, `np.zeros_like(a)` | arrays of zeros |
| `np.eye(n)` | identity matrix; its rows are one-hot vectors |
| `a.max(axis=1)`, `a.argmax(axis=1)` | best value / its position, per row |
| `a == a.max()`, `np.flatnonzero(mask)` | which entries tie; their positions |
| `a.any()` | is any entry non-zero? |
| `a.sum()`, `a.sum(axis=0)`, `np.mean(a)` | totals and averages |
| `a @ b`, `a.T` | matrix multiplication; transpose |
| `a + bias` | broadcasting: add to every row |
| `a[rows, cols]` | pick one item per (row, col) pair |
| `np.maximum(0, a)` | ReLU: negatives become 0 |
| `np.abs(a)` | absolute values |
| `a.copy()` | a separate copy |
| `random.seed(n)`, `random.random()`, `random.choice(x)` | repeatable randomness |
| `np.random.randn(r, c)` | bell-curve random numbers |
| `{k: v}`, `k in d` | dictionary; does a key exist? |
| `deque`, `.append`, `.popleft()` | a waiting line |
| `global name` | let a function change an outside name |
| `try: ... finally: ...` | cleanup that runs even after an error |
| `f"{x:6.1f}"` | print x 6 wide with 1 decimal |

# Glossary

- **Environment:** the world; given a state and action, returns the next state and a reward.
- **State:** where the agent is (here: a cell).
- **Action:** a choice (up, down, left, right).
- **Reward:** the immediate score after an action.
- **Episode:** one attempt, from the start to the goal (or a step limit).
- **Policy:** a rule from state to action.
- **Return:** total future reward, with later rewards discounted.
- **Gamma:** the discount factor, 0 to 1.
- **Q(s, a):** the estimated return of taking `a` in `s` and then playing well.
- **Alpha:** the learning rate: how far each update moves toward its target.
- **Epsilon:** the probability of acting randomly.
- **On/off-policy:** whether the update evaluates the behaviour actually followed (SARSA) or the best behaviour (Q-learning).
- **Bellman equation:** the self-consistency condition a converged Q-table satisfies: value = reward + gamma x best next value.
- **Replay memory / target network / loss / gradient:** the pieces of deep Q-learning from Steps 22 and 23.

# Check yourself

Say each answer out loud before reading it.

1. **What does `Q[s, a]` mean?** The estimated discounted return from taking `a` in `s` and then playing well.
2. **Which lines are the learning, and how many boxes change per step?** `target = reward + gamma * best_next` and `Q[s, a] += alpha * (target - Q[s, a])`. Exactly one box.
3. **Why is `best_next` zero at the goal?** The episode is over, so nothing more can be earned.
4. **Why does information spread backward from the goal?** A cell's target includes the best value of the cell it lands in. If that cell has learned something, this cell learns it on its next visit.
5. **Why did a wall bump costing 0 break the agent?** Standing still scored better than moving, so it learned to stand still. Rewards define the goal literally.
6. **Why did `epsilon = 0` still work here, but not in CartPole?** Here every reward is negative, so tried boxes fall below the untried zeros. In CartPole all rewards were positive.
7. **How is value iteration different from Q-learning?** It computes the table from the known rules, with sweeps; Q-learning samples experience and needs no rules.
8. **Why does a deep Q-learning program need replay memory and a target network?** Consecutive experiences are correlated, and targets computed from the network being trained keep moving; random sampling and a frozen copy steady both.
9. **How do you know the learner is right?** Compare against a trusted answer: here the BFS shortest path and the exact table from value iteration.